# ITSG Growth ARR Forecast — Foundation v5

Weekly ARR forecast for **ITSG Growth bookings** (net-new, `amount_towards_plan`), from raw Salesforce
OLI data to Delta output tables, using five models — Prophet, MSTL, ETS (Holt-Winters), DHR-ARIMA, and a
Global LightGBM quantile model — combined into an Adaptive Ensemble with market-level reconciliation, a
27-week production forward forecast, a Monte Carlo quarterly-distribution step, and a monitoring baseline.

ITSG is a small, noisy series (~$0.35M average weekly ARR — roughly a third of UCC), so this notebook is
deliberately conservative: every accuracy claim comes from data the models never trained on or tuned
against, and the quarterly Monte Carlo distributions matter at least as much as the point forecasts.

**Key methodology changes vs the prior version (V4), all flagged again at the step where they live:**

1. **Clean evaluation splits.** Three-way temporal split: CV region → 13-week validation slice (tuning
   only) → untouched holdout from `2026-01-05`. V4 ran 8 CV folds across the whole series, so folds 7-8
   trained on data inside the holdout; here the fold count is computed so no fold touches it, and asserts
   enforce the boundary.
2. **LightGBM panel rebuilt on a complete SFDC spine.** V4 built the LGB panel on `forecast_prophet`'s
   rows (~84% market-week coverage), leaving holes in the target and gappy lag features — the root cause
   of its ~120% holdout WAPE. v2 builds a full week × market grid, takes the target from SFDC ATP, and
   left-joins `forecast_prophet` as features only.
3. **Removed the holdout-derived global scaling.** V4 multiplied every production forecast by a 0.831
   "calibration" factor estimated from holdout errors — an adjustment applied in production but absent
   from every reported metric. v2 has no such transform: evaluation measures exactly what production does.
4. **Ensemble weights from clean CV, not the holdout.** Inverse mean CV WAPE (all folds pre-holdout).
   V4's fallback weighted by holdout WAPE, contaminating the ensemble's own holdout numbers.
5. **Tuned, not hand-picked, hyperparameters.** The training-window length (78 vs 104 vs all weeks),
   Prophet's changepoint prior, ETS seasonality, MSTL periods, DHR Fourier order, and LGB depth/rate are
   all selected on the validation slice — one place, never the holdout. V4 hardcoded a 78-week window and
   applied it inconsistently (benchmarks rolling, LGB expanding).
6. **Consistent, comparable metrics.** All models are evaluated at the weekly *total* level with the same
   metric code (V4 scored LGB at market level, producing meaningless 400,000% MAPEs on near-zero slices);
   quarter definitions come from `to_period('Q')` everywhere (V4 had three conflicting "Q1 actual" values).
7. **Deterministic LightGBM** (`deterministic=true`, single thread) so identical data reproduces identical
   results; recursive prediction keeps ≥60 weeks of history so long lags don't silently zero out (V4 kept
   26, starving `y_lag26/52` at prediction time only).
8. **Data-driven LGB promotion gate.** V4 excluded LGB from the ensemble via a hardcoded comment. v2
   tests the rebuilt LGB on the validation slice and includes it only if it's within 1.25x the best
   benchmark's validation WAPE — the decision is printed, not hand-maintained.
9. **All summaries generated from live DataFrames** — no hand-typed metrics anywhere (V4's summary
   contradicted its own cell outputs). Non-negativity is enforced inside every model's predict path.

**Additional improvements (v2.1 session):**

10. **Continuous quarter-end proximity features** (`qe_proximity`, `qe_proximity_sq`) added to
    LightGBM's feature set — a 0→1 ramp for weeks 9-13 that models the nonlinear surge shape better
    than the binary `quarter_end_flag`. Expected: -1 to -2 WAPE pts.
11. **Data freshness check** (Step 1B) — flags the last closed week if it falls below 15% of the
    trailing 8-week average, preventing stale/incomplete data from entering production.

**v2.2 session (2026-08-30) — ported UCC V5 accuracy improvements:**

12. **mc_actuals coverage guard** (Step 6) — when trailing 8-week non-zero authoritative coverage
    < 50%, substitutes magna_carta_actuals proxy at lag-4. Gave UCC a 7.22pp bias improvement
    (-18.62% → -11.40%); expected similar gain for ITSG.
13. **DHR_ARIMA flagged as skippable** (Step 5) — worst performer at 39.7% WAPE, already excluded
    from ensemble by Step 6B. Cell can be skipped in production to save compute.
14. **Stage-weighted pipeline + large-deal indicators** already present from v2.1.

**Status:** run top-to-bottom on Databricks after the Step 0 kernel restart.

### Step 0 — Install forecasting dependencies *(unchanged)*

`statsforecast` (MSTL/AutoETS/AutoARIMA), `prophet`, `statsmodels` (Holt-Winters ETS), `holidays`,
`lightgbm`, `scipy`, `shap`. Databricks restarts Python after this cell — re-run everything from the top.

In [ ]:
# STEP 0: INSTALL FORECASTING DEPENDENCIES
# ─────────────────────────────────────────────────────────────────────────────
# PURPOSE: Install all Python libraries not pre-loaded in the Databricks runtime.
# Run this cell ONCE per new cluster session.  After it completes, Databricks
# automatically restarts the Python kernel — you must re-run ALL subsequent
# cells from the top before any model results will be valid.
#
# Package breakdown:
#   statsforecast  — Fast vectorised time-series models: MSTL v2, AutoETS,
#                    AutoARIMA.  Used in Steps 4 and 5.
#   prophet        — Facebook/Meta Prophet: trend + seasonality + holiday
#                    decomposition model.  Used in Step 4.
#   statsmodels    — ExponentialSmoothing (Holt-Winters) and ARIMA utilities
#                    used inside the DHR-ARIMA helper (Step 5).
#   holidays       — Public-holiday calendars for US/UK/DE/IN/BR.
#                    Used to build the `us_holiday_binary` external regressor.
#   lightgbm       — Gradient-boosted trees for the Global LGB Quantile model
#                    (Step 6) and Adaptive Ensemble weighting.
#   scipy          — Required by statsforecast's internal optimisers.
#   shap           — SHAP TreeExplainer for LightGBM feature importance.
# ─────────────────────────────────────────────────────────────────────────────

# Pin exact versions for reproducibility — aligned with UCC Forecast Foundation V5.
# statsmodels pinned at 0.14.6 to prevent forecast() index breakage (0.15+ changes
# prediction-index generation; see DatetimeIndex fix in fit_predict_ets).
%pip install statsforecast==2.0.3 prophet==1.3.0 statsmodels==0.14.6 holidays==0.72 lightgbm==4.5.0 scipy==1.14.1 shap==0.46.0

### Step 1 — Load ITSG forecast base data **(CHANGED — complete spine, target/feature separation, consistency diagnostic)**

Business scope preserved from V4: closed-won SFDC opportunity line items, `product_group = 'ITSG'`,
`purchase_type_rollup = 'Growth'`, channels excluding Care / Sales Other, `close_date >= 2023-01-01`,
target = weekly Monday-bucketed sum of `amount_towards_plan` (V4's "ATP fix" — kept). Changes:

- **Calendar spine.** Weekly and week × market series are built on a complete Monday grid so
  zero-booking weeks exist as $0 rows instead of silently missing (critical for lag/rolling features on a
  series this sparse — some ITSG market-weeks genuinely book nothing).
- **`forecast_prophet` is features-only.** Its `Actuals` column is loaded as `fp_actuals_reference` and
  never used as a target or feature; the target always comes from SFDC ATP. A diagnostic prints how far
  the two series diverge (V4 found a 0.67x mismatch — that comparison is now a permanent, visible check).
- All pipeline-style features (`all_created_arr`, headcount, activity aggregates) come from
  `forecast_prophet`, which is an all-outcome source — so there is no won-only survivorship problem in the
  feature set. `datagroup.uc_forecast.actuals` (RSG-IT mapping) remains a feature-only input.
- The total-level feature frame for DHR is assembled on the spine with per-column forward-fill, so feature
  gaps in `forecast_prophet` don't punch holes in the regressor matrix.

### Step 0C - Improvement config, evidence-gate library and self-tests *(NEW 2026-10-08)*

Master switch `IMPROVEMENT_GATES_ENABLED`, scope switch `ITSG_INCLUDE_CARE_SALES_OTHER`, gate/Miradore config; pure-python evaluation functions (expanding window, no leakage) and synthetic self-tests that run first. Any self-test failure forces the switch off (previous behaviour).

In [ ]:
# =============================================================================
# STEP 0C: IMPROVEMENT CONFIG (NEW 2026-10-08) -- evidence gates, scope switch, Miradore outlook
# =============================================================================
# Master switch. False => EXACTLY the previous behaviour (no gate evaluated or applied).
IMPROVEMENT_GATES_ENABLED = True
RUN_IMPROVEMENT_SELF_TEST = True        # synthetic-data unit tests (no Spark) run first; any failure forces the switch off

# (E) Scope switch: include sales_channel 'Care' and 'Sales Other' in the ITSG target? Default False = today.
ITSG_INCLUDE_CARE_SALES_OTHER = False
RUN_SCOPE_DIAGNOSTIC = True             # one-time read-only diagnostic: actuals under both scopes by quarter

# (A) LGB feature ablation (evidence-gated). Decides before LGB tuning; production LGB uses the selected set.
LGB_ABLATION_ENABLED = True
LGB_ABLATION_GROUPS = {                 # variant name -> features dropped
    'no_mc': ['mc_actuals_lag4'],
    'no_miradore': ['miradore_lag4'],
    'no_plan_gap': ['plan_gap_lag1'],
    'no_all3': ['mc_actuals_lag4', 'miradore_lag4', 'plan_gap_lag1'],
}
LGB_ABLATION_MARGIN_PP = 0.30           # pooled WAPE must improve by >= this ...
LGB_ABLATION_BIAS_PP = 2.0              # ... or |bias| must fall by >= this with WAPE not worse
LGB_ABLATION_SEG_TOL_PP = 1.0           # and neither the CV nor the validation segment may worsen by more than this
LGB_FEATURE_DROP_OVERRIDE = None        # None = gate decides; or a list of features to force-drop (bypasses the gate)
LGB_FEATURE_DROP_SELECTED = []          # filled by the ablation cell

# (C) Quarter-end option gate: none / flat / mult (today) / data_add vs incumbent (today's multiplicative)
ITSG_QE_FLAT_USD = 0.0                  # flat $ add on the last week of the quarter; 0 for ITSG (not enabled today)
QE_EVAL_MIN_PRIOR_QUARTERS = 2          # ITSG has less OOS history than UCC (CV region + holdout)
QE_EVAL_MIN_SCORED_QUARTERS = 2
QE_EVAL_SHRINK_K = 2.0
QE_EVAL_MULT_CAP = 0.50
QE_EVAL_MARGIN_FRAC = 0.05              # challenger must cut last-week MAE by >= 5% AND win > half of scored quarters
QE_EVAL_MAX_HORIZON = 13
QE_METHOD_SELECTED = 'incumbent'        # incumbent | none | mult | flat | data_add
QE_DATA_ADD_USD = 0.0

# (D) LightGBM weight multiplier gate and level-shift gate
LGB_WEIGHT_MULT_OPTIONS = (1.0, 0.5, 0.0)
LGB_WEIGHT_MARGIN_PP = 0.10
LGB_HOLDOUT_BIAS_GATE_PCT = 5.0
LGB_WEIGHT_MULT_SELECTED = 1.0
LEVEL_ADJ_MODE = 'mult'
LEVEL_ADJ_K_WEEKS = 6
LEVEL_ADJ_SHRINK_K = 3.0
LEVEL_ADJ_H_WEEKS = 8
LEVEL_ADJ_HALFLIFE_WEEKS = 4.0
LEVEL_ADJ_CLAMP = 0.15
LEVEL_ADJ_MARGIN_PP = 0.20
LEVEL_ADJ_MIN_ORIGINS = 3               # ITSG has fewer rolling origins than UCC
LEVEL_ADJ_ENABLED_SELECTED = False
LEVEL_ADJ_DELTA_SELECTED = 0.0

# (F) Pace outlook
PACE_ASOF_WEEKS = (2, 5, 8, 11)
PACE_SHRINK_K = 3.0
PACE_CLAMP = (0.7, 1.4)
PACE_UPLIFT_CAP = 0.50
PACE_MARGIN_PP = 0.5
PACE_MIN_QUARTERS = 3
PACE_OUTLOOK_TABLE = 'datagroup_mdl.mdl_sales_analytics.itsg_forecast_pace_outlook'

# (B) Miradore outlook (SEPARATE from the Salesforce ITSG forecast; Miradore is not in the target)
MIRADORE_OUTLOOK_ENABLED = True
MIRADORE_EXTEND_ENABLED = True          # 2026-10-08: no FY plan after Dec 2026 -> extend a flat estimate (labelled 'extended')
MIRADORE_EXTEND_THROUGH = '2027-12'     # last month of the extension (flat trailing-3-month mean of known actuals)
MIRADORE_PLAN_TABLE = 'datagroup_mdl.mdl_sales_analytics.miradore_plan'
MIRADORE_ACTUALS_TABLE = 'datagroup_mdl.mdl_sales_analytics.mc_actuals'
MIRADORE_VALUE_COLS = ['Reported Bookings Total In USD Order Month Rate', 'ACV In USD Order Month Rate']  # first present wins
MIRADORE_PLAN_VERSION_FMT = 'FY{yy} Plan'   # plan version used for each month's calendar year
MIRADORE_GENUS = 'Miradore'
MIRADORE_ATT_WINDOW = 3                 # last N known complete months
MIRADORE_ATT_SHRINK_K = 2.0             # att = 1 + n/(n+k) * (raw - 1)
MIRADORE_ATT_CLAMP = (0.5, 1.5)
MIRADORE_MIN_HIST = 2
MIRADORE_MIN_BACKTEST = 3
MIRADORE_OUTLOOK_TABLE = 'datagroup_mdl.mdl_sales_analytics.itsg_miradore_outlook'
MIRADORE_TOTAL_TABLE = 'datagroup_mdl.mdl_sales_analytics.itsg_forecast_incl_miradore'
IG_RESULTS = {}
print(f'Step 0C config loaded: IMPROVEMENT_GATES_ENABLED={IMPROVEMENT_GATES_ENABLED}, '
      f'ITSG_INCLUDE_CARE_SALES_OTHER={ITSG_INCLUDE_CARE_SALES_OTHER}')

In [ ]:
# =============================================================================
# STEP 13H-LIB (1/3): EVIDENCE-GATE EVALUATION FUNCTIONS -- common helpers
# =============================================================================
# Pure python (pandas/numpy only; no Spark, no notebook globals) so they can be
# unit-tested offline. Every function uses ONLY information dated before the
# decision it informs (expanding window). Used by the gates that decide whether
# the quarter-end add-on, the LightGBM weight cut, the level-shift correction and
# the pace outlook are adopted. See the 13H self-test cell and the 13H run cell.
import numpy as np
import pandas as pd


def ig_is_last_week(ds):
    """True for the last Monday-week of a calendar quarter (same rule as the
    production QE additive block: week+7d falls in the next quarter)."""
    d = pd.to_datetime(pd.Series(ds)).reset_index(drop=True)
    return ((d + pd.Timedelta(days=7)).dt.quarter != d.dt.quarter).values


def ig_is_qe2(ds):
    """True for the last 2 weeks of a calendar quarter (QE_DOWNWEIGHT rule)."""
    d = pd.to_datetime(pd.Series(ds)).reset_index(drop=True)
    qend = d.dt.to_period('Q').dt.end_time.dt.normalize()
    return ((qend - d).dt.days <= 13).values


def ig_qlabel(ds):
    return pd.to_datetime(pd.Series(ds)).dt.to_period('Q').astype(str).values


def ig_shrunk_mean(x, k):
    """n/(n+k) * mean(x); 0 if empty."""
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    return 0.0 if len(x) == 0 else float(len(x) / (len(x) + k) * x.mean())


def ig_wape(actual, fc):
    a = np.asarray(actual, dtype=float); f = np.asarray(fc, dtype=float)
    m = ~(np.isnan(a) | np.isnan(f))
    return float(np.abs(a[m] - f[m]).sum() / max(a[m].sum(), 1.0) * 100) if m.any() else np.nan


def ig_bias_pct(actual, fc):
    """Signed bias % = (sum fc - sum actual)/sum actual (positive = over-forecast)."""
    a = np.asarray(actual, dtype=float); f = np.asarray(fc, dtype=float)
    m = ~(np.isnan(a) | np.isnan(f))
    return float((f[m].sum() - a[m].sum()) / max(a[m].sum(), 1.0) * 100) if m.any() else np.nan


def ig_ensemble_wide(long_df, weights, lgb_models=(), lgb_mult=1.0,
                     zero_models_on_qe=(), group_cols=('origin_date', 'ds')):
    """Weighted ensemble per (group_cols).
    long_df: model, ds, forecast, actual (+ origin_date/horizon_week if used).
    weights: dict model -> weight. lgb_models get weight*lgb_mult. zero_models_on_qe get
    weight 0 on last-2-week QE rows (mirrors qe_aware_weighted_average). Weights are
    renormalised over models that have a forecast for the row. Returns group cols +
    actual + ens (+ horizon_week if present)."""
    df = long_df[long_df['model'].isin(list(weights))].copy()
    df['ds'] = pd.to_datetime(df['ds'])
    w = df['model'].map(weights).astype(float)
    w = np.where(df['model'].isin(list(lgb_models)), w * lgb_mult, w)
    qe = ig_is_qe2(df['ds'])
    w = np.where(qe & df['model'].isin(list(zero_models_on_qe)), 0.0, w)
    df['_w'] = w
    df['_wf'] = df['_w'] * df['forecast']
    ok = df['forecast'].notna()
    df.loc[~ok, ['_w', '_wf']] = 0.0
    keys = list(group_cols)
    agg = {'_w': 'sum', '_wf': 'sum', 'actual': 'first'}
    if 'horizon_week' in df.columns:
        agg['horizon_week'] = 'first'
    g = df.groupby(keys, as_index=False).agg(agg)
    g['ens'] = np.where(g['_w'] > 0, g['_wf'] / g['_w'].where(g['_w'] > 0, 1), np.nan)
    return g.drop(columns=['_w', '_wf'])


def ig_oos_weekly(ens_df, max_h=13):
    """One forecast per ds = the most recent origin strictly before ds (smallest
    horizon, <= max_h). ens_df: origin_date, ds, actual, ens, horizon_week."""
    d = ens_df.dropna(subset=['ens', 'actual']).copy()
    d['ds'] = pd.to_datetime(d['ds'])
    if 'horizon_week' in d.columns:
        d = d[d['horizon_week'] <= max_h]
        d = d.sort_values(['ds', 'horizon_week'])
    else:
        d = d.sort_values(['ds', 'origin_date'])
    d = d.groupby('ds', as_index=False).first()
    return d[['ds', 'actual', 'ens']].rename(columns={'ens': 'fc'}).sort_values('ds').reset_index(drop=True)

# =============================================================================
# STEP 13H-LIB (2/3): QE-option gate and LightGBM-weight gate (pure python)
# =============================================================================
def ig_qe_options_eval(wk, flat_usd=200_000.0, min_prior=3, shrink_k=2.0, mult_cap=0.5,
                       margin_frac=0.05, incumbent_flat_enabled=True, min_weeks_q=12, min_scored=3):
    """wk: ds, actual, fc (one OOS forecast per week). For each quarter with >= min_prior
    EARLIER quarters, predict its LAST week under options and score. Options:
      none; flat (+flat_usd); mult (prior sum(actual)/sum(fc) of last weeks, in [1,1+cap]);
      data_add (shrunk mean of prior last-week misses); incumbent (mult then flat if enabled).
    Selection: best last-week MAE; incumbent kept unless the winner improves MAE by
    >= margin_frac (relative). Returns dict(table, selected, reason, data_add_usd, n_quarters)."""
    d = wk.dropna(subset=['actual', 'fc']).copy()
    d['ds'] = pd.to_datetime(d['ds']); d = d.sort_values('ds')
    d['q'] = ig_qlabel(d['ds']); d['last'] = ig_is_last_week(d['ds'])
    rows = []
    qs = [q for q, g in d.groupby('q') if len(g) >= min_weeks_q and g['last'].any()]
    for i, q in enumerate(qs):
        if i < min_prior:
            continue
        prior = d[d['q'].isin(qs[:i]) & d['last']]
        miss = (prior['actual'] - prior['fc']).values
        ratio = float(np.clip(prior['actual'].sum() / max(prior['fc'].sum(), 1.0), 1.0, 1.0 + mult_cap))
        dadd = ig_shrunk_mean(miss, shrink_k)
        g = d[d['q'] == q]; lw = g[g['last']].iloc[0]; f = float(lw['fc']); a = float(lw['actual'])
        adj = {'none': 0.0, 'flat': flat_usd, 'mult': f * (ratio - 1), 'data_add': dadd,
               'incumbent': f * (ratio - 1) + (flat_usd if incumbent_flat_enabled else 0.0)}
        tot_a, tot_f = float(g['actual'].sum()), float(g['fc'].sum())
        for m, ad in adj.items():
            rows.append({'quarter': q, 'method': m, 'last_wk_err': a - (f + ad),
                         'qtr_total_err': tot_a - (tot_f + ad)})
    if len({x['quarter'] for x in rows}) < min_scored:
        return {'table': pd.DataFrame(), 'detail': pd.DataFrame(columns=['quarter','method','last_wk_err']), 'selected': 'incumbent', 'n_quarters': 0, 'data_add_usd': 0.0,
                'reason': f'insufficient history (< {min_scored} scored quarters after {min_prior} prior)'}
    r = pd.DataFrame(rows)
    t = r.groupby('method').agg(n=('quarter', 'nunique'), mae=('last_wk_err', lambda x: float(np.abs(x).mean())),
                                bias=('last_wk_err', 'mean'),
                                qtr_total_mae=('qtr_total_err', lambda x: float(np.abs(x).mean())),
                                qtr_total_bias=('qtr_total_err', 'mean')).reset_index()
    t = t.sort_values('mae').reset_index(drop=True)
    best = t.iloc[0]; inc = t[t['method'] == 'incumbent'].iloc[0]
    if best['method'] != 'incumbent' and best['mae'] <= inc['mae'] * (1 - margin_frac):
        sel = best['method']; why = f"{sel} MAE ${best['mae']:,.0f} beats incumbent ${inc['mae']:,.0f} by >= {margin_frac:.0%}"
    else:
        sel = 'incumbent'; why = (f"no option beats incumbent (MAE ${inc['mae']:,.0f}) by >= {margin_frac:.0%}; "
                                   f"best={best['method']} ${best['mae']:,.0f}")
    allmiss = (d[d['q'].isin(qs) & d['last']].eval('actual - fc')).values
    return {'table': t, 'detail': r, 'selected': sel, 'reason': why, 'n_quarters': int(t['n'].max()),
            'data_add_usd': ig_shrunk_mean(allmiss, shrink_k)}


def ig_lgb_weight_eval(bt_ens_by_mult, hold_ens_by_mult=None, mults=(1.0, 0.5, 0.0), margin_pp=0.1,
                       bias_gate_pct=5.0, min_prior_origins=3):
    """bt_ens_by_mult: {mult: DataFrame(origin_date, ds, actual, ens)}. Rolling-origin WAPE per
    mult (pooled), plus an expanding-window selector (each origin decided from EARLIER origins
    only). hold_ens_by_mult: {mult: DataFrame(ds, actual, ens)} for holdout bias/WAPE.
    Selection: best pooled WAPE, tie-break to 1.0 unless gain >= margin_pp; if holdout |bias| at
    1.0 > bias_gate_pct and a reduced option is not worse in WAPE, prefer the reduced option with
    the smallest holdout |bias|."""
    rows = []
    for m in mults:
        b = bt_ens_by_mult[m].dropna(subset=['ens', 'actual'])
        rec = {'mult': m, 'bt_wape': ig_wape(b['actual'], b['ens']), 'bt_bias_pct': ig_bias_pct(b['actual'], b['ens'])}
        if hold_ens_by_mult is not None and m in hold_ens_by_mult:
            h = hold_ens_by_mult[m].dropna(subset=['ens', 'actual'])
            rec.update(ho_wape=ig_wape(h['actual'], h['ens']), ho_bias_pct=ig_bias_pct(h['actual'], h['ens']))
        rows.append(rec)
    t = pd.DataFrame(rows)
    base = t[t['mult'] == 1.0].iloc[0]
    best = t.sort_values('bt_wape').iloc[0]
    sel = 1.0; why = 'keep 1.0 (no option beats it by margin)'
    if best['mult'] != 1.0 and best['bt_wape'] <= base['bt_wape'] - margin_pp:
        sel = float(best['mult']); why = f"mult {sel} lowers rolling WAPE {base['bt_wape']:.2f} -> {best['bt_wape']:.2f}"
    if 'ho_bias_pct' in t.columns and abs(base['ho_bias_pct']) > bias_gate_pct:
        red = t[(t['mult'] < 1.0) & (t['bt_wape'] <= base['bt_wape'])]
        if len(red):
            r0 = red.reindex(red['ho_bias_pct'].abs().sort_values().index).iloc[0]
            if abs(r0['ho_bias_pct']) < abs(base['ho_bias_pct']):
                sel = float(r0['mult']); why = (f"holdout |bias| {abs(base['ho_bias_pct']):.1f}% > gate {bias_gate_pct}%; "
                                                f"mult {sel} not worse in WAPE and cuts |bias| to {abs(r0['ho_bias_pct']):.1f}%")
    # expanding-window selector OOS score
    origins = sorted(bt_ens_by_mult[1.0]['origin_date'].unique())
    oos_sel, oos_one = [], []
    for i, o in enumerate(origins):
        if i < min_prior_origins:
            continue
        past = {m: bt_ens_by_mult[m][bt_ens_by_mult[m]['origin_date'].isin(origins[:i])] for m in mults}
        w = {m: ig_wape(past[m]['actual'], past[m]['ens']) for m in mults}
        pick = 1.0
        bm = min(w, key=lambda k: w[k])
        if bm != 1.0 and w[bm] <= w[1.0] - margin_pp:
            pick = bm
        cur = lambda m: bt_ens_by_mult[m][bt_ens_by_mult[m]['origin_date'] == o]
        oos_sel.append(cur(pick)); oos_one.append(cur(1.0))
    oos = None
    if oos_sel:
        a, b = pd.concat(oos_sel), pd.concat(oos_one)
        oos = {'selector_wape': ig_wape(a['actual'], a['ens']), 'fixed_1_wape': ig_wape(b['actual'], b['ens']),
               'n_origins': len(oos_sel)}
    return {'table': t, 'selected': sel, 'reason': why, 'oos': oos}

# =============================================================================
# STEP 13H-LIB (3/3): level-shift gate and pace-outlook (pure python)
# =============================================================================
def ig_level_delta(wk_hist, K=6, shrink_k=3.0, mode='mult', clamp=0.15, min_hist=3):
    """Shrunk mean of the last K NON-quarter-end weekly residuals (actual - forecast known
    at that time). mode 'add': USD (clamped to +-clamp*mean forecast); 'mult': fraction
    (clamped to +-clamp). wk_hist: ds, actual, fc -- caller passes only weeks already closed."""
    h = wk_hist.dropna(subset=['actual', 'fc']).copy()
    h = h[~ig_is_qe2(h['ds'])].sort_values('ds').tail(K)
    if len(h) < min_hist:
        return 0.0
    if mode == 'add':
        lim = clamp * float(h['fc'].mean())
        return float(np.clip(ig_shrunk_mean((h['actual'] - h['fc']).values, shrink_k), -lim, lim))
    r = (h['actual'] / h['fc'].clip(lower=1.0) - 1.0).values
    return float(np.clip(ig_shrunk_mean(r, shrink_k), -clamp, clamp))


def ig_level_apply(fc, horizon, delta, mode='mult', H=8, halflife=4.0):
    """Apply delta to the first H forward weeks with exponential decay (half-life in weeks)."""
    fc = np.asarray(fc, dtype=float); hz = np.asarray(horizon, dtype=float)
    dec = np.where(hz <= H, 0.5 ** ((hz - 1) / halflife), 0.0)
    return fc + delta * dec if mode == 'add' else fc * (1.0 + delta * dec)


def ig_level_eval(bt_ens, wk, K=6, shrink_k=3.0, mode='mult', clamp=0.15, H=8, halflife=4.0,
                  margin_pp=0.2, min_origins=4, min_hist=3):
    """bt_ens: origin_date, ds, horizon_week, actual, ens (rolling-origin ensemble).
    wk: OOS weekly (ds, actual, fc). Per origin: delta uses wk rows with ds <= origin only.
    Adopt only if non-QE-week WAPE (h<=H) falls by >= margin_pp AND summed |quarter total
    error| per (origin, quarter) does not rise."""
    b = bt_ens.dropna(subset=['ens', 'actual']).copy()
    b['ds'] = pd.to_datetime(b['ds']); wk = wk.copy(); wk['ds'] = pd.to_datetime(wk['ds'])
    parts = []
    for o, g in b[b['horizon_week'] <= H].groupby('origin_date'):
        hist = wk[wk['ds'] <= pd.Timestamp(o)]
        dl = ig_level_delta(hist, K, shrink_k, mode, clamp, min_hist)
        g = g.copy(); g['adj'] = ig_level_apply(g['ens'], g['horizon_week'], dl, mode, H, halflife)
        g['delta'] = dl; parts.append(g)
    if len(parts) < min_origins:
        return {'adopt': False, 'reason': f'insufficient origins ({len(parts)} < {min_origins})', 'n_origins': len(parts)}
    r = pd.concat(parts); r['qe'] = ig_is_qe2(r['ds']); r['q'] = ig_qlabel(r['ds'])
    nq = r[~r['qe']]
    w0, w1 = ig_wape(nq['actual'], nq['ens']), ig_wape(nq['actual'], nq['adj'])
    tq = r.groupby(['origin_date', 'q']).agg(a=('actual', 'sum'), f=('ens', 'sum'), j=('adj', 'sum'))
    t0, t1 = float((tq['a'] - tq['f']).abs().sum()), float((tq['a'] - tq['j']).abs().sum())
    ok = (w1 <= w0 - margin_pp) and (t1 <= t0)
    return {'adopt': bool(ok), 'n_origins': len(parts), 'wape_none': w0, 'wape_adj': w1,
            'qtr_abs_none': t0, 'qtr_abs_adj': t1, 'mode': mode,
            'reason': (f'non-QE WAPE {w0:.2f}->{w1:.2f}, quarter-total abs err {t0:,.0f}->{t1:,.0f}'
                       f' ({"adopt" if ok else "reject"})')}


def ig_pace_calc(closed_a, closed_f, rem_f, rem_is_last, uplift, k=3.0, lo=0.7, hi=1.4):
    """Pace outlook arithmetic. closed_a/closed_f: closed weeks (actual, forecast); rem_f:
    remaining-week forecasts; rem_is_last: bool per remaining week. pace = 1 + w/(w+k)*(ratio-1),
    ratio = sum(closed_a)/sum(closed_f), clamped to [lo,hi]. Outlook = QTD + sum(rem_f*pace*
    (1+uplift on last week))."""
    ca, cf = np.asarray(closed_a, float), np.asarray(closed_f, float)
    w = len(ca); ratio = ca.sum() / max(cf.sum(), 1.0)
    pace = float(np.clip(1.0 + w / (w + k) * (ratio - 1.0), lo, hi))
    rf = np.asarray(rem_f, float); lastm = np.asarray(rem_is_last, bool)
    rem_adj = float((rf * pace * np.where(lastm, 1.0 + uplift, 1.0)).sum())
    return {'pace': pace, 'qtd': float(ca.sum()), 'remaining_forecast': float(rf.sum()),
            'remaining_adj': rem_adj, 'landing': float(ca.sum()) + rem_adj}


def ig_qe_uplift(prior_quarters, k=2.0, cap=0.5):
    """Learned from PRIOR quarters only: shrunk mean of a_last/(f_last*pace_full) - 1, clamped
    to [0, cap]. prior_quarters: list of DataFrames (ds, actual, fc), complete quarters."""
    u = []
    for g in prior_quarters:
        g = g.sort_values('ds'); lm = ig_is_last_week(g['ds'])
        if not lm.any() or (~lm).sum() < 3:
            continue
        pace_q = g.loc[~lm, 'actual'].sum() / max(g.loc[~lm, 'fc'].sum(), 1.0)
        fl = float(g.loc[lm, 'fc'].iloc[0]) * pace_q
        u.append(float(g.loc[lm, 'actual'].iloc[0]) / max(fl, 1.0) - 1.0)
    return float(np.clip(ig_shrunk_mean(u, k), 0.0, cap)) if u else 0.0


def ig_complete_quarters(wk, min_weeks=12):
    """Split an OOS weekly frame into complete quarters (first week within 6 days of the quarter
    start, >= min_weeks rows). Returns list of (label, DataFrame)."""
    d = wk.dropna(subset=['actual', 'fc']).copy(); d['ds'] = pd.to_datetime(d['ds'])
    d = d.sort_values('ds'); d['q'] = ig_qlabel(d['ds']); out = []
    for q, g in d.groupby('q', sort=True):
        qs = pd.Period(q).start_time
        if len(g) >= min_weeks and (g['ds'].min() - qs).days <= 6:
            out.append((q, g.reset_index(drop=True)))
    return out


def ig_pace_eval(wk, asof_weeks=(2, 5, 8, 11), k=3.0, lo=0.7, hi=1.4, uplift_cap=0.5,
                 margin_pp=0.5, min_prior=2, min_quarters=3):
    """Backtest the pace outlook for every complete quarter with >= min_prior earlier quarters.
    Compared with the static model quarter forecast (sum of all weekly forecasts). Validated per
    as-of week when outlook MAPE < model MAPE - margin_pp over >= min_quarters quarters."""
    qs = ig_complete_quarters(wk); rows = []
    for i, (q, g) in enumerate(qs):
        if i < min_prior:
            continue
        upl = ig_qe_uplift([x[1] for x in qs[:i]], cap=uplift_cap)
        act = float(g['actual'].sum()); stat = float(g['fc'].sum()); last = ig_is_last_week(g['ds'])
        for w in asof_weeks:
            if w >= len(g):
                continue
            c = ig_pace_calc(g['actual'][:w], g['fc'][:w], g['fc'][w:], last[w:], upl, k, lo, hi)
            rows.append({'quarter': q, 'asof_week': w, 'actual': act, 'outlook': c['landing'], 'model': stat,
                         'pace': c['pace'], 'uplift': upl,
                         'outlook_err_pct': (c['landing'] - act) / act * 100, 'model_err_pct': (stat - act) / act * 100})
    if not rows:
        return {'detail': pd.DataFrame(), 'summary': pd.DataFrame()}
    d = pd.DataFrame(rows)
    s = d.groupby('asof_week').agg(n_quarters=('quarter', 'nunique'),
                                   outlook_mape=('outlook_err_pct', lambda x: float(np.abs(x).mean())),
                                   outlook_bias=('outlook_err_pct', 'mean'),
                                   model_mape=('model_err_pct', lambda x: float(np.abs(x).mean())),
                                   model_bias=('model_err_pct', 'mean')).reset_index()
    s['validated'] = (s['outlook_mape'] < s['model_mape'] - margin_pp) & (s['n_quarters'] >= min_quarters)
    return {'detail': d, 'summary': s}

# =============================================================================
# STEP 13H-LIB (ITSG extras): ablation selector, QE confirmation, Miradore outlook maths
# =============================================================================
# Pure python (pandas/numpy only, no Spark). Unit-tested in the self-test cell below.
import numpy as np
import pandas as pd


def ig_ablation_select(tbl, full='full', margin_pp=0.30, bias_pp=2.0, seg_tol_pp=1.0):
    """tbl: variant, cv_wape, cv_bias, val_wape, val_bias, pool_wape, pool_bias (bias % signed).
    Adopt a reduced variant only if (pooled WAPE improves by >= margin_pp) OR (|pooled bias| falls by
    >= bias_pp with pooled WAPE no worse), AND neither segment (CV, validation) WAPE worsens by more
    than seg_tol_pp. Otherwise keep the full feature set."""
    t = tbl.copy().reset_index(drop=True)
    base = t[t['variant'] == full].iloc[0]
    best, why = full, 'keep full feature set (no reduced set clears the margin)'
    cands = []
    for _, r in t[t['variant'] != full].iterrows():
        if not np.isfinite(r['pool_wape']):
            continue
        if (r['cv_wape'] > base['cv_wape'] + seg_tol_pp) or (r['val_wape'] > base['val_wape'] + seg_tol_pp):
            continue
        gain = base['pool_wape'] - r['pool_wape']
        bgain = abs(base['pool_bias']) - abs(r['pool_bias'])
        if gain >= margin_pp:
            cands.append((gain, r['variant'], f"pooled WAPE {base['pool_wape']:.2f} -> {r['pool_wape']:.2f}"))
        elif bgain >= bias_pp and r['pool_wape'] <= base['pool_wape']:
            cands.append((bgain * 0.01, r['variant'], f"|bias| {abs(base['pool_bias']):.1f} -> {abs(r['pool_bias']):.1f} pp, WAPE not worse"))
    if cands:
        cands.sort(reverse=True)
        best, why = cands[0][1], cands[0][2]
    return best, why


def ig_qe_confirm(detail, selected, incumbent='incumbent'):
    """A challenger must also beat the incumbent (abs last-week error) in MORE THAN HALF of the scored
    quarters; otherwise the incumbent is kept. detail: quarter, method, last_wk_err."""
    if selected == incumbent or detail is None or len(detail) == 0:
        return selected, ''
    p = detail.pivot_table(index='quarter', columns='method', values='last_wk_err', aggfunc='first').abs()
    if selected not in p.columns or incumbent not in p.columns:
        return incumbent, 'method missing from detail'
    wins = int((p[selected] < p[incumbent]).sum()); n = len(p)
    if wins * 2 > n:
        return selected, f'beats incumbent in {wins}/{n} quarters'
    return incumbent, f'challenger {selected} wins only {wins}/{n} quarters -> incumbent kept'


def ig_qe2_diag(wk, min_prior=2, mult_cap=0.5):
    """Informational: last-2-weeks-of-quarter mean |error| under none vs multiplicative (ratio from
    PRIOR quarters' last-2 weeks only)."""
    d = wk.dropna(subset=['actual', 'fc']).copy(); d['ds'] = pd.to_datetime(d['ds']).values
    d['q'] = ig_qlabel(d['ds']); d['l2'] = ig_is_qe2(d['ds'])
    qs = [q for q, g in d.groupby('q') if len(g) >= 12 and g['l2'].any()]
    rows = []
    for i, q in enumerate(qs):
        if i < min_prior:
            continue
        pr = d[d['q'].isin(qs[:i]) & d['l2']]
        ratio = float(np.clip(pr['actual'].sum() / max(pr['fc'].sum(), 1.0), 1.0, 1.0 + mult_cap))
        g = d[(d['q'] == q) & d['l2']]
        rows.append({'quarter': q, 'none_abs_err': float((g['actual'] - g['fc']).abs().mean()),
                     'mult_abs_err': float((g['actual'] - g['fc'] * ratio).abs().mean()), 'ratio': ratio})
    return pd.DataFrame(rows)

def ig_write_table(pdf, tbl):
    """Overwrite-pattern Delta write + GRANT SELECT to the app service principal; never raises.
    Mirrors the 14F / notebook-03 pattern. Uses the notebook `spark` and APP_SERVICE_PRINCIPAL."""
    try:
        spark.createDataFrame(pdf).write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(tbl)
        print(f'OK  wrote {tbl} ({len(pdf)} rows)')
    except Exception as e:
        print(f'XX  WRITE FAILED for {tbl}: {str(e)[:250]}')
        return False
    try:
        sp = globals().get('APP_SERVICE_PRINCIPAL', '324a6ec7-e988-42c7-8a7f-55465f5bea37')
        spark.sql(f'GRANT SELECT ON TABLE {tbl} TO `{sp}`')
        print(f'OK  granted SELECT on {tbl}')
    except Exception as e:
        print(f'!!  GRANT failed on {tbl} (table WAS written): {str(e)[:200]}')
    return True

# =============================================================================
# STEP 13M-LIB: Miradore outlook maths (pure python; outside the Salesforce ITSG target)
# =============================================================================
def mira_attainment(plan, actual, known, N=3, k=2.0, lo=0.5, hi=1.5):
    """Shrunk attainment from the last N KNOWN months (lists/arrays aligned in time order).
    raw = sum(actual)/sum(plan) over those months; att = 1 + n/(n+k)*(raw-1), clamped to [lo,hi].
    Months with known=False (plan but no actual rows) are EXCLUDED, never treated as zero."""
    p = np.asarray(plan, float); a = np.asarray(actual, float); kn = np.asarray(known, bool)
    idx = [i for i in range(len(p)) if kn[i] and np.isfinite(a[i]) and np.isfinite(p[i]) and p[i] > 0][-N:]
    if not idx:
        return 1.0, 0
    raw = a[idx].sum() / p[idx].sum()
    n = len(idx)
    return float(np.clip(1.0 + n / (n + k) * (raw - 1.0), lo, hi)), n


def mira_backtest(m, N=3, k=2.0, lo=0.5, hi=1.5, min_hist=2):
    """m: DataFrame month, plan, actual (NaN = unknown), known. Expanding window: for each month t
    that has plan AND a known actual and >= min_hist earlier known months, forecast with data
    strictly before t. Returns (detail, summary) with MAE per method: raw_plan, attainment, trail3."""
    m = m.sort_values('month').reset_index(drop=True); rows = []
    for t in range(len(m)):
        if not (m.loc[t, 'known'] and m.loc[t, 'plan'] > 0):
            continue
        hist = m.iloc[:t]; hk = hist[hist['known']]
        if len(hk) < min_hist:
            continue
        att, _ = mira_attainment(hist['plan'], hist['actual'], hist['known'], N, k, lo, hi)
        rows.append({'month': m.loc[t, 'month'], 'actual': m.loc[t, 'actual'], 'raw_plan': m.loc[t, 'plan'],
                     'attainment': m.loc[t, 'plan'] * att, 'trail3': float(hk['actual'].tail(3).mean())})
    d = pd.DataFrame(rows)
    if d.empty:
        return d, {}
    return d, {c: float((d['actual'] - d[c]).abs().mean()) for c in ('raw_plan', 'attainment', 'trail3')}


def mira_choose(summary, n_obs, min_obs=3):
    """Lowest MAE wins; raw plan wins ties and any case with fewer than min_obs backtest months."""
    if not summary or n_obs < min_obs:
        return 'raw_plan', False, f'insufficient backtest months ({n_obs} < {min_obs}) -> raw plan'
    best = min(summary.values())
    if summary['raw_plan'] <= best + 1e-9:
        return 'raw_plan', True, 'raw plan has the lowest (or tied) MAE'
    name = min(summary, key=summary.get)
    return name, True, f'{name} MAE {summary[name]:,.0f} < raw plan {summary["raw_plan"]:,.0f}'


def mira_spread_weeks(month_value, ds_weeks):
    """Spread monthly values (dict Period->value) to weekly ds: value / Mondays in the calendar month."""
    ds = pd.to_datetime(pd.Series(ds_weeks)); per = ds.dt.to_period('M')
    cnt = per.map(lambda p: len(pd.date_range(p.start_time, p.end_time, freq='W-MON')))
    return np.array([month_value.get(p, np.nan) / c for p, c in zip(per, cnt)], float)

In [ ]:
# =============================================================================
# STEP 13H-TEST: SELF-TESTS FOR THE EVIDENCE-GATE FUNCTIONS (no Spark, synthetic data)
# =============================================================================
# Known-answer tests: a series with a constant quarter-end miss (data-driven add must win),
# a leakage trap (future quarters must not change an earlier decision), LightGBM weight
# cut when LGB is injected with a large bias, level-shift correction beats none on a step
# and does NOT on pure noise, hand-computed pace arithmetic, incumbent kept on thin history.
# A failure prints loudly and switches IMPROVEMENT_GATES_ENABLED off (today's behaviour).
def _ig_weekly(n_q=16, qe_miss=400_000.0, seed=0):
    rng = np.random.default_rng(seed)
    ds = pd.date_range('2022-01-03', periods=n_q * 13, freq='W-MON')
    last = ig_is_last_week(ds)
    fc = 1e6 * (1 + 0.5 * np.sin(np.arange(len(ds)) * 0.7)) * np.where(last, 1.0, 0.8)
    act = fc + np.where(last, qe_miss, 0.0) + rng.normal(0, 5_000, len(ds))
    return pd.DataFrame({'ds': ds, 'actual': act, 'fc': fc})


def ig_run_self_tests():
    out = []
    wk = _ig_weekly()
    r = ig_qe_options_eval(wk, flat_usd=200_000, shrink_k=0.5, min_prior=3)
    assert r['selected'] == 'data_add', r
    out.append('QE: data-driven add wins on a constant +$400K quarter-end miss')
    # leakage trap: mutate the final two quarters wildly; earlier decisions must not move
    wk2 = wk.copy(); wk2.loc[wk2['ds'] >= pd.Timestamp('2025-07-01'), 'actual'] *= 5.0
    d1 = ig_qe_options_eval(wk, shrink_k=0.5)['detail']; d2 = ig_qe_options_eval(wk2, shrink_k=0.5)['detail']
    old = d1[d1['quarter'] < '2025Q3'].reset_index(drop=True); old2 = d2[d2['quarter'] < '2025Q3'].reset_index(drop=True)
    assert len(old) > 0 and np.allclose(old['last_wk_err'], old2['last_wk_err']), 'LEAK: future quarter moved an earlier decision'
    out.append('QE: leakage trap passed (future quarters do not change earlier decisions)')
    thin = ig_qe_options_eval(wk[wk['ds'] < '2023-01-01'])
    assert thin['selected'] == 'incumbent' and 'insufficient' in thin['reason']
    out.append('QE: insufficient history keeps the incumbent')
    # LightGBM weight: inject large LGB bias
    rng = np.random.default_rng(1); rows = []
    ds = pd.date_range('2024-01-01', periods=40, freq='W-MON'); act = 1e6 * (1 + 0.1 * rng.standard_normal(40))
    for oi in range(8):
        o = ds[oi * 4]
        for h in range(1, 5):
            j = oi * 4 + h
            if j >= 40: continue
            for m, bias in [('A', 1.0), ('B', 1.0), ('LGB', 0.7)]:
                rows.append({'model': m, 'origin_date': o.date(), 'ds': ds[j], 'horizon_week': h,
                             'actual': act[j], 'forecast': act[j] * bias * (1 + 0.02 * rng.standard_normal())})
    long = pd.DataFrame(rows); W = {'A': 0.4, 'B': 0.4, 'LGB': 0.2}
    bt = {m: ig_ensemble_wide(long, W, ['LGB'], m) for m in (1.0, 0.5, 0.0)}
    ho = {m: bt[m].assign(origin_date=1)[['ds', 'actual', 'ens']] for m in bt}
    lr = ig_lgb_weight_eval(bt, ho, margin_pp=0.1)
    assert lr['selected'] == 0.0, lr
    out.append('LGB: weight 0.0 selected when LGB carries a -30% bias')
    # level shift: step-up beats none, pure noise does not
    def _lv(step):
        rng = np.random.default_rng(2); d = pd.date_range('2023-01-02', periods=70, freq='W-MON')
        a = 1e6 * (1 + (0.2 if step else 0.0) * (np.arange(70) >= 30)) * (1 + 0.03 * rng.standard_normal(70))
        w = pd.DataFrame({'ds': d, 'actual': a, 'fc': 1e6})
        b = pd.DataFrame([{'origin_date': d[i].date(), 'ds': d[i + h], 'horizon_week': h, 'actual': a[i + h], 'ens': 1e6}
                          for i in range(8, 60) for h in range(1, 9)])
        b['origin_date'] = pd.to_datetime(b['origin_date'])
        return ig_level_eval(b, w, margin_pp=0.2)
    assert _lv(True)['adopt'] and not _lv(False)['adopt'], (_lv(True), _lv(False))
    out.append('LEVEL: correction adopted on a +20% step-up, rejected on pure noise')
    c = ig_pace_calc([100, 100], [80, 80], [80, 80, 100], [False, False, True], 0.1, k=3.0)
    assert abs(c['pace'] - 1.1) < 1e-9 and abs(c['landing'] - 497.0) < 1e-9, c
    out.append('PACE: hand-computed outlook 497.0 (pace 1.10) matches')
    wp = _ig_weekly(qe_miss=0.0); wp['actual'] = wp['fc'] * 1.2
    pe = ig_pace_eval(wp)['summary']
    assert bool(pe.loc[pe['asof_week'] == 8, 'validated'].iloc[0]), pe
    out.append('PACE: validated at as-of week 8 when actuals run 20% above the model')
    return out


def ig_run_self_tests_itsg():
    out = []
    nan = np.nan
    t = pd.DataFrame({'variant': ['full', 'no_mc', 'no_all3'], 'cv_wape': [30.0, 29.0, 31.0], 'cv_bias': [-8.0, -1.0, -9.0],
                      'val_wape': [25.0, 25.0, 24.0], 'val_bias': [-7.0, -1.0, -8.0], 'pool_wape': [28.0, 27.5, 28.5], 'pool_bias': [-8.0, -1.0, -9.0]})
    assert ig_ablation_select(t)[0] == 'no_mc', ig_ablation_select(t)
    t2 = t.copy(); t2.loc[1, ['pool_wape', 'pool_bias']] = [27.95, -7.5]
    assert ig_ablation_select(t2)[0] == 'full'
    t3 = t.copy(); t3.loc[1, 'val_wape'] = 27.0     # segment guard: validation worse by 2pp
    assert ig_ablation_select(t3)[0] != 'no_mc'
    out.append('ABLATION: reduced set adopted on bias+WAPE evidence; kept on marginal gain; segment guard works')
    det = pd.DataFrame({'quarter': ['a', 'a', 'b', 'b', 'c', 'c'], 'method': ['incumbent', 'data_add'] * 3,
                        'last_wk_err': [100, 10, 100, 10, 10, 100]})
    assert ig_qe_confirm(det, 'data_add')[0] == 'data_add'
    det.loc[det['quarter'] == 'b', 'last_wk_err'] = [10, 100]
    assert ig_qe_confirm(det, 'data_add')[0] == 'incumbent'
    out.append('QE: challenger must win >half of scored quarters')
    a, n = mira_attainment([100, 100, 100], [80, 80, 80], [True] * 3, N=3, k=2.0)
    assert abs(a - 0.88) < 1e-9 and n == 3, (a, n)
    a2, n2 = mira_attainment([100, 100, 100, 100], [80, nan, 80, 80], [True, False, True, True], N=3, k=2.0)
    assert abs(a2 - 0.88) < 1e-9 and n2 == 3, (a2, n2)       # unknown month excluded, NOT counted as zero
    a3, _ = mira_attainment([100, 100], [200, 200], [True, True], N=3, k=0.0, hi=1.5)
    assert a3 == 1.5
    out.append('MIRADORE: attainment 0.88 hand-computed; unknown month excluded not zero; clamp works')
    m = pd.DataFrame({'month': pd.period_range('2026-01', periods=8, freq='M'),
                      'plan': [100.0] * 8, 'actual': [80, 82, 78, 80, 81, 79, 80, 80.0]})
    m['known'] = True
    d1, s1 = mira_backtest(m)
    assert s1['attainment'] < s1['raw_plan'] and mira_choose(s1, len(d1))[0] in ('attainment', 'trail3')
    m2 = m.copy(); m2.loc[7, 'actual'] = 5000.0
    d2, _ = mira_backtest(m2)
    assert np.allclose(d1['attainment'].values[:-1], d2['attainment'].values[:-1]), 'LEAK in Miradore backtest'
    mu = m.copy(); mu.loc[3, 'actual'] = nan; mu.loc[3, 'known'] = False
    du, _ = mira_backtest(mu)
    assert pd.Period('2026-04', 'M') not in set(du['month']), 'unknown month must not be scored'
    assert mira_choose({'raw_plan': 5.0, 'attainment': 5.0, 'trail3': 6.0}, 5)[0] == 'raw_plan'
    assert mira_choose(s1, 2)[0] == 'raw_plan'
    out.append('MIRADORE: leakage trap passed; unknown month never scored; ties and thin history -> raw plan')
    w = mira_spread_weeks({pd.Period('2026-10', 'M'): 400.0}, pd.to_datetime(['2026-10-05', '2026-10-12']))
    assert np.allclose(w, [100.0, 100.0]), w
    out.append('MIRADORE: monthly outlook spread by Mondays (Oct-2026 has 4) matches')
    return out


if globals().get('IMPROVEMENT_GATES_ENABLED', True) and globals().get('RUN_IMPROVEMENT_SELF_TEST', True):
    try:
        for _m in ig_run_self_tests() + ig_run_self_tests_itsg():
            print('  PASS  ' + _m)
        print('Step 13H self-tests: ALL PASSED')
    except Exception as _exc_st:
        IMPROVEMENT_GATES_ENABLED = False
        print(f'*** Step 13H SELF-TEST FAILED ({type(_exc_st).__name__}: {str(_exc_st)[:300]}) ***')
        print('*** IMPROVEMENT_GATES_ENABLED forced to False -> notebook behaves exactly as before. ***')

In [ ]:
# STEP 1: LOAD ITSG FORECAST BASE DATA
# =============================================================================
# PURPOSE: Build the weekly modeling universe from raw SFDC data.  Produces:
#   * weekly_itsg            — total ITSG Growth ARR time series (forecast target)
#   * weekly_itsg_by_market  — per-market panel for LightGBM global training
#   * feature_source_total_hist — total-level feature frame for DHR-ARIMA
#
# SOURCE TABLE: datagroup.datalake_transform.cds_sfdc_opp_products_latest
#   One row per SFDC Opportunity Line Item.  Filters:
#   product_group='ITSG', purchase_type_rollup='Growth', is_won, is_closed,
#   sales_channel NOT IN ('Care','Sales Other'), close_date >= 2023-01-01
#
# KEY COLUMNS:
#   ds (week_start)           — Monday bucket of close_date (target grain)
#   sales_market              — geographic segment: NA, EMEA, APAC, LATAM, Unknown
#   arr (amount_towards_plan) — THE TARGET: weekly sum = y
#   deal_count                — number of closed OLIs per week (volume signal)
#
# FEATURE SOURCE: datagroup_mdl.mdl_sales_analytics.forecast_prophet
#   READ ONLY — supplies headcount, pipeline, deal size, cycle time features.
#   Its Actuals column is loaded as fp_actuals_reference for DIAGNOSTIC ONLY.
#
# CHANGED vs V4:
#   * Complete Monday calendar spine for total and week x market series
#     (zero weeks exist as $0 rows; V4 dropped them).
#   * forecast_prophet.Actuals loaded as fp_actuals_reference — DIAGNOSTIC
#     ONLY, never a target or feature. Target is always SFDC ATP.
#   * Total-level feature frame built on the spine with ffill (no gaps).
# UNCHANGED: business filters, ATP target definition, table names,
#   RSG-IT mapping for authoritative actuals.
# =============================================================================

import numpy as np
import pandas as pd

BASE_TABLE = 'datagroup.datalake_transform.cds_sfdc_opp_products_latest'
FEATURE_TABLE = 'datagroup_mdl.mdl_sales_analytics.forecast_prophet'          # READ ONLY
AUTHORITATIVE_ACTUALS_TABLE = 'datagroup.uc_forecast.actuals'                 # READ ONLY
PRODUCT_SCOPE = 'ITSG'
AUTHORITATIVE_ACTUALS_PRODUCT = 'RSG-IT'
START_DATE = '2023-01-01'
FIRST_FULL_MONDAY = pd.Timestamp('2023-01-02')
CURRENT_WEEK_START = pd.Timestamp.today().normalize() - pd.to_timedelta(pd.Timestamp.today().weekday(), unit='D')
LAST_CLOSED_WEEK = CURRENT_WEEK_START - pd.Timedelta(weeks=1)
MARKET_MAP = {'AUS/ROW': 'APAC', 'INDIA': 'APAC', 'UNKNOWN': 'Unknown', '': 'Unknown'}
GLOBAL_MARKETS = ['APAC', 'EMEA', 'LATAM', 'NA', 'Unknown']


def normalize_market(series: pd.Series) -> pd.Series:
    out = series.fillna('Unknown').astype(str).str.strip().str.upper()
    out = out.replace(MARKET_MAP)
    return out.replace({'UNKNOWN': 'Unknown'}).fillna('Unknown')


# FIX (2026-10-05): ported from UCC's Step 1 fix (2026-10-02/10-05). is_won /
# is_closed were quoted string literals ('True'/'True'), an exact-case match
# against a column this notebook's own boolean-parsing helpers elsewhere treat
# as arriving in multiple casings ('true'/'True'/'TRUE'/'1'); any row stored in
# a casing other than exactly 'True' was silently dropped from the target.
# Switched to unquoted boolean literals (is_won = true), which Spark casts
# case-insensitively against a string column, matching UCC's fixed query.
# purchase_type_rollup had the same bare case-sensitive match as UCC's Query A/B
# before that fix; wrapped in UPPER(COALESCE(...)) to match product_group's
# existing (correct) pattern just above it.
# --- Target query: closed-won ITSG Growth OLIs, ATP metric (unchanged scope) --
# Scope switch (2026-10-08): default keeps today's exclusion of 'Care' and 'Sales Other'.
ITSG_CHANNEL_FILTER_SQL = '' if globals().get('ITSG_INCLUDE_CARE_SALES_OTHER', False) else \
    "AND COALESCE(sales_channel, '') NOT IN ('Care', 'Sales Other')"
print(f"ITSG scope: Care/Sales Other {'INCLUDED' if not ITSG_CHANNEL_FILTER_SQL else 'EXCLUDED (default)'}")
itsg_raw = spark.sql(f"""
SELECT
    salesforce_opportunity_line_item_id,
    opportunity_product_id,
    account_id_18_char,
    owner_id,
    CAST(close_date AS DATE) AS close_date,
    DATE_TRUNC('WEEK', CAST(close_date AS DATE)) AS ds,
    UPPER(COALESCE(sales_market, 'Unknown')) AS sales_market,
    UPPER(COALESCE(product_group, 'Unknown')) AS product_group,
    purchase_type_rollup,
    sales_channel,
    CAST(amount_towards_plan AS DOUBLE) AS arr
FROM {BASE_TABLE}
WHERE is_won = true
  AND is_closed = true
  AND close_date >= DATE '{START_DATE}'
  AND close_date <= CURRENT_DATE()
  AND UPPER(COALESCE(product_group, '')) = '{PRODUCT_SCOPE}'
  AND UPPER(COALESCE(purchase_type_rollup, '')) = 'GROWTH'
  {ITSG_CHANNEL_FILTER_SQL}
ORDER BY ds
""").toPandas()

itsg_raw['ds'] = pd.to_datetime(itsg_raw['ds'])
itsg_raw['arr'] = pd.to_numeric(itsg_raw['arr'], errors='coerce').fillna(0.0)
itsg_raw['sales_market'] = normalize_market(itsg_raw['sales_market'])
itsg_raw = itsg_raw[itsg_raw['ds'] >= FIRST_FULL_MONDAY].copy()
assert len(itsg_raw) > 0, 'Target query returned no rows — check filters.'

# --- Complete Monday spine (CHANGED: V4 had no spine; zero weeks vanished) ----
calendar = pd.DataFrame({'ds': pd.date_range(FIRST_FULL_MONDAY, LAST_CLOSED_WEEK, freq='W-MON')})
ALL_MARKETS = sorted(set(itsg_raw['sales_market'].unique()) | set(GLOBAL_MARKETS))
market_calendar = (
    calendar.assign(_k=1)
    .merge(pd.DataFrame({'sales_market': ALL_MARKETS}).assign(_k=1), on='_k')
    .drop(columns='_k')
)

weekly_itsg = (
    calendar.merge(itsg_raw.groupby('ds', as_index=False).agg(y=('arr', 'sum'), deal_count=('arr', 'count')), on='ds', how='left')
    .sort_values('ds').reset_index(drop=True)
)
weekly_itsg[['y', 'deal_count']] = weekly_itsg[['y', 'deal_count']].fillna(0.0)
assert len(weekly_itsg) == len(calendar), 'weekly_itsg must have one row per spine week.'
assert weekly_itsg['y'].sum() > 0

# ── EARLY fiscal-year / quarter labels (Fix, 2026-09-28 cleanup pass) ────────
# Computed HERE (Step 1 -- the earliest point weekly_itsg exists) rather than in
# a later cell, because Databricks notebooks execute top-to-bottom sharing one
# namespace: any EARLIER cell that needs "this FY's Q1/Q2/etc." would otherwise
# have no CURRENT_FY to reference yet. Every downstream cell should use these
# instead of hardcoding a specific year/quarter literal, so the notebook keeps
# working unmodified once it runs past the current fiscal year.
# Fiscal year == calendar year in this notebook (quarters are calendar quarters
# via dt.to_period('Q') throughout).
CURRENT_FY = int(weekly_itsg['ds'].max().year)
CURRENT_FY_QUARTER = weekly_itsg['ds'].max().to_period('Q')
CURRENT_FY_LABEL = f'FY{CURRENT_FY}'
CURRENT_FY_Q1_LABEL = f'{CURRENT_FY}Q1'
CURRENT_FY_Q2_LABEL = f'{CURRENT_FY}Q2'
CURRENT_FY_Q3_LABEL = f'{CURRENT_FY}Q3'
CURRENT_FY_Q4_LABEL = f'{CURRENT_FY}Q4'
NEXT_FY = CURRENT_FY + 1
NEXT_FY_LABEL = f'FY{NEXT_FY}'
print(f'Fiscal-year labels (derived from data): CURRENT_FY={CURRENT_FY}, '
      f'CURRENT_FY_QUARTER={CURRENT_FY_QUARTER}, CURRENT_FY_LABEL={CURRENT_FY_LABEL}')

weekly_itsg_by_market = (
    market_calendar.merge(
        itsg_raw.groupby(['ds', 'sales_market'], as_index=False).agg(y=('arr', 'sum'), deal_count=('arr', 'count')),
        on=['ds', 'sales_market'], how='left',
    )
    .sort_values(['ds', 'sales_market']).reset_index(drop=True)
)
weekly_itsg_by_market[['y', 'deal_count']] = weekly_itsg_by_market[['y', 'deal_count']].fillna(0.0)
# Market series must reconcile exactly to the total series
_mkt_total = weekly_itsg_by_market.groupby('ds')['y'].sum().sort_index()
assert np.allclose(_mkt_total.values, weekly_itsg.sort_values('ds')['y'].values, atol=1.0)

# --- Feature source: forecast_prophet (FEATURES ONLY — Actuals is diagnostic) -
feature_source_all = spark.sql(f"""
SELECT
    CAST(ds AS DATE) AS ds,
    UPPER(COALESCE(sales_market, 'Unknown')) AS sales_market,
    CAST(Actuals AS DOUBLE) AS fp_actuals_reference,
    CAST(Forecast_Median AS DOUBLE) AS prophet_ml,
    CAST(all_created_arr AS DOUBLE) AS all_created_arr,
    CAST(headcount AS DOUBLE) AS headcount,
    CAST(avg_deal_size AS DOUBLE) AS avg_deal_size,
    CAST(avg_sales_cycle AS DOUBLE) AS avg_sales_cycle,
    CAST(avg_push_counter AS DOUBLE) AS avg_push_counter,
    CAST(avg_dials AS DOUBLE) AS avg_dials,
    CAST(avg_touches AS DOUBLE) AS avg_touches,
    CAST(avg_opportunity_age AS DOUBLE) AS avg_opportunity_age,
    CAST(total_partner_pipeline AS DOUBLE) AS total_partner_pipeline,
    CAST(magna_carta_actuals AS DOUBLE) AS magna_carta_actuals,
    CAST(miradore_plan AS DOUBLE) AS miradore_plan
FROM {FEATURE_TABLE}
WHERE UPPER(COALESCE(product, '')) = '{PRODUCT_SCOPE}'
  AND ds >= DATE '{START_DATE}'
ORDER BY ds, sales_market
""").toPandas()
feature_source_all['ds'] = pd.to_datetime(feature_source_all['ds'])
feature_source_all['sales_market'] = normalize_market(feature_source_all['sales_market'])
FP_FEATURE_COLS = ['prophet_ml', 'all_created_arr', 'headcount', 'avg_deal_size', 'avg_sales_cycle',
                   'avg_push_counter', 'avg_dials', 'avg_touches', 'avg_opportunity_age',
                   'total_partner_pipeline', 'magna_carta_actuals', 'miradore_plan']
# Collapse duplicate (ds, market) rows defensively
feature_source_all = feature_source_all.groupby(['ds', 'sales_market'], as_index=False).agg(
    {c: 'sum' for c in ['fp_actuals_reference'] + FP_FEATURE_COLS}
)
OBSERVED_END = weekly_itsg['ds'].max()

# --- Target-source consistency diagnostic (visible, permanent) ----------------
fp_total = (
    feature_source_all[feature_source_all['ds'] <= OBSERVED_END]
    .groupby('ds', as_index=False)['fp_actuals_reference'].sum()
)
target_consistency = weekly_itsg[['ds', 'y']].rename(columns={'y': 'sfdc_atp'}).merge(fp_total, on='ds', how='left')
target_consistency['diff_dollars'] = target_consistency['sfdc_atp'] - target_consistency['fp_actuals_reference']
_overlap = target_consistency.dropna(subset=['fp_actuals_reference'])
print('Target consistency (SFDC ATP vs forecast_prophet.Actuals — the latter is NEVER used as target):')
if len(_overlap):
    print(f"  Overlapping weeks: {len(_overlap):,} | SFDC total ${_overlap['sfdc_atp'].sum()/1e6:.2f}M "
          f"vs fp ${_overlap['fp_actuals_reference'].sum()/1e6:.2f}M "
          f"(ratio {_overlap['fp_actuals_reference'].sum()/max(_overlap['sfdc_atp'].sum(),1):.2f}x)")
    print(f"  Mean abs weekly diff: ${_overlap['diff_dollars'].abs().mean():,.0f}")
# Feature-table spine coverage (the V4 LGB failure mode — now just a diagnostic)
_fp_pairs = set(map(tuple, feature_source_all[feature_source_all['ds'] <= OBSERVED_END][['ds', 'sales_market']].values.tolist()))
_spine_pairs = set(map(tuple, market_calendar[['ds', 'sales_market']].values.tolist()))
print(f"  forecast_prophet market-week coverage vs full spine: {100.0 * len(_fp_pairs & _spine_pairs) / len(_spine_pairs):.1f}% "
      f"(v2 builds the LGB panel on the FULL spine, so gaps become ffilled features, not missing targets)")

# --- Total-level feature frame on the spine (for DHR, Step 5) -----------------
feature_source_total_hist = (
    calendar.merge(
        feature_source_all[feature_source_all['ds'] <= OBSERVED_END]
        .groupby('ds', as_index=False)
        .agg(
            prophet_ml=('prophet_ml', 'sum'),
            all_created_arr=('all_created_arr', 'sum'),
            headcount=('headcount', 'sum'),
            avg_deal_size=('avg_deal_size', 'mean'),
            avg_sales_cycle=('avg_sales_cycle', 'mean'),
            avg_push_counter=('avg_push_counter', 'mean'),
            avg_dials=('avg_dials', 'mean'),
            avg_touches=('avg_touches', 'mean'),
            avg_opportunity_age=('avg_opportunity_age', 'mean'),
            total_partner_pipeline=('total_partner_pipeline', 'sum'),
            magna_carta_actuals=('magna_carta_actuals', 'sum'),
            miradore_plan=('miradore_plan', 'sum'),
        ),
        on='ds', how='left',
    )
    .merge(weekly_itsg[['ds', 'y', 'deal_count']], on='ds', how='left')
    .sort_values('ds').reset_index(drop=True)
)
for col in FP_FEATURE_COLS:
    feature_source_total_hist[col] = pd.to_numeric(feature_source_total_hist[col], errors='coerce').ffill().fillna(0.0)
assert len(feature_source_total_hist) == len(calendar)

itsg_overview = pd.DataFrame([
    {'metric': 'weekly_rows', 'value': str(len(weekly_itsg))},
    {'metric': 'market_rows', 'value': str(len(weekly_itsg_by_market))},
    {'metric': 'history_start', 'value': str(weekly_itsg['ds'].min().date())},
    {'metric': 'history_end', 'value': str(OBSERVED_END.date())},
    {'metric': 'total_arr_m', 'value': str(round(weekly_itsg['y'].sum() / 1e6, 3))},
    {'metric': 'avg_weekly_arr_k', 'value': str(round(weekly_itsg['y'].mean() / 1e3, 1))},
    {'metric': 'zero_weeks_total_level', 'value': str(int((weekly_itsg['y'] == 0).sum()))},
])
display(itsg_overview)
print('\nWeekly ITSG tail:')
display(weekly_itsg.tail(5))
print('\nTarget consistency tail:')
display(target_consistency.tail(8))

In [ ]:
# =============================================================================
# STEP 1C: SCOPE DIAGNOSTIC -- ITSG weekly actuals with vs without 'Care' / 'Sales Other' (NEW 2026-10-08)
# =============================================================================
# Read-only, informational. Does NOT change the target (see ITSG_INCLUDE_CARE_SALES_OTHER, default False).
# Prints the quarterly difference so the business can decide whether the exclusion should stay.
if globals().get('RUN_SCOPE_DIAGNOSTIC', False):
    try:
        _sc = spark.sql(f"""
        SELECT DATE_TRUNC('WEEK', CAST(close_date AS DATE)) AS ds,
               CASE WHEN COALESCE(sales_channel, '') IN ('Care', 'Sales Other') THEN 'excluded_channels' ELSE 'in_scope' END AS bucket,
               SUM(CAST(amount_towards_plan AS DOUBLE)) AS arr
        FROM {BASE_TABLE}
        WHERE is_won = true AND is_closed = true
          AND close_date >= DATE '{START_DATE}' AND close_date <= CURRENT_DATE()
          AND UPPER(COALESCE(product_group, '')) = '{PRODUCT_SCOPE}'
          AND UPPER(COALESCE(purchase_type_rollup, '')) = 'GROWTH'
        GROUP BY 1, 2
        """).toPandas()
        _sc['ds'] = pd.to_datetime(_sc['ds'])
        _sc = _sc[(_sc['ds'] >= FIRST_FULL_MONDAY) & (_sc['ds'] <= LAST_CLOSED_WEEK)]
        _sc['quarter'] = _sc['ds'].dt.to_period('Q').astype(str)
        _pv = _sc.pivot_table(index='quarter', columns='bucket', values='arr', aggfunc='sum').fillna(0.0)
        for _c in ('in_scope', 'excluded_channels'):
            if _c not in _pv.columns:
                _pv[_c] = 0.0
        _pv['with_both'] = _pv['in_scope'] + _pv['excluded_channels']
        _pv['added_pct'] = _pv['excluded_channels'] / _pv['in_scope'].replace(0, np.nan) * 100
        print('=== 1C  ITSG Growth ARR by quarter: current scope vs Care + Sales Other added (USD) ===')
        print(_pv[['in_scope', 'excluded_channels', 'with_both', 'added_pct']].round(1).to_string())
        _wk = _sc.pivot_table(index='ds', columns='bucket', values='arr', aggfunc='sum').fillna(0.0)
        if 'in_scope' in _wk and 'excluded_channels' in _wk:
            print(f"  weekly: excluded channels average {(_wk['excluded_channels'] / _wk['in_scope'].replace(0, np.nan)).mean() * 100:.1f}% "
                  f"of in-scope weekly ARR. Current setting: ITSG_INCLUDE_CARE_SALES_OTHER={ITSG_INCLUDE_CARE_SALES_OTHER} (unchanged).")
    except Exception as _exc_sc:
        print(f'1C scope diagnostic skipped (fail-soft): {type(_exc_sc).__name__}: {str(_exc_sc)[:200]}')

In [ ]:
# STEP 1B: DATA FRESHNESS CHECK
# =============================================================================
# PURPOSE: Guard against stale or incomplete source data silently entering
# production.  ITSG is noisier than UCC (~$352K avg vs ~$1.1M), so the
# threshold is lower (15% vs UCC's implicit ~25%).
#
# Pass criteria: last_week_arr / trailing_8wk_avg >= 0.15
# Caution zone: ratio between 0.15 and 0.30 (printed warning, continues)
# Fail action: sets DATA_FRESH=False; downstream cells should respect this.
#
# For a series averaging $352.5K/week, an abnormally low last week likely
# indicates incomplete SFDC close-date stamping rather than a genuine dip.
# Threshold: < 15% of trailing 8-week average flags as potentially stale.
# =============================================================================

FRESHNESS_THRESHOLD = 0.15  # ITSG is noisier than UCC, so use a lower bar

last_week_y = float(weekly_itsg['y'].iloc[-1])
last_week_ds = weekly_itsg['ds'].iloc[-1]
r8w_avg = float(weekly_itsg['y'].iloc[-9:-1].mean())

ratio = last_week_y / r8w_avg if r8w_avg > 0 else 1.0

print(f"Data freshness check (ITSG):")
print(f"  Last closed week    : {last_week_ds.date()}")
print(f"  Last week ARR       : ${last_week_y:,.0f}")
print(f"  Trailing 8-wk avg   : ${r8w_avg:,.0f}")
print(f"  Ratio               : {ratio:.1%}")

if ratio < FRESHNESS_THRESHOLD:
    print(f"\n\u26a0\ufe0f  WARNING: Last week is only {ratio:.1%} of the 8-week average.")
    print(f"   This likely indicates an INCOMPLETE week in the data.")
    print(f"   Consider re-running after SFDC close-dates are fully stamped.")
    DATA_FRESH = False
elif ratio < 0.30:
    print(f"\n\u26a0\ufe0f  CAUTION: Last week is {ratio:.1%} of the 8-week average \u2014 unusually low for ITSG.")
    DATA_FRESH = True
else:
    print(f"\n\u2713 Data appears fresh and complete.")
    DATA_FRESH = True

### Step 2 — EDA and three-way backtest split **(CHANGED — validation slice added)**

The weekly plot and market mix as in V4, plus the structural change: a **three-way temporal split**.
CV region (through 13 weeks before the holdout) → **validation slice** (final 13 pre-holdout weeks,
2025-10-06 → 2025-12-29, used *only* for hyperparameter and inclusion decisions) → **holdout**
(`2026-01-05` onward, never trained on, never tuned against). V4 had only train/holdout, then made
several decisions (78-week window, ensemble weights, the 0.831 scale) using holdout information —
that path is closed here by construction.

In [ ]:
# STEP 2: ITSG EXPLORATORY DATA ANALYSIS AND THREE-WAY BACKTEST SPLIT
# =============================================================================
# PURPOSE: (1) Visualise the time series to build intuition about level,
# seasonality, and noise; (2) partition data into three non-overlapping regions
# with strict temporal separation.
#
# Split design:
#   CV region       — everything before VALIDATION_START; walk-forward CV lives here
#   Validation slice— 13 weeks (2025-10-06 → 2025-12-29); tuning ONLY
#   Holdout         — >= 2026-01-05; untouched until final leaderboard
#
# Outputs:
#   model_df, train_df, test_df, validation_df, cv_region_df
#   BACKTEST_START, VALIDATION_START constants
#
# CHANGED vs V4: adds the 13-week validation (tuning) slice between the CV
# region and the holdout. All tuning happens there and only there.
# =============================================================================

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


def millions(x, pos):
    return f'${x:,.2f}M'


BACKTEST_START = pd.Timestamp('2026-01-05')   # holdout boundary (unchanged)
VALIDATION_WEEKS = 13

model_df = weekly_itsg[['ds', 'y']].copy().sort_values('ds').reset_index(drop=True)
pre_holdout_weeks = model_df.loc[model_df['ds'] < BACKTEST_START, 'ds'].tolist()
assert len(pre_holdout_weeks) > VALIDATION_WEEKS + 52, 'Not enough history for the three-way split.'
VALIDATION_START = pre_holdout_weeks[-VALIDATION_WEEKS]

model_df['split'] = np.where(
    model_df['ds'] >= BACKTEST_START, 'holdout',
    np.where(model_df['ds'] >= VALIDATION_START, 'validation', 'cv_region')
)
train_df = model_df[model_df['ds'] < BACKTEST_START].copy()          # all pre-holdout (final retrain)
test_df = model_df[model_df['ds'] >= BACKTEST_START].copy()          # untouched holdout
validation_df = model_df[model_df['split'] == 'validation'].copy()   # tuning only
cv_region_df = model_df[model_df['split'] == 'cv_region'].copy()

assert len(validation_df) == VALIDATION_WEEKS
assert cv_region_df['ds'].max() < VALIDATION_START
assert validation_df['ds'].max() < BACKTEST_START

eda_summary = pd.DataFrame([
    {'metric': 'weeks', 'value': f"{len(weekly_itsg):,}"},
    {'metric': 'start_date', 'value': str(weekly_itsg['ds'].min().date())},
    {'metric': 'end_date', 'value': str(weekly_itsg['ds'].max().date())},
    {'metric': 'total_arr_m', 'value': f"{weekly_itsg['y'].sum() / 1e6:,.2f}"},
    {'metric': 'avg_weekly_arr_m', 'value': f"{weekly_itsg['y'].mean() / 1e6:,.3f}"},
    {'metric': 'weekly_cv_pct', 'value': f"{100.0 * weekly_itsg['y'].std() / max(weekly_itsg['y'].mean(), 1):,.1f}"},
    {'metric': 'markets', 'value': ', '.join(ALL_MARKETS)},
])
display(eda_summary)

split_summary = pd.DataFrame([
    {'split': name, 'rows': len(frame), 'start': str(frame['ds'].min().date()),
     'end': str(frame['ds'].max().date()), 'arr_m': round(frame['y'].sum() / 1e6, 3)}
    for name, frame in [('cv_region', cv_region_df), ('validation', validation_df), ('holdout', test_df)]
])
print('Three-way split (validation slice is for tuning ONLY — never reported as accuracy):')
display(split_summary)

total_market_actuals = (
    weekly_itsg_by_market.groupby('sales_market', as_index=False)
    .agg(total_arr=('y', 'sum'))
    .assign(pct=lambda d: (100 * d['total_arr'] / d['total_arr'].sum()).round(1))
    .sort_values('total_arr', ascending=False)
)
display(total_market_actuals)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(weekly_itsg['ds'], weekly_itsg['y'] / 1e6, linewidth=1.6, color='#1f77b4')
ax.axvline(VALIDATION_START, color='orange', ls='--', lw=1.2, label=f'validation start {VALIDATION_START.date()}')
ax.axvline(BACKTEST_START, color='red', ls='--', lw=1.2, label=f'holdout start {BACKTEST_START.date()}')
ax.set_title('ITSG weekly Growth ARR actuals (ATP) with split boundaries')
ax.set_ylabel('ARR ($M)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(millions))
ax.grid(alpha=0.2)
ax.legend(fontsize=9)
plt.show()

### Step 3 — Walk-forward validation framework **(CHANGED — clean folds, tunable training window)**

Expanding-window walk-forward CV built **only from the CV region**, with the fold count computed from
available history (5 folds x 13 weeks with the current data) and asserts that no fold's train or test
window touches the validation slice or holdout — V4's 8 hardcoded folds trained into the holdout.

Two further corrections. First, V4 hardcoded a 78-week rolling training window ("recency weighting")
justified by narrative, and applied it to the benchmarks but not to LightGBM. Here the window length is a
**per-model hyperparameter** (78 / 104 / all weeks), applied inside each model's fit function via
`tail(window)` so CV, validation, holdout, and production all use identical logic, and *chosen on the
validation slice* like every other setting. Second, all shared metric code lives here (WAPE/MAPE/MPE/
sMAPE/RMSE + interval coverage), with quantile-ordering and interval helpers carried over from V4 —
except that interval width calibration for the statistical models now comes from **CV residuals** (clean)
rather than holdout residuals.

In [ ]:
# STEP 3: WALK-FORWARD VALIDATION FRAMEWORK
# =============================================================================
# PURPOSE: Define walk-forward expanding-window CV folds and the validation-
# slice tuning helper.  Each fold trains on all data up to a cutoff, then
# tests on the next 13 weeks (one quarter).  Produces the fold specification
# and tune_on_validation() used by every model in Steps 4–6.
#
# Design:
#   * Expanding training window (each fold sees more history)
#   * Fixed 13-week test horizon (= 1 fiscal quarter)
#   * Minimum 79 weeks training ensures stable estimates
#   * All folds ENTIRELY within the CV region (pre-validation)
#   * tune_on_validation(): trains on CV region, scores on validation slice
#
# Outputs:
#   fold_specs     — DataFrame with fold boundaries
#   CV_FOLDS       — number of folds (5 with current history)
#   tune_on_validation() / best_candidate() — tuning helpers
#
# CHANGED vs V4:
#   * Folds built ONLY from cv_region_df; fold count computed; boundaries
#     asserted (V4: 8 hardcoded folds, folds 7-8 trained inside the holdout).
#   * Training-window length is a tunable per-model hyperparameter applied
#     via tail(window) inside fit functions (V4: hardcoded 78, inconsistently
#     applied).
#   * tune_on_validation(): the ONE place hyperparameters are chosen.
# UNCHANGED: metric definitions, quantile-order enforcement, calendar features.
# =============================================================================

import warnings
from dataclasses import dataclass

import numpy as np
# Compatibility shim: prophet 1.3.0 / cmdstanpy 1.3.0 use np.float_ which
# was removed in numpy 2.0. Restore as safe aliases for np.float64 / np.bytes_.
if not hasattr(np, 'float_'):
    np.float_ = np.float64
if not hasattr(np, 'string_'):
    np.string_ = np.bytes_

import holidays
import mlflow
from prophet import Prophet
from statsforecast import StatsForecast
from statsforecast.models import AutoETS, MSTL
from statsmodels.tsa.holtwinters import ExponentialSmoothing

warnings.filterwarnings('ignore')

CV_HORIZON = 13
CV_MIN_TRAIN_FLOOR = 79
N_CV_WEEKS = len(cv_region_df)
CV_FOLDS = int(min(6, (N_CV_WEEKS - CV_MIN_TRAIN_FLOOR) // CV_HORIZON))
assert CV_FOLDS >= 3, f'Only {CV_FOLDS} clean folds available — check split constants.'
MIN_TRAIN_WEEKS = N_CV_WEEKS - CV_FOLDS * CV_HORIZON
FULL_WINDOW = 10_000   # sentinel: use all available history
us_holidays = holidays.US()


def add_calendar_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['ds'] = pd.to_datetime(out['ds'])
    out['year'] = out['ds'].dt.year
    out['quarter'] = out['ds'].dt.quarter
    out['week_of_qtr'] = out.groupby(['year', 'quarter']).cumcount() + 1
    out['quarter_end_flag'] = (out['week_of_qtr'] >= 11).astype(int)
    out['is_q4'] = (out['quarter'] == 4).astype(int)
    out['q4_week_countdown'] = np.where(out['quarter'] == 4, 14 - out['week_of_qtr'], 0)
    out['woq_sin'] = np.sin(2 * np.pi * out['week_of_qtr'] / 13.0)
    out['woq_cos'] = np.cos(2 * np.pi * out['week_of_qtr'] / 13.0)
    out['qtr_sin'] = np.sin(2 * np.pi * out['quarter'] / 4.0)
    out['qtr_cos'] = np.cos(2 * np.pi * out['quarter'] / 4.0)
    out['us_holiday_binary'] = out['ds'].apply(lambda d: int(d.date() in us_holidays or (d + pd.Timedelta(days=1)).date() in us_holidays))
    return out


def ci_coverage(actual, lower, upper):
    """Fraction of actuals falling within [lower, upper]. Generic on the bound
    columns passed in — used for both the outer (p05/p95, nominal 90%) and
    inner (p25/p75, nominal 50%) bands under the 5-column interval convention."""
    actual = np.asarray(actual, dtype=float)
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)
    valid = np.isfinite(actual) & np.isfinite(lower) & np.isfinite(upper)
    if not valid.any():
        return np.nan
    return float(100.0 * np.mean((actual[valid] >= lower[valid]) & (actual[valid] <= upper[valid])))


def forecast_metrics(actual, forecast, p05=None, p95=None, p25=None, p75=None):
    """CHANGED: p10/p90 -> p05/p95 (outer, nominal 90% CI) plus new optional
    p25/p75 (inner, nominal 50% CI), matching the notebook-wide move from the
    3-column (p10/p50/p90) to the 5-column (p05/p25/p50/p75/p95) interval
    convention. Both coverage figures are optional and independent — a caller
    that only has the outer band still gets 'ci_coverage'."""
    actual = np.asarray(actual, dtype=float)
    forecast = np.asarray(forecast, dtype=float)
    assert actual.shape == forecast.shape, f'Metric shape mismatch: {actual.shape} vs {forecast.shape}'
    denom = np.clip(np.abs(actual).sum(), 1e-9, None)
    mask = np.abs(actual) > 1e-9
    wape = 100.0 * np.abs(actual - forecast).sum() / denom
    mape = 100.0 * np.mean(np.abs((actual[mask] - forecast[mask]) / actual[mask])) if mask.any() else np.nan
    mpe = 100.0 * np.mean((forecast[mask] - actual[mask]) / actual[mask]) if mask.any() else np.nan
    smape_denom = np.abs(actual) + np.abs(forecast)
    smape = np.nanmean(np.where(smape_denom > 1e-9, 200.0 * np.abs(actual - forecast) / smape_denom, np.nan))
    rmse = np.sqrt(np.nanmean((actual - forecast) ** 2))
    out = {'wape': float(wape), 'mape': float(mape), 'mpe': float(mpe), 'smape': float(smape), 'rmse': float(rmse)}
    out['ci_coverage'] = ci_coverage(actual, p05, p95) if p05 is not None and p95 is not None else np.nan
    out['ci_coverage_inner'] = ci_coverage(actual, p25, p75) if p25 is not None and p75 is not None else np.nan
    return out


@dataclass
class FoldSpec:
    fold: int
    train_start: pd.Timestamp
    train_end: pd.Timestamp
    test_start: pd.Timestamp
    test_end: pd.Timestamp
    train_rows: int
    test_rows: int


def build_walkforward_folds(df: pd.DataFrame, horizon: int = CV_HORIZON, n_folds: int = CV_FOLDS, min_train: int = MIN_TRAIN_WEEKS):
    df = df.sort_values('ds').reset_index(drop=True)
    specs = []
    for fold_idx in range(n_folds):
        train_end_idx = min_train + fold_idx * horizon
        test_end_idx = train_end_idx + horizon
        tr = df.iloc[:train_end_idx]
        te = df.iloc[train_end_idx:test_end_idx]
        specs.append(FoldSpec(fold_idx + 1, tr['ds'].min(), tr['ds'].max(), te['ds'].min(), te['ds'].max(), len(tr), len(te)))
    return specs


fold_specs = build_walkforward_folds(cv_region_df[['ds', 'y']])
fold_spec_df = pd.DataFrame([s.__dict__ for s in fold_specs])
# Fix guards: every fold (train AND test) ends before the validation slice
assert (pd.to_datetime(fold_spec_df['train_end']) < VALIDATION_START).all(), 'CV fold trains past VALIDATION_START!'
assert (pd.to_datetime(fold_spec_df['test_end']) < VALIDATION_START).all(), 'CV fold tests past VALIDATION_START!'
assert (pd.to_datetime(fold_spec_df['test_end']) < BACKTEST_START).all()


def run_walkforward_cv(model_name, fit_predict_fn, df, **kwargs):
    """Shared expanding-window CV. df must be the CV region only; asserted per fold.
    fit_predict_fn(train_hist, horizon, **kwargs) -> np.array of length horizon."""
    df = df.sort_values('ds').reset_index(drop=True)
    fold_rows, fold_preds = [], []
    for spec in fold_specs:
        train_fold = df[df['ds'] <= spec.train_end].copy()
        test_fold = df[(df['ds'] >= spec.test_start) & (df['ds'] <= spec.test_end)].copy()
        assert train_fold['ds'].max() < VALIDATION_START, f'{model_name} fold {spec.fold} trains past VALIDATION_START'
        assert test_fold['ds'].max() < BACKTEST_START, f'{model_name} fold {spec.fold} tests inside holdout'
        pred = np.asarray(fit_predict_fn(train_fold, horizon=len(test_fold), **kwargs), dtype=float)
        m = forecast_metrics(test_fold['y'].values, pred)
        fold_rows.append({'model': model_name, 'fold': spec.fold,
                          'train_start': str(train_fold['ds'].min().date()), 'train_end': str(train_fold['ds'].max().date()),
                          'test_start': str(test_fold['ds'].min().date()), 'test_end': str(test_fold['ds'].max().date()),
                          'train_rows': len(train_fold), 'test_rows': len(test_fold),
                          **{k: round(v, 3) for k, v in m.items() if k != 'ci_coverage'}})
        fold_preds.append(pd.DataFrame({'model': model_name, 'fold': spec.fold, 'ds': test_fold['ds'],
                                        'actual': test_fold['y'].values, 'forecast': pred}))
    fold_df = pd.DataFrame(fold_rows)
    summary = {'model': model_name, 'folds': len(fold_df)}
    for k in ['wape', 'mape', 'mpe', 'smape', 'rmse']:
        summary[f'mean_{k}'] = round(fold_df[k].mean(), 3)
        summary[f'std_{k}'] = round(fold_df[k].std(ddof=0), 3)
    return fold_df, pd.DataFrame([summary]), pd.concat(fold_preds, ignore_index=True)


VALIDATION_HORIZON = len(validation_df)
validation_actuals = validation_df['y'].to_numpy(dtype=float)


def tune_on_validation(model_name, fit_predict_fn, series_df, candidate_grid, fixed_kwargs=None):
    """Train on the CV region, score a 13-week forecast on the validation slice.
    The ONLY place hyperparameters are chosen. Failed candidates -> inf."""
    fixed_kwargs = fixed_kwargs or {}
    tuning_train = series_df[series_df['ds'] < VALIDATION_START].copy().reset_index(drop=True)
    tuning_valid = series_df[(series_df['ds'] >= VALIDATION_START) & (series_df['ds'] < BACKTEST_START)].copy()
    assert len(tuning_valid) == VALIDATION_HORIZON
    rows = []
    for candidate in candidate_grid:
        row = {'model': model_name}
        row.update({k: (str(v) if isinstance(v, (list, tuple, dict)) else v) for k, v in candidate.items()})
        try:
            pred = np.asarray(fit_predict_fn(tuning_train.copy(), horizon=VALIDATION_HORIZON, **candidate, **fixed_kwargs), dtype=float)
            m = forecast_metrics(tuning_valid['y'].values, pred)
            row.update({'val_wape': round(m['wape'], 3), 'val_mape': round(m['mape'], 3), 'val_mpe': round(m['mpe'], 3), 'error': ''})
        except Exception as exc:
            row.update({'val_wape': np.inf, 'val_mape': np.inf, 'val_mpe': np.nan, 'error': str(exc)[:160]})
        rows.append(row)
    result = pd.DataFrame(rows).sort_values('val_wape').reset_index(drop=True)
    if not np.isfinite(result['val_wape'].iloc[0]):
        print(f'\n⚠️  All {model_name} candidates failed. Errors:')
        for _, r in result.iterrows():
            if r.get('error'):
                print(f"  {r['error']}")
        raise RuntimeError(f'All {model_name} tuning candidates failed on the validation slice.')
    return result


def best_candidate(tuning_table, candidate_grid):
    best_idx, best_wape = None, np.inf
    for i, candidate in enumerate(candidate_grid):
        key = {k: (str(v) if isinstance(v, (list, tuple, dict)) else v) for k, v in candidate.items()}
        match = tuning_table
        for k, v in key.items():
            match = match[match[k] == v]
        if len(match) and np.isfinite(match['val_wape'].iloc[0]) and match['val_wape'].iloc[0] < best_wape:
            best_wape, best_idx = float(match['val_wape'].iloc[0]), i
    assert best_idx is not None
    return candidate_grid[best_idx]


def compute_woq_seasonal_profile(train_hist: pd.DataFrame) -> pd.DataFrame:
    profile_frame = add_calendar_columns(train_hist[['ds', 'y']].copy())
    profile = profile_frame.groupby('week_of_qtr', as_index=False).agg(woq_level=('y', 'mean'))
    scale = profile['woq_level'].mean()
    profile['woq_factor'] = np.where(scale > 1e-9, profile['woq_level'] / scale, 1.0)
    profile['woq_factor'] = profile['woq_factor'].replace([np.inf, -np.inf], np.nan).fillna(1.0)
    return profile[['week_of_qtr', 'woq_factor']]


def enforce_quantile_order(frame: pd.DataFrame, cols=('p05', 'p25', 'p50', 'p75', 'p95')) -> pd.DataFrame:
    """CHANGED: generalized from the fixed 3-column (p10, p50, p90) signature to
    an arbitrary ordered column list, defaulting to the new 5-column production
    convention (p05 <= p25 <= p50 <= p75 <= p95). Clips outward from the median
    so every column is non-negative and monotonically nested around it. Still
    works for a 3-column stack if a caller passes e.g. ['p05', 'p50', 'p95']."""
    out = frame.copy()
    cols = list(cols)
    for c in cols:
        out[c] = pd.to_numeric(out[c], errors='coerce').fillna(0.0).clip(lower=0.0)
    mid = len(cols) // 2
    for i in range(mid - 1, -1, -1):
        out[cols[i]] = np.minimum(out[cols[i]], out[cols[i + 1]])
    for i in range(mid + 1, len(cols)):
        out[cols[i]] = np.maximum(out[cols[i]], out[cols[i - 1]])
    return out


def derive_interval_adjustments(history_frame, actual_col='actual', forecast_col='forecast',
                                quantiles=(0.05, 0.25, 0.75, 0.95),
                                fallback=(-0.25, -0.10, 0.15, 0.30)):
    """CHANGED: was a 2-quantile (0.10/0.90) helper feeding p10/p90; now derives
    FOUR relative-error quantiles (0.05/0.25/0.75/0.95) from a residual frame,
    feeding the 5-column p05/p25/p50/p75/p95 convention. v2: call this with CV
    predictions (clean), not holdout predictions. Returns (p05_adj, p25_adj,
    p75_adj, p95_adj), each guaranteed non-decreasing and split around zero
    (the two lower adjustments <= 0, the two upper adjustments >= 0) so p50
    stays the median once applied."""
    if history_frame.empty:
        return fallback
    actual = pd.to_numeric(history_frame[actual_col], errors='coerce')
    forecast = pd.to_numeric(history_frame[forecast_col], errors='coerce')
    ratios = np.where(np.abs(forecast) > 1e-9, (actual - forecast) / forecast, np.nan)
    ratios = pd.Series(ratios).replace([np.inf, -np.inf], np.nan).dropna().clip(lower=-0.9, upper=3.0)
    if ratios.empty:
        return fallback
    raw = [float(np.nanquantile(ratios, q)) for q in quantiles]
    # p05, p25 adjustments must be <= 0; p75, p95 adjustments must be >= 0
    adj = [min(raw[0], 0.0), min(raw[1], 0.0), max(raw[2], 0.0), max(raw[3], 0.0)]
    # enforce monotonic nesting: p05_adj <= p25_adj <= p75_adj <= p95_adj
    for i in range(1, len(adj)):
        if adj[i] < adj[i - 1]:
            adj[i] = adj[i - 1]
    if adj[0] > adj[-1]:
        return fallback
    return tuple(adj)


def apply_interval_adjustments(forecast_frame, p05_adj, p25_adj, p75_adj, p95_adj, forecast_col='forecast'):
    """CHANGED: was a 2-adjustment (lower_adj/upper_adj) helper writing
    p10/p50/p90; now takes FOUR relative adjustments and writes the 5-column
    p05/p25/p50/p75/p95 convention."""
    out = forecast_frame.copy()
    out['forecast'] = pd.to_numeric(out[forecast_col], errors='coerce').fillna(0.0).clip(lower=0.0)
    out['p05'] = out['forecast'] * (1.0 + p05_adj)
    out['p25'] = out['forecast'] * (1.0 + p25_adj)
    out['p50'] = out['forecast']
    out['p75'] = out['forecast'] * (1.0 + p75_adj)
    out['p95'] = out['forecast'] * (1.0 + p95_adj)
    return enforce_quantile_order(out, ['p05', 'p25', 'p50', 'p75', 'p95'])


print(f'CV region weeks: {N_CV_WEEKS} | folds: {CV_FOLDS} | horizon: {CV_HORIZON} | minimum train: {MIN_TRAIN_WEEKS}')
print(f'Boundaries: CV end {cv_region_df["ds"].max().date()} < validation {VALIDATION_START.date()} < holdout {BACKTEST_START.date()}')
print(f'Validation slice actual: ${validation_df["y"].sum()/1e6:.3f}M over {VALIDATION_HORIZON} weeks')
print('\nFold specification (all strictly pre-validation, pre-holdout):')
display(fold_spec_df)

### Step 4 — Benchmark models: Prophet, MSTL, ETS **(CHANGED — tuned grids, clean CV/holdout)**

Same three statistical baselines as V4, with each model's settings — including the training-window
length — selected on the validation slice: Prophet (`changepoint_prior_scale` x window), ETS
(window x seasonal type, damped trend), MSTL (seasonal periods x window; the 52-week component requires
≥104 training weeks so short windows are excluded for it). All forecasts are clipped at zero **inside the
fit functions**, so CV, holdout, and production apply identical post-processing. CV runs on the clean CV
region via the shared framework; holdout forecasts retrain on all pre-holdout data with the frozen tuned
settings. No expected-accuracy numbers appear in comments — V4's hand-typed figures did not match its own
output.

In [ ]:
# STEP 4: BENCHMARK MODELS AND DIAGNOSTICS
# =============================================================================
# PURPOSE: Fit three statistical baseline models on the CV region, score them
# via walk-forward CV, then retrain on all pre-holdout data for the holdout
# evaluation.  These three form the core of the Adaptive Ensemble (Step 7).
#
# Models trained here:
#   Prophet_trend — Meta Prophet with custom quarterly Fourier seasonality
#                   and empirical WoQ seasonal overlay (no built-in seasonality)
#   MSTL_v2       — STL decomposition + AutoETS on remainder (statsforecast)
#   ETS           — Holt-Winters Exponential Smoothing (statsmodels)
#
# Outputs:
#   cv_benchmark_summary       — per-model CV WAPE across folds
#   holdout_benchmark_predictions — weekly holdout forecasts for all models
#   holdout_benchmark_summary  — leaderboard (WAPE, bias, directional accuracy)
#   BEST_*_PARAMS              — frozen hyperparameters for production
#
# CHANGED vs V4:
#   * Hyperparameters (incl. training window) tuned on the validation slice.
#     V4 hardcoded cps=0.05 and a 78-week window chosen from holdout-era
#     reasoning.
#   * CV on the clean CV region via the shared framework (Fix: no holdout
#     contact).
#   * Non-negativity clip inside each fit function (identical in CV /
#     holdout / production).
# UNCHANGED: model lineup and names, WoQ seasonal overlay for Prophet,
#   MLflow logging, holdout plot, quarterly comparison.
# =============================================================================

def fit_predict_prophet_trend(train_hist, horizon, changepoint_prior_scale=0.05, window=FULL_WINDOW):
    hist = train_hist[['ds', 'y']].tail(min(window, len(train_hist))).copy()
    future_dates = pd.DataFrame({'ds': pd.date_range(hist['ds'].max() + pd.Timedelta(weeks=1), periods=horizon, freq='W-MON')})
    future_input = add_calendar_columns(future_dates)
    woq_profile = compute_woq_seasonal_profile(hist)
    future_input = future_input.merge(woq_profile, on='week_of_qtr', how='left')
    future_input['woq_factor'] = future_input['woq_factor'].fillna(1.0)
    model = Prophet(
        yearly_seasonality=False, weekly_seasonality=False, daily_seasonality=False,
        seasonality_mode='additive',
        changepoint_prior_scale=changepoint_prior_scale,
        changepoint_range=0.85,
        n_changepoints=min(8, max(1, len(hist) // 20)),
    )
    model.fit(hist)
    fcst = model.predict(future_dates)
    # Non-negativity inside the predict path (identical everywhere)
    return np.clip(fcst['yhat'].to_numpy() * future_input['woq_factor'].to_numpy(), 0.0, None)


def fit_predict_mstl_v2(train_hist, horizon, season_length=(13, 52), window=FULL_WINDOW):
    hist = train_hist[['ds', 'y']].tail(min(window, len(train_hist))).copy()
    hist.insert(0, 'unique_id', 'itsg_total')
    sf = StatsForecast(models=[MSTL(season_length=list(season_length), trend_forecaster=AutoETS(model='ZZN'))], freq='W-MON', n_jobs=1)
    fcst = sf.forecast(df=hist, h=horizon)
    model_col = [c for c in fcst.columns if c not in {'unique_id', 'ds'}][0]
    return np.clip(fcst[model_col].to_numpy(dtype=float), 0.0, None)


def fit_predict_ets(train_hist, horizon, window=FULL_WINDOW, seasonal='mul', damped=True):
    tail = train_hist[['ds', 'y']].tail(min(window, len(train_hist))).copy()
    values = pd.to_numeric(tail['y'], errors='coerce').fillna(0.0)
    if seasonal == 'mul':
        values = values.clip(lower=1.0)   # multiplicative seasonality needs strictly positive values
    # Attach a proper DatetimeIndex with freq so statsmodels can extend the
    # prediction index (required since statsmodels 0.14.2+).
    values.index = pd.DatetimeIndex(tail['ds'].values, freq='W-MON')
    seasonal_periods = 52 if len(values) >= 104 else 13
    fitted = ExponentialSmoothing(
        values, trend='add', damped_trend=damped, seasonal=seasonal,
        seasonal_periods=seasonal_periods, initialization_method='estimated',
    ).fit(optimized=True, use_brute=True)
    return np.clip(fitted.forecast(horizon).to_numpy(dtype=float), 0.0, None)


# --- Grids, tuned ONLY on the validation slice ---------------------------------
prophet_grid = [{'changepoint_prior_scale': cps, 'window': w} for cps in (0.03, 0.05, 0.10) for w in (78, FULL_WINDOW)]
ets_grid = [{'window': w, 'seasonal': s} for w in (78, 104, FULL_WINDOW) for s in ('add', 'mul')]
mstl_grid = [{'season_length': sl, 'window': w} for sl in ((13, 52), (4, 13, 52)) for w in (104, FULL_WINDOW)]

series_total = model_df[['ds', 'y']].copy()
prophet_tuning = tune_on_validation('Prophet_trend', fit_predict_prophet_trend, series_total, prophet_grid)
ets_tuning = tune_on_validation('ETS', fit_predict_ets, series_total, ets_grid)
mstl_tuning = tune_on_validation('MSTL_v2', fit_predict_mstl_v2, series_total, mstl_grid)

BEST_PROPHET_PARAMS = best_candidate(prophet_tuning, prophet_grid)
BEST_ETS_PARAMS = best_candidate(ets_tuning, ets_grid)
BEST_MSTL_PARAMS = best_candidate(mstl_tuning, mstl_grid)

print('Tuned benchmark hyperparameters (frozen from validation slice):')
print(f'  Prophet_trend : {BEST_PROPHET_PARAMS}')
print(f'  ETS           : {BEST_ETS_PARAMS}')
print(f'  MSTL_v2       : {BEST_MSTL_PARAMS}')
print('\nProphet tuning table:')
display(prophet_tuning)
print('\nETS tuning table:')
display(ets_tuning)
print('\nMSTL tuning table:')
display(mstl_tuning)

# --- Clean CV + holdout ---------------------------------------------------------
benchmark_model_specs = [
    ('Prophet_trend', fit_predict_prophet_trend, BEST_PROPHET_PARAMS),
    ('MSTL_v2', fit_predict_mstl_v2, BEST_MSTL_PARAMS),
    ('ETS', fit_predict_ets, BEST_ETS_PARAMS),
]

cv_fold_frames, cv_summary_frames, cv_pred_frames = [], [], []
holdout_pred_frames, holdout_rows = [], []
for model_name, fit_fn, kwargs in benchmark_model_specs:
    fold_df, summary_df, pred_df = run_walkforward_cv(model_name, fit_fn, cv_region_df[['ds', 'y']].copy(), **kwargs)
    cv_fold_frames.append(fold_df)
    cv_summary_frames.append(summary_df)
    cv_pred_frames.append(pred_df)

    holdout_pred = np.asarray(fit_fn(train_df[['ds', 'y']].copy(), horizon=len(test_df), **kwargs), dtype=float)
    holdout_pred_frames.append(pd.DataFrame({'model': model_name, 'ds': test_df['ds'].values,
                                             'actual': test_df['y'].values, 'forecast': holdout_pred}))
    hm = forecast_metrics(test_df['y'].values, holdout_pred)
    holdout_rows.append({'model': model_name, 'test_start': str(test_df['ds'].min().date()),
                         'test_end': str(test_df['ds'].max().date()), 'test_rows': len(test_df),
                         **{k: round(v, 3) for k, v in hm.items() if k != 'ci_coverage'}})

    with mlflow.start_run(run_name=f'itsg_benchmark_{model_name.lower()}_v2', nested=False):
        mlflow.log_param('model_name', model_name)
        mlflow.log_param('product_group', PRODUCT_SCOPE)
        mlflow.log_param('cv_folds', CV_FOLDS)
        mlflow.log_param('cv_region_end', str(cv_region_df['ds'].max().date()))
        mlflow.log_param('validation_slice', f'{VALIDATION_START.date()}..{validation_df["ds"].max().date()}')
        mlflow.log_param('tuned_params', str(kwargs))
        for k, v in summary_df.iloc[0].to_dict().items():
            if k not in ('model',) and pd.notna(v):
                mlflow.log_metric(str(k), float(v))
        for k in ['wape', 'mape', 'mpe', 'smape', 'rmse']:
            mlflow.log_metric(f'holdout_{k}', float(holdout_rows[-1][k]))

cv_fold_metrics = pd.concat(cv_fold_frames, ignore_index=True)
cv_benchmark_summary = pd.concat(cv_summary_frames, ignore_index=True).sort_values('mean_wape').reset_index(drop=True)
cv_fold_predictions = pd.concat(cv_pred_frames, ignore_index=True)
holdout_benchmark_predictions = pd.concat(holdout_pred_frames, ignore_index=True)
holdout_benchmark_summary = pd.DataFrame(holdout_rows).sort_values('wape').reset_index(drop=True)

print('\nBenchmark walk-forward summary (clean CV region):')
display(cv_benchmark_summary)
print(f'\nBenchmark holdout summary ({CURRENT_FY_Q1_LABEL} onward, never tuned against):')
display(holdout_benchmark_summary)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(model_df['ds'], model_df['y'] / 1e6, color='black', linewidth=1.8, label='Actual')
for model_name in ['Prophet_trend', 'MSTL_v2', 'ETS']:
    tmp = holdout_benchmark_predictions[holdout_benchmark_predictions['model'] == model_name]
    ax.plot(tmp['ds'], tmp['forecast'] / 1e6, linewidth=1.6, ls='--', label=model_name)
ax.set_title('ITSG holdout: actual versus benchmark forecasts')
ax.set_ylabel('ARR ($M)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(millions))
ax.grid(alpha=0.2)
ax.legend()
plt.show()

# Quarterly comparison — quarters via to_period (V4 mixed three Q1 definitions)
quarterly_compare = holdout_benchmark_predictions.copy()
quarterly_compare['quarter'] = quarterly_compare['ds'].dt.to_period('Q').astype(str)
quarterly_compare = (
    quarterly_compare.groupby(['model', 'quarter'], as_index=False)
    .agg(weeks=('ds', 'nunique'), actual_arr=('actual', 'sum'), forecast_arr=('forecast', 'sum'))
)
quarterly_compare['actual_arr_m'] = (quarterly_compare['actual_arr'] / 1e6).round(3)
quarterly_compare['forecast_arr_m'] = (quarterly_compare['forecast_arr'] / 1e6).round(3)
quarterly_compare['pct_error'] = np.where(
    quarterly_compare['actual_arr'].abs() > 1e-9,
    (100.0 * (quarterly_compare['forecast_arr'] - quarterly_compare['actual_arr']) / quarterly_compare['actual_arr']).round(2),
    np.nan,
)
display(quarterly_compare.sort_values(['quarter', 'model']))
last_q = weekly_itsg['ds'].max().to_period('Q')
if int((weekly_itsg['ds'].dt.to_period('Q') == last_q).sum()) < 13:
    print(f'Note: {last_q} is quarter-to-date through {weekly_itsg["ds"].max().date()}.')

### Step 5 — Benchmark model: DHR-ARIMA **(CHANGED — training-only future regressors, tuned order, fallback)**

Dynamic Harmonic Regression + AutoARIMA at the total level (as in V4 — ITSG's markets are too thin for
per-market ARIMA), with Fourier terms and external regressors (created pipeline, headcount, pipeline
velocity, win rate, quarter-end and holiday flags) built on the complete spine. Changes: (1) **future
regressors come from training-only week-of-quarter median profiles** — V4 fed actual future
`forecast_prophet` rows into the forecast horizon, and its future win-rate silently collapsed to zero
because future `y` is NaN; (2) the Fourier harmonic order and training window are tuned on the validation
slice; (3) AutoARIMA is wrapped with a seasonal-naive fallback; (4) forecasts are clipped at zero inside
the predict function. Win rate remains a backward-shifted rolling ratio of won ARR to created pipeline —
`all_created_arr` is an all-outcome series, so there is no survivorship issue.

In [ ]:
# STEP 5: BENCHMARK MODEL — DHR-ARIMA (TOTAL LEVEL)
# ============================================================================
# NOTE: DHR_ARIMA is ITSG's worst performer (holdout WAPE 39.7%, CV WAPE
# 40.3%) and is excluded from the ensemble by Step 6B. It is retained here
# for diagnostic/comparison purposes only. To save ~2 min of compute in
# production, skip this cell entirely — Step 6B and Step 7 do not depend on it.
# ============================================================================
# =============================================================================
# PURPOSE: Fit Dynamic Harmonic Regression + AutoARIMA at the total level
# (ITSG's markets are too thin for per-market ARIMA), with Fourier terms
# and external regressors, then evaluate via CV and holdout.
#
# Model design:
#   Total-level only (all markets summed) — unlike UCC which fits per-market
#   Fourier seasonality (period=13, tuned K)
#   External regressors: created_pipeline_lag4, headcount_lag4,
#     pipeline_velocity_lag4, win_rate_woq, quarter_end_flag, us_holiday
#   Future regressors: WoQ median profiles from training only (no lookahead)
#   Fallback: seasonal naive if AutoARIMA fails to converge
#
# CHANGED vs V4:
#   * Future xreg = training-only WoQ median profiles (V4 used future
#     forecast_prophet values and a future win-rate that collapsed to 0).
#   * Fourier order and training window tuned on the validation slice.
#   * try/except seasonal-naive fallback; non-negativity clip inside predict.
#   * CV via the shared clean framework.
# =============================================================================

from statsforecast.models import AutoARIMA

DHR_MODEL_NAME = 'DHR_ARIMA'
DHR_BASE_XCOLS = ['created_lag4', 'headcount_lag4', 'pipeline_velocity_lag4',
                  'quarter_end_flag', 'us_holiday_binary', 'win_rate_woq']
DHR_MAX_FOURIER_K = 2


def dhr_fourier_cols(k):
    cols = []
    for order in range(1, k + 1):
        for period in (13, 52):
            cols += [f'sin_{period}_{order}', f'cos_{period}_{order}']
    return cols


def add_dhr_fourier(df):
    out = df.copy()
    woy = pd.to_datetime(out['ds']).dt.isocalendar().week.astype(int).values
    woq = out['week_of_qtr'].values
    for order in range(1, DHR_MAX_FOURIER_K + 1):
        for period in (13, 52):
            idx = woq if period == 13 else woy
            out[f'sin_{period}_{order}'] = np.sin(2 * np.pi * order * (idx - 1) / period)
            out[f'cos_{period}_{order}'] = np.cos(2 * np.pi * order * (idx - 1) / period)
    return out


def build_total_dhr_frame() -> pd.DataFrame:
    total = feature_source_total_hist.copy().sort_values('ds').reset_index(drop=True)
    total['pipeline_velocity'] = (total['all_created_arr'] / total['headcount'].replace(0, np.nan)).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    total['created_lag4'] = total['all_created_arr'].shift(4).fillna(0.0)
    total['headcount_lag4'] = total['headcount'].shift(4).bfill().fillna(0.0)
    total['pipeline_velocity_lag4'] = total['pipeline_velocity'].shift(4).fillna(0.0)
    # Backward-shifted rolling conversion: won ARR / created pipeline (all-outcome)
    _won_roll = total['y'].shift(1).rolling(4, min_periods=1).sum()
    _created_roll = total['all_created_arr'].shift(1).rolling(4, min_periods=1).sum().replace(0, np.nan)
    total['win_rate_woq'] = (_won_roll / _created_roll).clip(0.0, 3.0).fillna(0.0)
    total = add_calendar_columns(total)
    total = add_dhr_fourier(total)
    all_cols = dhr_fourier_cols(DHR_MAX_FOURIER_K) + DHR_BASE_XCOLS
    total[all_cols] = total[all_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return total


def build_future_dhr_xreg(train_feat: pd.DataFrame, horizon: int) -> pd.DataFrame:
    # CHANGED (leak-safety): dynamic regressors for future weeks come from
    # training-only WoQ median profiles, never from future table values.
    future = pd.DataFrame({'ds': pd.date_range(train_feat['ds'].max() + pd.Timedelta(weeks=1), periods=horizon, freq='W-MON')})
    future = add_calendar_columns(future)
    future = add_dhr_fourier(future)
    lookback = train_feat.sort_values('ds').tail(104)
    for col in ['created_lag4', 'headcount_lag4', 'pipeline_velocity_lag4', 'win_rate_woq']:
        profile = lookback.groupby('week_of_qtr')[col].median().to_dict()
        default = float(lookback[col].median()) if len(lookback) else 0.0
        future[col] = [profile.get(wk, default) for wk in future['week_of_qtr']]
    future['quarter_end_flag'] = future['quarter_end_flag'].astype(float)
    all_cols = dhr_fourier_cols(DHR_MAX_FOURIER_K) + DHR_BASE_XCOLS
    future[all_cols] = future[all_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return future


def fit_predict_dhr_arima(train_hist, horizon, full_feature_df, fourier_k=2, window=FULL_WINDOW):
    x_cols = dhr_fourier_cols(fourier_k) + DHR_BASE_XCOLS
    train_end = pd.to_datetime(train_hist['ds']).max()
    train_feat = full_feature_df[full_feature_df['ds'] <= train_end].copy().tail(min(window, len(full_feature_df)))
    future_feat = build_future_dhr_xreg(train_feat, horizon)
    train_input = train_feat[['ds', 'y'] + x_cols].copy()
    train_input.insert(0, 'unique_id', 'itsg_total')
    future_input = future_feat[['ds'] + x_cols].copy()
    future_input.insert(0, 'unique_id', 'itsg_total')
    try:
        sf = StatsForecast(models=[AutoARIMA(season_length=13)], freq='W-MON', n_jobs=1)
        fcst = sf.forecast(df=train_input, h=horizon, X_df=future_input)
        model_col = [c for c in fcst.columns if c not in {'unique_id', 'ds'}][0]
        pred = fcst[model_col].to_numpy(dtype=float)
    except Exception as exc:
        print(f'WARNING: AutoARIMA failed ({type(exc).__name__}: {exc}); using seasonal-naive fallback.')
        hist = train_input.sort_values('ds')['y'].to_numpy(dtype=float)
        season = hist[-13:] if len(hist) >= 13 else hist
        pred = np.tile(season, int(np.ceil(horizon / max(1, len(season)))))[:horizon]
    # Non-negativity inside the predict path (identical everywhere)
    return np.clip(pred, 0.0, None)


dhr_df = build_total_dhr_frame()
assert len(dhr_df) == len(weekly_itsg)
assert np.allclose(dhr_df['y'].values, weekly_itsg.sort_values('ds')['y'].values, atol=1.0)

# --- Tune on the validation slice ----------------------------------------------
dhr_grid = [{'fourier_k': k, 'window': w} for k in (1, 2) for w in (104, FULL_WINDOW)]
dhr_tuning = tune_on_validation(DHR_MODEL_NAME, fit_predict_dhr_arima, dhr_df[['ds', 'y']].copy(), dhr_grid,
                                fixed_kwargs={'full_feature_df': dhr_df})
BEST_DHR_PARAMS = best_candidate(dhr_tuning, dhr_grid)
print(f'Tuned DHR params: {BEST_DHR_PARAMS}')
display(dhr_tuning)

# --- Clean CV + holdout ----------------------------------------------------------
dhr_fold_df, dhr_summary_df, dhr_pred_df = run_walkforward_cv(
    DHR_MODEL_NAME, fit_predict_dhr_arima, cv_region_df[['ds', 'y']].copy(),
    full_feature_df=dhr_df, **BEST_DHR_PARAMS,
)
dhr_holdout_pred = np.asarray(
    fit_predict_dhr_arima(train_df[['ds', 'y']].copy(), horizon=len(test_df), full_feature_df=dhr_df, **BEST_DHR_PARAMS),
    dtype=float,
)
dhr_hm = forecast_metrics(test_df['y'].values, dhr_holdout_pred)
dhr_holdout_row = pd.DataFrame([{'model': DHR_MODEL_NAME, 'test_start': str(test_df['ds'].min().date()),
                                 'test_end': str(test_df['ds'].max().date()), 'test_rows': len(test_df),
                                 **{k: round(v, 3) for k, v in dhr_hm.items() if k != 'ci_coverage'}}])
dhr_holdout_predictions = pd.DataFrame({'model': DHR_MODEL_NAME, 'ds': test_df['ds'].values,
                                        'actual': test_df['y'].values, 'forecast': dhr_holdout_pred})

cv_fold_metrics = pd.concat([cv_fold_metrics[cv_fold_metrics['model'] != DHR_MODEL_NAME], dhr_fold_df], ignore_index=True)
cv_benchmark_summary = pd.concat([cv_benchmark_summary[cv_benchmark_summary['model'] != DHR_MODEL_NAME], dhr_summary_df],
                                 ignore_index=True).sort_values('mean_wape').reset_index(drop=True)
cv_fold_predictions = pd.concat([cv_fold_predictions[cv_fold_predictions['model'] != DHR_MODEL_NAME], dhr_pred_df], ignore_index=True)
holdout_benchmark_summary = pd.concat([holdout_benchmark_summary[holdout_benchmark_summary['model'] != DHR_MODEL_NAME], dhr_holdout_row],
                                      ignore_index=True).sort_values('wape').reset_index(drop=True)
holdout_benchmark_predictions = pd.concat([holdout_benchmark_predictions[holdout_benchmark_predictions['model'] != DHR_MODEL_NAME],
                                           dhr_holdout_predictions], ignore_index=True)

with mlflow.start_run(run_name='itsg_benchmark_dhr_arima_v2', nested=False):
    mlflow.log_param('model_name', DHR_MODEL_NAME)
    mlflow.log_param('product_group', PRODUCT_SCOPE)
    mlflow.log_param('level', 'total')
    mlflow.log_param('tuned_params', str(BEST_DHR_PARAMS))
    mlflow.log_param('future_xreg_strategy', 'training_only_woq_profiles')
    for k, v in dhr_summary_df.iloc[0].to_dict().items():
        if k != 'model' and pd.notna(v):
            mlflow.log_metric(str(k), float(v))
    for k in ['wape', 'mape', 'mpe', 'smape', 'rmse']:
        mlflow.log_metric(f'holdout_{k}', float(dhr_holdout_row.iloc[0][k]))

print('\nUpdated CV summary including DHR-ARIMA:')
display(cv_benchmark_summary)
print('\nUpdated holdout summary:')
display(holdout_benchmark_summary)

### Step 6 — Global LightGBM quantile model **(CHANGED — complete-spine panel, deterministic, tuned, gated)**

One gradient-boosted quantile model (p05/p25/p50/p75/p95) trained across all five markets, with recursive
week-by-week prediction (V4's recursive design is kept — it correctly matches production inference).
The substantive fixes:

- **Panel rebuilt on the complete week x market spine.** The target comes from SFDC ATP on the full grid
  (zero weeks are $0, not missing); `forecast_prophet` columns are left-joined as features and
  forward-filled within each market. V4 used `forecast_prophet`'s rows as the spine (~84% coverage), which
  punched holes in the target and lag features and produced its ~120% holdout WAPE.
- **Deterministic training** (`deterministic=true`, `force_row_wise`, single thread, fixed seed) so
  reruns on identical data reproduce identical results.
- **Recursive prediction keeps 60 weeks of history**, so `y_lag26`/`y_lag52` stay populated (V4 kept 26
  weeks, zeroing those lags at prediction time only — a train/serve skew).
- **Evaluation at the weekly total level** with the shared metric code, so LGB is directly comparable to
  the other four models (V4 scored it per market row, where near-zero actuals make MAPE meaningless);
  market-level WAPE is reported as a diagnostic.
- **Hyperparameters tuned on the validation slice** (leaves x learning rate x min-leaf), CV folds
  replicated exactly from the shared specification (asserted).
- **Data-driven ensemble promotion gate:** LGB joins the production ensemble only if its validation-slice
  WAPE is within 1.25x the best benchmark's — the decision is computed and printed, not hardcoded.
  V4's dead-code features (`deal_count_lag1`, `qe_deal_ramp` — engineered but never in the feature list)
  are removed.

In [ ]:
# STEP 6: GLOBAL LIGHTGBM QUANTILE (ITSG ONLY, GLOBAL BY MARKET)
# =============================================================================
# PURPOSE: Train one gradient-boosted quantile model across all five markets
# simultaneously ("global" panel training), producing p05/p25/p50/p75/p95 weekly
# forecasts.  Uses recursive multi-step prediction with exponential recency
# sample weights.
#
# Key design choices:
#   * Panel built on COMPLETE week x market spine (V4's 72% coverage was the
#     root cause of its ~120% holdout WAPE — target rows were missing)
#   * deterministic=True, force_row_wise=True, num_threads=1 for bitwise
#     reproducibility across runs and team members
#   * qe_proximity / qe_proximity_sq (continuous 0→1 ramp for weeks 9–13)
#   * Recursive predictor keeps 60 weeks of history (V4 kept 26, starving
#     y_lag26/52 at prediction time)
#   * Data-driven promotion gate: included in ensemble only if validation
#     WAPE <= 1.25 x best benchmark's validation WAPE
#
# CHANGED vs V4:
#   * Panel on the COMPLETE week x market spine; target from SFDC ATP;
#     forecast_prophet joined as features only (root-cause fix for the ~120%
#     holdout WAPE).
#   * Deterministic LGB params; tuned on the validation slice.
#   * Recursive predictor keeps 60 weeks of history (V4: 26 -> starved lags).
#   * Metrics at weekly TOTAL level (+ market-level diagnostic).
#   * CV folds asserted identical to the shared fold specification.
#   * Ensemble inclusion decided by a printed validation gate, not a comment.
# UNCHANGED: quantile lineup, early stopping, recursive design, region
#   holidays, MLflow + Unity Catalog registry (now wrapped defensively).
# =============================================================================

import lightgbm as lgb
import holidays as holidays_lib

GLOBAL_MODEL_NAME = 'Global_LGB_Q50_ITSG'

# mc_actuals coverage guard — ported from UCC V5 third patch session.
# When trailing 8-week non-zero authoritative coverage < threshold,
# fall back to magna_carta_actuals (the marketing proxy). This fixed a
# 7.22pp bias improvement in UCC (-18.62% -> -11.40%).
MC_AUTH_COV_THRESHOLD = 0.50
GLOBAL_QUANTILES = {'P05': 0.05, 'P25': 0.25, 'P50': 0.50, 'P75': 0.75, 'P95': 0.95}
GLOBAL_CAT_COLS = ['sales_market']
GLOBAL_EARLYSTOP_WEEKS = 13
RECURSIVE_HISTORY_WEEKS = 60   # >= 53 so y_lag52 / yoy stay populated (V4 used 26)
GLOBAL_FEATURE_COLS = [
    'week_of_qtr', 'quarter_end_flag', 'is_q4', 'q4_week_countdown',
    'qe_proximity', 'qe_proximity_sq',
    'woq_sin', 'woq_cos', 'qtr_sin', 'qtr_cos',
    'sales_market',
    'created_lag4', 'created_lag5', 'headcount_lag4', 'win_rate_woq',
    'pipeline_velocity_lag4', 'deal_size_lag1', 'sales_cycle_lag1',
    'push_ctr_lag1', 'opp_age_lag1', 'avg_dials_lag4', 'avg_touches_lag4',
    'mc_actuals_lag4', 'mkt_opened_lag4', 'miradore_lag4', 'plan_gap_lag1', 'partner_pipe_lag4',
    'sales_rep_ramp_ratio', 'y_lag1', 'y_lag4', 'y_lag8', 'y_lag13', 'y_lag26', 'y_lag52',
    'r4w', 'rolling_vol_4w', 'rolling_vol_13w', 'yoy_growth', 'qoq_momentum',
    'us_holiday_binary', 'region_holiday',
    # REMOVED (point-in-time leakage fix): stage_weighted_lag4, commit_pipeline_lag4,
    # commit_ratio_lag4, large_deal_arr_lag1, large_deal_flag_lag1,
    # large_deal_trailing_4w were computed from the CURRENT-SNAPSHOT probability/
    # stage of cds_sfdc_opp_products_latest deals (see removed
    # _load_stage_weighted_and_large_deal_features() below) — the same
    # point-in-time leakage pattern already fixed in the sibling UCC notebook.
    # Even lagged by 4 weeks, these values reflect TODAY's assessment of those
    # deals, not their probability/stage at the historical point in time being
    # trained on, which contradicts this notebook's documented claim that
    # pipeline features come only from forecast_prophet (an all-outcome source).
]
BASE_LGB_PARAMS = {
    'objective': 'quantile',
    'feature_fraction': 0.9,
    'bagging_fraction': 0.9,
    'bagging_freq': 1,
    'lambda_l2': 1.0,
    'verbosity': -1,
    'seed': 42,
    'metric': 'quantile',
    # Determinism (lesson from the UCC rebuild: multi-threaded LGB flipped
    # near-tied tuning candidates between identical runs)
    'deterministic': True,
    'force_row_wise': True,
    'num_threads': 1,
}

REGION_HOLIDAY_COUNTRIES = {'NA': ['US'], 'EMEA': ['UK', 'DE'], 'APAC': ['IN'], 'LATAM': ['BR'], 'Unknown': ['US']}


# Holiday-year window: computed relative to today rather than a fixed literal,
# so the calendar doesn't silently go stale and need a manual bump every year.
# 3 years back covers all historical training data; 5 years forward covers the
# rolling multi-year forecast horizon (Step 8C) without needing maintenance.
_HOLIDAY_CURRENT_YEAR = pd.Timestamp.today().year
HOLIDAY_YEAR_RANGE = range(_HOLIDAY_CURRENT_YEAR - 3, _HOLIDAY_CURRENT_YEAR + 5)


def compute_region_holiday(ds_series: pd.Series, market_series: pd.Series) -> pd.Series:
    result = pd.Series(0, index=ds_series.index, dtype=int)
    for market, country_codes in REGION_HOLIDAY_COUNTRIES.items():
        mask = market_series.astype(str) == market
        if not mask.any():
            continue
        hol_dates = set()
        for cc in country_codes:
            try:
                for yr in HOLIDAY_YEAR_RANGE:
                    hol_dates.update(holidays_lib.country_holidays(cc, years=yr).keys())
            except Exception:
                pass
        for idx in ds_series[mask].index:
            week_start = pd.to_datetime(ds_series[idx])
            if any(d.date() in hol_dates for d in pd.date_range(week_start, periods=7, freq='D')):
                result.loc[idx] = 1
    return result


def load_authoritative_actuals(start_date, end_date) -> pd.DataFrame:
    actuals = spark.sql(f"""
    SELECT
        CAST(period AS DATE) AS period_month,
        UPPER(COALESCE(market, 'Unknown')) AS sales_market,
        CAST(purchase_amount_in_usd_current_rate AS DOUBLE) AS authoritative_actuals
    FROM {AUTHORITATIVE_ACTUALS_TABLE}
    WHERE period >= ADD_MONTHS(DATE '{pd.Timestamp(start_date).date()}', -1)
      AND period <= DATE '{pd.Timestamp(end_date).date()}'
      AND UPPER(version) = 'ACTUALS'
      AND UPPER(COALESCE(product_genus, product_family, product_group, '')) = '{AUTHORITATIVE_ACTUALS_PRODUCT.upper()}'
      AND purchase_type = 'Growth'
    """).toPandas()
    if actuals.empty:
        return pd.DataFrame(columns=['month_key', 'sales_market', 'authoritative_actuals'])
    actuals['period_month'] = pd.to_datetime(actuals['period_month'], errors='coerce')
    actuals['sales_market'] = normalize_market(actuals['sales_market'])
    actuals['month_key'] = actuals['period_month'].dt.to_period('M').astype(str)
    return actuals.groupby(['month_key', 'sales_market'], as_index=False)['authoritative_actuals'].sum()


def _engineer_panel_features(panel: pd.DataFrame) -> pd.DataFrame:
    """Shared feature engineering for history and forward panels (per slice)."""
    panel = panel.sort_values(['slice_id', 'ds']).reset_index(drop=True)
    for col in FP_FEATURE_COLS + ['authoritative_actuals']:
        panel[col] = pd.to_numeric(panel[col], errors='coerce').fillna(0.0)
    # Continuous quarter-end proximity: 0 for weeks 1-8, ramps 0→1 for weeks 9-13
    panel['qe_proximity'] = np.maximum(0, panel['week_of_qtr'] - 8) / 5.0
    panel['qe_proximity_sq'] = panel['qe_proximity'] ** 2
    panel['pipeline_velocity'] = (panel['all_created_arr'] / panel['headcount'].replace(0, np.nan)).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    panel['plan_gap'] = panel['miradore_plan'] - panel['y'].fillna(0.0)
    grouped = panel.groupby('slice_id', group_keys=False)
    panel['sales_rep_ramp_ratio'] = (
        grouped['headcount'].diff().clip(lower=0).fillna(0.0)
        .groupby(panel['slice_id']).transform(lambda s: s.rolling(26, min_periods=1).sum())
        / panel['headcount'].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    panel['created_lag4'] = grouped['all_created_arr'].shift(4).fillna(0.0)
    panel['created_lag5'] = grouped['all_created_arr'].shift(5).fillna(0.0)
    panel['headcount_lag4'] = grouped['headcount'].shift(4)
    panel['headcount_lag4'] = panel.groupby('slice_id')['headcount_lag4'].ffill().bfill().fillna(0.0)
    panel['pipeline_velocity_lag4'] = grouped['pipeline_velocity'].shift(4).fillna(0.0)
    panel['deal_size_lag1'] = grouped['avg_deal_size'].shift(1).fillna(0.0)
    panel['sales_cycle_lag1'] = grouped['avg_sales_cycle'].shift(1).fillna(0.0)
    panel['push_ctr_lag1'] = grouped['avg_push_counter'].shift(1).fillna(0.0)
    # Fix: avg_opportunity_age shifted to lag-1 (was same-week = leakage)
    panel['opp_age_lag1'] = grouped['avg_opportunity_age'].shift(1).fillna(0.0)
    panel['avg_dials_lag4'] = grouped['avg_dials'].shift(4).fillna(0.0)
    panel['avg_touches_lag4'] = grouped['avg_touches'].shift(4).fillna(0.0)
    # mc_actuals coverage guard: when trailing 8-week non-zero coverage < threshold,
    # substitute magna_carta_actuals proxy (ported from UCC V5 — 7pp bias fix)
    _auth_lag4 = grouped['authoritative_actuals'].shift(4).fillna(0.0)
    _mc_cov = grouped['authoritative_actuals'].transform(
        lambda s: s.shift(4).rolling(8, min_periods=1).apply(lambda w: (w != 0).mean(), raw=True)
    ).fillna(0.0)
    _mc_proxy = grouped['magna_carta_actuals'].shift(4).fillna(0.0)
    panel['mc_actuals_lag4'] = np.where(_mc_cov >= MC_AUTH_COV_THRESHOLD, _auth_lag4, _mc_proxy)
    panel['mkt_opened_lag4'] = grouped['magna_carta_actuals'].shift(4).fillna(0.0)
    panel['miradore_lag4'] = grouped['miradore_plan'].shift(4).fillna(0.0)
    panel['partner_pipe_lag4'] = grouped['total_partner_pipeline'].shift(4).fillna(0.0)
    panel['plan_gap_lag1'] = grouped['plan_gap'].shift(1).fillna(0.0)
    for lag in [1, 4, 8, 13, 26, 52]:
        panel[f'y_lag{lag}'] = grouped['y'].shift(lag).fillna(0.0)
    panel['r4w'] = grouped['y'].transform(lambda s: s.shift(1).rolling(4, min_periods=1).mean()).fillna(0.0)
    panel['rolling_vol_4w'] = grouped['y'].transform(lambda s: s.shift(1).rolling(4, min_periods=2).std()).fillna(0.0)
    panel['rolling_vol_13w'] = grouped['y'].transform(lambda s: s.shift(1).rolling(13, min_periods=2).std()).fillna(0.0)
    _lag53_safe = grouped['y'].shift(53).replace(0, np.nan)
    panel['yoy_growth'] = ((panel['y_lag1'] / _lag53_safe) - 1.0).replace([np.inf, -np.inf], np.nan).clip(-3.0, 3.0).fillna(0.0)
    _lag13_safe = panel['y_lag13'].replace(0, np.nan)
    panel['qoq_momentum'] = ((panel['y_lag1'] / _lag13_safe) - 1.0).replace([np.inf, -np.inf], np.nan).clip(-3.0, 3.0).fillna(0.0)
    panel['region_holiday'] = compute_region_holiday(panel['ds'], panel['sales_market'])
    # REMOVED (point-in-time leakage fix, see GLOBAL_FEATURE_COLS comment above):
    # stage-weighted-pipeline and large-deal enrichment features and their guard
    # block used to live here. Deleted along with
    # _load_stage_weighted_and_large_deal_features() and its call site in
    # build_global_panel() — they leaked current-snapshot deal probability/stage
    # into historical training rows.
    # win_rate_woq: shift-safe rolling conversion per slice x week-of-quarter
    panel['win_rate_woq'] = np.nan
    for (_sid, _wk), grp in panel.groupby(['slice_id', 'week_of_qtr']):
        won_roll = grp['y'].shift(1).rolling(4, min_periods=1).sum()
        created_roll = grp['all_created_arr'].shift(1).rolling(4, min_periods=1).sum().replace(0, np.nan)
        panel.loc[grp.index, 'win_rate_woq'] = (won_roll / created_roll).clip(0.0, 3.0).values
    global_rate = (
        panel.groupby('slice_id')['y'].transform(lambda s: s.shift(1).rolling(13, min_periods=1).sum()) /
        panel.groupby('slice_id')['all_created_arr'].transform(lambda s: s.shift(1).rolling(13, min_periods=1).sum()).replace(0, np.nan)
    ).clip(0.0, 3.0)
    panel['win_rate_woq'] = panel['win_rate_woq'].fillna(global_rate).fillna(0.0)
    for col in GLOBAL_FEATURE_COLS:
        if col not in panel.columns:
            panel[col] = 0.0
    fill_cols = [c for c in GLOBAL_FEATURE_COLS if c not in GLOBAL_CAT_COLS]
    panel[fill_cols] = panel[fill_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return panel


# REMOVED: _load_stage_weighted_and_large_deal_features().
# It queried cds_sfdc_opp_products_latest filtered to is_closed = FALSE (today's
# open pipeline) and derived stage_weighted_pipeline (probability * amount) and
# commit_pipeline (probability >= 70%) from each deal's CURRENT-SNAPSHOT
# probability/stage. Even lagged by 4 weeks before entering the model, the
# probability/stage values themselves reflect today's assessment of those
# deals, not their probability/stage at the historical point in time being
# trained on -- point-in-time leakage. This is the same class of issue already
# fixed in the sibling UCC notebook (there, forecast_category_name-derived
# leakage), and the code comment here ("Same features that improved UCC LGB...
# ported") confirms it is the same leaky idea ported over. It also contradicted
# this notebook's own documentation claim that "pipeline features come only
# from forecast_prophet, an all-outcome source." No historical snapshot of
# SFDC probability/stage exists to reconstruct point-in-time-correct versions
# of these features, so they are removed rather than faked.


def build_global_panel() -> pd.DataFrame:
    # CHANGED (root-cause fix): spine = COMPLETE week x market grid; target from
    # SFDC ATP; forecast_prophet features left-joined and ffilled per market.
    panel = market_calendar.merge(weekly_itsg_by_market[['ds', 'sales_market', 'y', 'deal_count']],
                                  on=['ds', 'sales_market'], how='left')
    panel[['y', 'deal_count']] = panel[['y', 'deal_count']].fillna(0.0)
    panel = panel.merge(feature_source_all.drop(columns=['fp_actuals_reference']),
                        on=['ds', 'sales_market'], how='left')
    panel['slice_id'] = PRODUCT_SCOPE + '|' + panel['sales_market']
    panel = panel.sort_values(['slice_id', 'ds']).reset_index(drop=True)
    for col in FP_FEATURE_COLS:
        panel[col] = panel.groupby('slice_id')[col].ffill()
    panel = add_calendar_columns(panel)
    panel['month_key'] = panel['ds'].dt.to_period('M').astype(str)
    auth = load_authoritative_actuals(panel['ds'].min(), panel['ds'].max())
    panel = panel.merge(auth, on=['month_key', 'sales_market'], how='left')
    return _engineer_panel_features(panel)


def fit_global_quantile_models(train_frame: pd.DataFrame, params: dict = None, quantiles: dict = None):
    params = dict(BASE_LGB_PARAMS | (params or {}))
    quantiles = quantiles or GLOBAL_QUANTILES
    train_weeks = sorted(train_frame['ds'].unique())
    if len(train_weeks) <= GLOBAL_EARLYSTOP_WEEKS:
        raise ValueError('Not enough training weeks for LGB early-stopping holdout.')
    valid_start = train_weeks[-GLOBAL_EARLYSTOP_WEEKS]
    fit_train = train_frame[train_frame['ds'] < valid_start]
    fit_valid = train_frame[train_frame['ds'] >= valid_start]
    X_train = fit_train[GLOBAL_FEATURE_COLS].copy()
    X_valid = fit_valid[GLOBAL_FEATURE_COLS].copy()
    for col in GLOBAL_CAT_COLS:
        X_train[col] = pd.Categorical(X_train[col], categories=GLOBAL_MARKETS)
        X_valid[col] = pd.Categorical(X_valid[col], categories=GLOBAL_MARKETS)
    train_ds = lgb.Dataset(X_train, label=fit_train['y'].astype(float), categorical_feature=GLOBAL_CAT_COLS, free_raw_data=False)
    valid_ds = lgb.Dataset(X_valid, label=fit_valid['y'].astype(float), categorical_feature=GLOBAL_CAT_COLS, reference=train_ds, free_raw_data=False)
    models = {}
    for label, alpha in quantiles.items():
        models[label] = lgb.train(params | {'alpha': alpha}, train_ds, num_boost_round=1500,
                                  valid_sets=[valid_ds], callbacks=[lgb.early_stopping(stopping_rounds=75, verbose=False)])
    p50_key = 'P50' if 'P50' in models else list(models.keys())[0]
    importance_df = pd.DataFrame({
        'feature': GLOBAL_FEATURE_COLS,
        'importance_gain': models[p50_key].feature_importance(importance_type='gain'),
        'importance_split': models[p50_key].feature_importance(importance_type='split'),
    }).sort_values('importance_gain', ascending=False).reset_index(drop=True)
    return models, importance_df


def recursive_lgb_predict(models, panel, predict_start, predict_end):
    """Recursive week-by-week prediction, identical in CV / holdout / production.
    Keeps RECURSIVE_HISTORY_WEEKS of history so long lags stay populated.
    Returns slice rows with p05/p25/p50/p75/p95 (clipped >= 0, ordered) and y_actual."""
    history_cutoff = pd.Timestamp(predict_start) - pd.Timedelta(weeks=RECURSIVE_HISTORY_WEEKS)
    work = panel[(panel['ds'] >= history_cutoff) & (panel['ds'] <= predict_end)].copy()
    work = work.sort_values(['slice_id', 'ds']).reset_index(drop=True)
    pred_mask = work['ds'] >= pd.Timestamp(predict_start)
    work['y_actual'] = work['y']
    work.loc[pred_mask, 'y'] = np.nan
    predict_weeks = sorted(work.loc[pred_mask, 'ds'].unique())
    preds = []
    for step_ds in predict_weeks:
        _grp = work.groupby('slice_id', group_keys=False)
        for lag in [1, 4, 8, 13, 26, 52]:
            work[f'y_lag{lag}'] = _grp['y'].shift(lag).fillna(0.0)
        work['r4w'] = _grp['y'].transform(lambda s: s.shift(1).rolling(4, min_periods=1).mean()).fillna(0.0)
        work['rolling_vol_4w'] = _grp['y'].transform(lambda s: s.shift(1).rolling(4, min_periods=2).std()).fillna(0.0)
        work['rolling_vol_13w'] = _grp['y'].transform(lambda s: s.shift(1).rolling(13, min_periods=2).std()).fillna(0.0)
        _lag53_safe = _grp['y'].shift(53).replace(0, np.nan)
        work['yoy_growth'] = ((work['y_lag1'] / _lag53_safe) - 1.0).replace([np.inf, -np.inf], np.nan).clip(-3.0, 3.0).fillna(0.0)
        _lag13_safe = work['y_lag13'].replace(0, np.nan)
        work['qoq_momentum'] = ((work['y_lag1'] / _lag13_safe) - 1.0).replace([np.inf, -np.inf], np.nan).clip(-3.0, 3.0).fillna(0.0)
        work['plan_gap'] = work['miradore_plan'] - work['y'].fillna(0.0)
        work['plan_gap_lag1'] = _grp['plan_gap'].shift(1).fillna(0.0)

        step_mask = work['ds'] == step_ds
        step_rows = work.loc[step_mask].copy()
        X_step = step_rows[GLOBAL_FEATURE_COLS].copy()
        for cat in GLOBAL_CAT_COLS:
            X_step[cat] = pd.Categorical(X_step[cat], categories=GLOBAL_MARKETS)
        # CHANGED: quantile lineup generalized from the fixed 3-level
        # ['P10', 'P50', 'P90'] to the dict-driven 5-level set (GLOBAL_QUANTILES
        # keys), so retraining at new quantile levels (Step 6) automatically
        # flows through to prediction without touching this loop again.
        for name in GLOBAL_QUANTILES:
            if name in models:
                step_rows[name.lower()] = np.clip(models[name].predict(X_step, num_iteration=models[name].best_iteration), 0.0, None)
        for _fallback_col in ['p05', 'p25', 'p75', 'p95']:
            if _fallback_col not in step_rows.columns:
                step_rows[_fallback_col] = step_rows['p50']
        work.loc[step_mask, 'y'] = step_rows['p50'].values   # feed p50 back
        preds.append(step_rows)
    result = pd.concat(preds, ignore_index=True)
    return enforce_quantile_order(result, ['p05', 'p25', 'p50', 'p75', 'p95'])


global_panel = build_global_panel()
# Root-cause sanity checks (the V4 failure mode)
assert global_panel.groupby('ds').size().nunique() == 1, 'Panel must be a complete week x market grid.'
_panel_total = global_panel.groupby('ds')['y'].sum().sort_index()
assert np.allclose(_panel_total.values, weekly_itsg.sort_values('ds')['y'].values, atol=1.0), \
    'LGB panel target does not reconcile to weekly_itsg.y.'
print(f'LGB panel: {len(global_panel):,} rows = {global_panel["ds"].nunique()} weeks x {global_panel["sales_market"].nunique()} markets (complete grid)')

validation_weeks_list = sorted(validation_df['ds'].tolist())


### Step 6-ABL - Evidence-gated LGB feature ablation *(NEW)*

With vs without `mc_actuals_lag4` (+MC proxy), `miradore_lag4`, `plan_gap_lag1` on the validation slice and every walk-forward CV fold. Reduced set adopted only per margin rule; the tuning, CV, holdout and production LGB below then use the selected `GLOBAL_FEATURE_COLS`.

In [ ]:
# =============================================================================
# STEP 6-ABL: EVIDENCE GATE -- LGB FEATURE ABLATION (mc_actuals_lag4, miradore_lag4, plan_gap_lag1)
# =============================================================================
# Uses the Step 6 helpers above (fit_global_quantile_models / recursive_lgb_predict, which read the
# module-level GLOBAL_FEATURE_COLS). Each variant is fitted with default params, P50 only, on
#   (a) the validation slice (train < VALIDATION_START) and
#   (b) every walk-forward CV fold (train <= fold train_end; expanding window; tests are earlier than validation).
# Weekly TOTAL forecasts are scored (WAPE, signed bias). The reduced set is adopted only per
# ig_ablation_select (WAPE margin, or bias gain with no WAPE loss, and no segment regression).
# Fail-soft: any problem keeps the full feature set. Holdout is NOT used here.
LGB_FEATURES_FULL = list(GLOBAL_FEATURE_COLS)
LGB_FEATURE_DROP_SELECTED = []
if globals().get('IMPROVEMENT_GATES_ENABLED', False) and globals().get('LGB_FEATURE_DROP_OVERRIDE') is not None:
    LGB_FEATURE_DROP_SELECTED = [f for f in LGB_FEATURE_DROP_OVERRIDE if f in LGB_FEATURES_FULL]
    GLOBAL_FEATURE_COLS = [f for f in LGB_FEATURES_FULL if f not in LGB_FEATURE_DROP_SELECTED]
    print(f'6-ABL: manual override -> dropping {LGB_FEATURE_DROP_SELECTED} (gate bypassed).')
elif globals().get('IMPROVEMENT_GATES_ENABLED', False) and globals().get('LGB_ABLATION_ENABLED', False):
    try:
        assert 'fold_specs' in globals() and len(fold_specs) > 0, 'fold_specs missing (Step 3)'
        _variants = {'full': []}
        for _k, _drop in LGB_ABLATION_GROUPS.items():
            _miss = [f for f in _drop if f not in LGB_FEATURES_FULL]
            assert not _miss, f'ablation group {_k} names features not in GLOBAL_FEATURE_COLS: {_miss}'
            _variants[_k] = list(_drop)
        _val_end = validation_df['ds'].max()
        _rows = []
        for _name, _drop in _variants.items():
            GLOBAL_FEATURE_COLS = [f for f in LGB_FEATURES_FULL if f not in _drop]
            try:
                _m, _ = fit_global_quantile_models(global_panel[global_panel['ds'] < VALIDATION_START], quantiles={'P50': 0.50})
                _p = recursive_lgb_predict(_m, global_panel, VALIDATION_START, _val_end)
                _v = _p.groupby('ds', as_index=False).agg(a=('y_actual', 'sum'), f=('p50', 'sum'))
                _ca, _cf = [], []
                for _sp in fold_specs:
                    _mf, _ = fit_global_quantile_models(global_panel[global_panel['ds'] <= _sp.train_end], quantiles={'P50': 0.50})
                    _pf = recursive_lgb_predict(_mf, global_panel, _sp.test_start, _sp.test_end)
                    _w = _pf.groupby('ds', as_index=False).agg(a=('y_actual', 'sum'), f=('p50', 'sum'))
                    _ca.append(_w['a'].values); _cf.append(_w['f'].values)
                _ca, _cf = np.concatenate(_ca), np.concatenate(_cf)
                _pa, _pf_ = np.concatenate([_ca, _v['a'].values]), np.concatenate([_cf, _v['f'].values])
                _rows.append({'variant': _name, 'dropped': ','.join(_drop) or '-',
                              'cv_wape': ig_wape(_ca, _cf), 'cv_bias': ig_bias_pct(_ca, _cf),
                              'val_wape': ig_wape(_v['a'], _v['f']), 'val_bias': ig_bias_pct(_v['a'], _v['f']),
                              'pool_wape': ig_wape(_pa, _pf_), 'pool_bias': ig_bias_pct(_pa, _pf_)})
            except Exception as _e_v:
                print(f'  variant {_name} failed ({type(_e_v).__name__}: {str(_e_v)[:120]}) -> excluded')
        GLOBAL_FEATURE_COLS = list(LGB_FEATURES_FULL)
        _tbl = pd.DataFrame(_rows)
        assert len(_tbl) and 'full' in set(_tbl['variant']), 'full-feature baseline failed'
        print(f'=== 6-ABL  LGB feature ablation ({len(fold_specs)} CV folds + validation slice; WAPE %, bias % fc vs actual) ===')
        print(_tbl.round(2).to_string(index=False))
        _best, _why = ig_ablation_select(_tbl, 'full', LGB_ABLATION_MARGIN_PP, LGB_ABLATION_BIAS_PP, LGB_ABLATION_SEG_TOL_PP)
        if _best != 'full':
            LGB_FEATURE_DROP_SELECTED = list(_variants[_best])
            GLOBAL_FEATURE_COLS = [f for f in LGB_FEATURES_FULL if f not in LGB_FEATURE_DROP_SELECTED]
        IG_RESULTS['ablation'] = {'table': _tbl, 'selected': _best, 'reason': _why, 'dropped': list(LGB_FEATURE_DROP_SELECTED)}
        print(f"  DECISION: variant '{_best}' -> LGB_FEATURE_DROP_SELECTED = {LGB_FEATURE_DROP_SELECTED} -- {_why}")
    except Exception as _exc_abl:
        GLOBAL_FEATURE_COLS = list(LGB_FEATURES_FULL); LGB_FEATURE_DROP_SELECTED = []
        print(f'6-ABL skipped, full feature set kept (fail-soft): {type(_exc_abl).__name__}: {str(_exc_abl)[:250]}')
else:
    print('6-ABL: gates/ablation disabled -> full LGB feature set (previous behaviour).')
print(f'LGB features in use: {len(GLOBAL_FEATURE_COLS)} (dropped: {LGB_FEATURE_DROP_SELECTED})')

In [ ]:
# --- Tune on the validation slice (P50 only for speed) --------------------------
lgb_grid = [{'num_leaves': nl, 'learning_rate': lr, 'min_data_in_leaf': mdl}
            for nl in (15, 31) for lr in (0.03, 0.06) for mdl in (10, 20)]
lgb_tuning_rows = []
for candidate in lgb_grid:
    row = dict(candidate)
    try:
        cand_models, _ = fit_global_quantile_models(global_panel[global_panel['ds'] < VALIDATION_START], params=candidate, quantiles={'P50': 0.50})
        cand_pred = recursive_lgb_predict(cand_models, global_panel, VALIDATION_START, validation_df['ds'].max())
        cand_weekly = cand_pred.groupby('ds', as_index=False)['p50'].sum().sort_values('ds')
        m = forecast_metrics(validation_actuals, cand_weekly['p50'].to_numpy(dtype=float))
        row.update({'val_wape': round(m['wape'], 3), 'val_mape': round(m['mape'], 3), 'val_mpe': round(m['mpe'], 3), 'error': ''})
    except Exception as exc:
        row.update({'val_wape': np.inf, 'val_mape': np.inf, 'val_mpe': np.nan, 'error': str(exc)[:160]})
    lgb_tuning_rows.append(row)
lgb_tuning = pd.DataFrame(lgb_tuning_rows).sort_values('val_wape').reset_index(drop=True)
assert np.isfinite(lgb_tuning['val_wape'].iloc[0]), 'All LGB tuning candidates failed.'
BEST_LGB_PARAMS = {'num_leaves': int(lgb_tuning.iloc[0]['num_leaves']),
                   'min_data_in_leaf': int(lgb_tuning.iloc[0]['min_data_in_leaf']),
                   'learning_rate': float(lgb_tuning.iloc[0]['learning_rate'])}
print(f'Tuned LGB params (deterministic training): {BEST_LGB_PARAMS}')
display(lgb_tuning)

# --- Ensemble promotion gate (validation slice only) -----------------------------
LGB_VALIDATION_WAPE = float(lgb_tuning['val_wape'].iloc[0])
BENCHMARK_VALIDATION_WAPES = {
    'Prophet_trend': float(prophet_tuning['val_wape'].iloc[0]),
    'MSTL_v2': float(mstl_tuning['val_wape'].iloc[0]),
    'ETS': float(ets_tuning['val_wape'].iloc[0]),
    'DHR_ARIMA': float(dhr_tuning['val_wape'].iloc[0]),
}
LGB_GATE_THRESHOLD = 1.25 * min(BENCHMARK_VALIDATION_WAPES.values())
LGB_IN_ENSEMBLE = LGB_VALIDATION_WAPE <= LGB_GATE_THRESHOLD
print(f'\nLGB ensemble gate: validation WAPE {LGB_VALIDATION_WAPE:.2f} vs threshold {LGB_GATE_THRESHOLD:.2f} '
      f'(1.25 x best benchmark) -> {"INCLUDED in" if LGB_IN_ENSEMBLE else "EXCLUDED from"} the production ensemble')

# --- Clean CV replicating shared folds -------------------------------------------
lgb_cv_rows, lgb_cv_pred_frames = [], []
for spec in fold_specs:
    fold_train = global_panel[global_panel['ds'] <= spec.train_end]
    assert fold_train['ds'].max() < VALIDATION_START
    fold_models, _ = fit_global_quantile_models(fold_train, params=BEST_LGB_PARAMS)
    fold_pred = recursive_lgb_predict(fold_models, global_panel, spec.test_start, spec.test_end)
    fold_weekly = fold_pred.groupby('ds', as_index=False).agg(actual=('y_actual', 'sum'), forecast=('p50', 'sum'),
                                                              p05=('p05', 'sum'), p25=('p25', 'sum'),
                                                              p75=('p75', 'sum'), p95=('p95', 'sum')).sort_values('ds')
    m = forecast_metrics(fold_weekly['actual'].values, fold_weekly['forecast'].values,
                         fold_weekly['p05'].values, fold_weekly['p95'].values,
                         fold_weekly['p25'].values, fold_weekly['p75'].values)
    lgb_cv_rows.append({'model': GLOBAL_MODEL_NAME, 'fold': spec.fold,
                        'train_start': str(fold_train['ds'].min().date()), 'train_end': str(fold_train['ds'].max().date()),
                        'test_start': str(spec.test_start.date()), 'test_end': str(spec.test_end.date()),
                        'train_rows': fold_train['ds'].nunique(), 'test_rows': len(fold_weekly),
                        'wape': round(m['wape'], 3), 'mape': round(m['mape'], 3), 'mpe': round(m['mpe'], 3),
                        'smape': round(m['smape'], 3), 'rmse': round(m['rmse'], 3),
                        'ci_coverage': round(m['ci_coverage'], 1) if pd.notna(m['ci_coverage']) else np.nan})
    lgb_cv_pred_frames.append(pd.DataFrame({'model': GLOBAL_MODEL_NAME, 'fold': spec.fold, 'ds': fold_weekly['ds'].values,
                                            'actual': fold_weekly['actual'].values, 'forecast': fold_weekly['forecast'].values}))

lgb_cv_fold_results = pd.DataFrame(lgb_cv_rows)
lgb_cv_summary_row = {'model': GLOBAL_MODEL_NAME, 'folds': len(lgb_cv_fold_results)}
for k in ['wape', 'mape', 'mpe', 'smape', 'rmse']:
    lgb_cv_summary_row[f'mean_{k}'] = round(lgb_cv_fold_results[k].mean(), 3)
    lgb_cv_summary_row[f'std_{k}'] = round(lgb_cv_fold_results[k].std(ddof=0), 3)
global_cv_summary = pd.DataFrame([lgb_cv_summary_row])
global_cv_predictions = pd.concat(lgb_cv_pred_frames, ignore_index=True)

# --- Holdout: retrain on all pre-holdout weeks; evaluate at TOTAL level ----------
global_quantile_models, global_feature_importance = fit_global_quantile_models(
    global_panel[global_panel['ds'] < BACKTEST_START], params=BEST_LGB_PARAMS)
global_test_predictions = recursive_lgb_predict(global_quantile_models, global_panel, BACKTEST_START, global_panel['ds'].max())
global_holdout_weekly = (
    global_test_predictions.groupby('ds', as_index=False)
    .agg(actual=('y_actual', 'sum'), forecast=('p50', 'sum'),
         p05=('p05', 'sum'), p25=('p25', 'sum'), p75=('p75', 'sum'), p95=('p95', 'sum'))
    .sort_values('ds')
)
ghm = forecast_metrics(global_holdout_weekly['actual'].values, global_holdout_weekly['forecast'].values,
                       global_holdout_weekly['p05'].values, global_holdout_weekly['p95'].values,
                       global_holdout_weekly['p25'].values, global_holdout_weekly['p75'].values)
global_holdout_summary = pd.DataFrame([{'model': GLOBAL_MODEL_NAME,
                                        'test_start': str(global_holdout_weekly['ds'].min().date()),
                                        'test_end': str(global_holdout_weekly['ds'].max().date()),
                                        'test_rows': len(global_holdout_weekly),
                                        'wape': round(ghm['wape'], 3), 'mape': round(ghm['mape'], 3),
                                        'mpe': round(ghm['mpe'], 3), 'smape': round(ghm['smape'], 3),
                                        'rmse': round(ghm['rmse'], 3),
                                        'ci_coverage': round(ghm['ci_coverage'], 1) if pd.notna(ghm['ci_coverage']) else np.nan}])
# Market-level diagnostic (NOT the headline metric — near-zero slices distort %)
_slice_m = forecast_metrics(global_test_predictions['y_actual'].values, global_test_predictions['p50'].values)
print(f'LGB holdout — TOTAL weekly WAPE: {ghm["wape"]:.2f}% | market-slice diagnostic WAPE: {_slice_m["wape"]:.2f}%')

# --- Merge into shared result frames ----------------------------------------------
cv_benchmark_summary = pd.concat([cv_benchmark_summary[cv_benchmark_summary['model'] != GLOBAL_MODEL_NAME],
                                  global_cv_summary], ignore_index=True).sort_values('mean_wape').reset_index(drop=True)
cv_fold_metrics = pd.concat([cv_fold_metrics[cv_fold_metrics['model'] != GLOBAL_MODEL_NAME],
                             lgb_cv_fold_results.drop(columns=['ci_coverage'])], ignore_index=True)
cv_fold_predictions = pd.concat([cv_fold_predictions[cv_fold_predictions['model'] != GLOBAL_MODEL_NAME],
                                 global_cv_predictions], ignore_index=True)
holdout_benchmark_summary = pd.concat([holdout_benchmark_summary[holdout_benchmark_summary['model'] != GLOBAL_MODEL_NAME],
                                       global_holdout_summary], ignore_index=True).sort_values('wape').reset_index(drop=True)
holdout_benchmark_predictions = pd.concat([
    holdout_benchmark_predictions[holdout_benchmark_predictions['model'] != GLOBAL_MODEL_NAME],
    global_holdout_weekly.rename(columns={'forecast': 'forecast'}).assign(model=GLOBAL_MODEL_NAME)[['model', 'ds', 'actual', 'forecast']],
], ignore_index=True)

# --- MLflow + Unity Catalog registry (defensive) -----------------------------------
with mlflow.start_run(run_name='itsg_global_lgb_quantile_cv_v2', nested=False):
    mlflow.log_param('model_name', GLOBAL_MODEL_NAME)
    mlflow.log_param('panel_spine', 'complete_week_x_market_grid_sfdc_target')
    mlflow.log_param('tuned_params', str(BEST_LGB_PARAMS))
    mlflow.log_param('deterministic', True)
    mlflow.log_param('recursive_history_weeks', RECURSIVE_HISTORY_WEEKS)
    mlflow.log_param('lgb_in_ensemble', bool(LGB_IN_ENSEMBLE))
    mlflow.log_param('cv_folds', CV_FOLDS)
    for k, v in global_cv_summary.iloc[0].to_dict().items():
        if k != 'model' and pd.notna(v):
            mlflow.log_metric(str(k), float(v))
    for k in ['wape', 'mape', 'mpe', 'smape', 'rmse', 'ci_coverage']:
        v = global_holdout_summary.iloc[0][k]
        if pd.notna(v):
            mlflow.log_metric(f'holdout_{k}', float(v))
    try:
        mlflow.set_registry_uri('databricks-uc')
        LGB_REGISTERED_MODEL_NAME = 'datagroup_mdl.mdl_sales_analytics.itsg_global_lgb_p50'
        from mlflow.models import infer_signature
        _sample_X = global_panel[global_panel['ds'] < BACKTEST_START][GLOBAL_FEATURE_COLS].head(5).copy()
        for cat in GLOBAL_CAT_COLS:
            _sample_X[cat] = pd.Categorical(_sample_X[cat], categories=GLOBAL_MARKETS)
        _sig = infer_signature(_sample_X, global_quantile_models['P50'].predict(_sample_X, num_iteration=global_quantile_models['P50'].best_iteration))
        mlflow.lightgbm.log_model(global_quantile_models['P50'], name='lgb_p50_model',
                                  signature=_sig, registered_model_name=LGB_REGISTERED_MODEL_NAME)
        print(f'LGB P50 registered: {LGB_REGISTERED_MODEL_NAME}')
    except Exception as exc:
        print(f'Model registry step skipped ({type(exc).__name__}: {exc}) — training results are unaffected.')

print('\nGlobal LGB CV summary (weekly total level, clean folds):')
display(global_cv_summary)
print('\nGlobal LGB CV fold detail (incl. per-fold CI coverage):')
display(lgb_cv_fold_results)
print('\nGlobal LGB holdout summary (weekly total level):')
display(global_holdout_summary)
print('\nTop LGB feature importance:')
display(global_feature_importance.head(15))
print('\nUpdated CV / holdout leaderboards:')
display(cv_benchmark_summary)
display(holdout_benchmark_summary)

### Step 6B — Ensemble membership validation **(NEW — UCC-style governance)**

This notebook now validates the live `Adaptive_Ensemble` membership explicitly before production. Instead of automatically ensembling every model that runs, it evaluates candidate 2-, 3-, and 4-model combinations built from the strongest holdout performers and balances:

* **Holdout weekly WAPE** as the primary business-facing criterion
* **Closed-quarter error** as the quarter-level guardrail
* **Walk-forward CV WAPE** as the robustness tie-breaker

The selected member list is written to `ENSEMBLE_COMPONENT_MODELS`, and Step 7 uses that authoritative live set for the forward production forecast.

In [ ]:
# STEP 6B: ENSEMBLE MEMBERSHIP VALIDATION
# =============================================================================
# PURPOSE: Choose the live Adaptive Ensemble membership explicitly, using a
# UCC-style governance pass before production forecasting.
#
# CHANGED (holdout-contamination fix): the previous version picked the live
# combo primarily by HOLDOUT WAPE, which used the "untouched, out-of-sample"
# holdout set to make a model-inclusion decision -- contaminating this
# notebook's own claim that holdout is never used for decisions (the same
# class of issue as the UCC ensemble-weighting fix, though not identical).
# The combo-level walk-forward CV WAPE (computed from cv_fold_predictions,
# entirely pre-holdout) is genuinely available at combination granularity, so
# it is now the PRIMARY selection criterion. Per-model validation-slice WAPE
# (from Step 6's BENCHMARK_VALIDATION_WAPES / LGB_VALIDATION_WAPE, also
# pre-holdout) is used as a SECONDARY filter to build the candidate pool,
# since only single-model validation WAPE exists here -- there is no
# combination-level validation-slice score to rank by directly, so combo-level
# CV WAPE is the best pre-holdout signal for ranking combinations, per the
# documented fallback ("if validation-slice-level combination scores aren't
# available, use CV WAPE as primary and validation-slice per-model WAPE as a
# secondary filter"). Holdout WAPE and closed-quarter error are still computed
# and displayed, but strictly as REPORTED/DIAGNOSTIC numbers -- they no longer
# drive which combination is selected.
#
# This cell also now actually reads LGB_IN_ENSEMBLE (Step 6's validation gate,
# logged to MLflow) instead of leaving it decorative: candidate combinations
# containing the LGB model are only considered if LGB_IN_ENSEMBLE is True.
#
# Selection workflow:
#   1. Build a candidate pool from per-model VALIDATION-slice WAPE (pre-
#      holdout), gated by the Step 6 LGB_IN_ENSEMBLE flag.
#   2. Score all 2-, 3-, and 4-model combinations with inverse-CV-WAPE weights.
#   3. Pick the live set by combo-level walk-forward CV WAPE (primary), tie-
#      broken by CV bias and the validation-WAPE proxy within a narrow CV
#      band. Holdout WAPE / closed-quarter error are reported for visibility
#      only and do not drive the selection.
#
# Output:
#   * ENSEMBLE_COMPONENT_MODELS (authoritative live production set)
#   * ensemble_membership_validation (comparison table)
# =============================================================================

from itertools import combinations

# --- Candidate pool: per-model VALIDATION-slice WAPE (pre-holdout), not holdout --
ALL_VALIDATION_WAPES = dict(BENCHMARK_VALIDATION_WAPES)
ALL_VALIDATION_WAPES[GLOBAL_MODEL_NAME] = LGB_VALIDATION_WAPE
validation_rank = pd.DataFrame(
    {'model': list(ALL_VALIDATION_WAPES.keys()), 'val_wape': list(ALL_VALIDATION_WAPES.values())}
).sort_values('val_wape').reset_index(drop=True)

# CV-slice bias (mean_mpe from the clean walk-forward folds) as the bias filter,
# replacing the old holdout-mpe-based filter -- also pre-holdout.
cv_bias_map = cv_benchmark_summary.set_index('model')['mean_mpe'].to_dict()
validation_rank['cv_mean_mpe'] = validation_rank['model'].map(cv_bias_map)

best_single_val_wape = float(validation_rank['val_wape'].min())
candidate_pool = validation_rank[
    (validation_rank['val_wape'] <= best_single_val_wape * 1.20)
    & (validation_rank['cv_mean_mpe'].abs() <= 20)  # tightened from 35 — excludes high-bias models
]['model'].tolist()
if len(candidate_pool) < 2:  # minimum 2 models for a valid ensemble
    candidate_pool = validation_rank.head(min(4, len(validation_rank)))['model'].tolist()

candidate_pool = list(dict.fromkeys(candidate_pool))

# Enforce the Step 6 promotion gate: LGB only stays in the candidate pool if it
# passed LGB_IN_ENSEMBLE (validation WAPE <= 1.25x best benchmark validation WAPE).
if GLOBAL_MODEL_NAME in candidate_pool and not LGB_IN_ENSEMBLE:
    candidate_pool.remove(GLOBAL_MODEL_NAME)
    print(f'{GLOBAL_MODEL_NAME} excluded from the candidate pool: failed the Step 6 '
          f'LGB_IN_ENSEMBLE validation gate.')

print(f'Candidate pool for live ensemble validation (validation-WAPE + CV-bias gated): {candidate_pool}')

cv_wape_map = cv_benchmark_summary.set_index('model')['mean_wape'].to_dict()

def _combo_weights(models):
    w = pd.DataFrame({'model': list(models)})
    w['mean_wape'] = w['model'].map(cv_wape_map)
    if w['mean_wape'].isna().any():
        missing = w.loc[w['mean_wape'].isna(), 'model'].tolist()
        raise ValueError(f'Missing CV WAPE for models: {missing}')
    w['raw_weight'] = 1.0 / w['mean_wape']
    w['ensemble_weight'] = w['raw_weight'] / w['raw_weight'].sum()
    return w[['model', 'ensemble_weight', 'mean_wape']]

def _aggregate_ensemble(pred_df, models):
    weights = _combo_weights(models)
    tmp = pred_df[pred_df['model'].isin(models)].merge(weights[['model', 'ensemble_weight']], on='model', how='inner').copy()
    tmp['weighted_forecast'] = tmp['forecast'] * tmp['ensemble_weight']
    group_cols = [c for c in ['fold', 'origin', 'train_end', 'test_start', 'test_end'] if c in tmp.columns] + ['ds']
    agg = tmp.groupby(group_cols, as_index=False).agg(actual=('actual', 'first'), forecast=('weighted_forecast', 'sum'))
    return agg, weights

def _mean_fold_wape(pred_df):
    fold_cols = [c for c in ['fold', 'origin', 'train_end', 'test_start', 'test_end'] if c in pred_df.columns]
    if not fold_cols:
        return float(forecast_metrics(pred_df['actual'].values, pred_df['forecast'].values)['wape'])
    fold_metrics = []
    for _, grp in pred_df.groupby(fold_cols):
        fold_metrics.append(float(forecast_metrics(grp['actual'].values, grp['forecast'].values)['wape']))
    return float(np.mean(fold_metrics)) if fold_metrics else np.nan

def _closed_quarter_error(pred_df):
    qdf = pred_df.copy()
    qdf['quarter'] = pd.to_datetime(qdf['ds']).dt.to_period('Q').astype(str)
    qsum = qdf.groupby('quarter', as_index=False).agg(actual=('actual', 'sum'), forecast=('forecast', 'sum'))
    qsum['pct_error'] = np.where(
        qsum['actual'].abs() > 1e-9,
        100.0 * (qsum['forecast'] - qsum['actual']) / qsum['actual'],
        np.nan,
    )
    return float(qsum['pct_error'].abs().max()) if not qsum.empty else np.nan, qsum

validation_rows = []
for r in range(2, min(4, len(candidate_pool)) + 1):
    for combo in combinations(candidate_pool, r):
        # PRIMARY (pre-holdout): combo-level walk-forward CV WAPE.
        cv_combo, weights = _aggregate_ensemble(cv_fold_predictions, combo)
        cv_metrics = forecast_metrics(cv_combo['actual'].values, cv_combo['forecast'].values)
        # SECONDARY (pre-holdout): weighted average of each member's own
        # validation-slice WAPE, using the same inverse-CV-WAPE weights as the
        # ensemble itself. Approximates a combo-level validation score, since
        # no combination-level validation-slice prediction series exists here
        # (only single-model validation WAPE was tuned in Step 6).
        val_wape_proxy = float(
            sum(w * ALL_VALIDATION_WAPES[m] for m, w in zip(weights['model'], weights['ensemble_weight']))
        )
        # DIAGNOSTIC ONLY (holdout) — reported for visibility, not used to rank.
        holdout_combo, _ = _aggregate_ensemble(holdout_benchmark_predictions, combo)
        holdout_metrics = forecast_metrics(holdout_combo['actual'].values, holdout_combo['forecast'].values)
        max_q_error, qsum = _closed_quarter_error(holdout_combo)
        validation_rows.append({
            'config': ' + '.join(combo),
            'n_members': len(combo),
            'members': list(combo),
            'cv_mean_wape': round(_mean_fold_wape(cv_combo), 3),
            'cv_mpe': round(float(cv_metrics['mpe']), 3),
            'abs_cv_bias': round(abs(float(cv_metrics['mpe'])), 3),
            'val_wape_proxy': round(val_wape_proxy, 3),
            'holdout_wape': round(float(holdout_metrics['wape']), 3),  # diagnostic only
            'holdout_mape': round(float(holdout_metrics['mape']), 3),  # diagnostic only
            'holdout_mpe': round(float(holdout_metrics['mpe']), 3),  # diagnostic only
            'max_closed_q_error_pct': round(max_q_error, 2),  # diagnostic only
            'weights': ', '.join(f"{m}={w:.1%}" for m, w in zip(weights['model'], weights['ensemble_weight'])),
        })

ensemble_membership_validation = pd.DataFrame(validation_rows).sort_values(
    ['cv_mean_wape', 'abs_cv_bias', 'val_wape_proxy', 'n_members']
).reset_index(drop=True)

# --- Selection: PRIMARY = combo-level CV WAPE, tie-broken within a narrow CV band --
best_cv = float(ensemble_membership_validation['cv_mean_wape'].min())
selection_band = ensemble_membership_validation[
    ensemble_membership_validation['cv_mean_wape'] <= best_cv + 0.75
].copy()
selected_live = selection_band.sort_values(
    ['abs_cv_bias', 'val_wape_proxy', 'n_members', 'cv_mean_wape']
).iloc[0]

ENSEMBLE_COMPONENT_MODELS = list(selected_live['members'])
ENSEMBLE_SELECTION_METHOD = 'cv_wape_primary_validation_proxy_tiebreak_holdout_diagnostic_only'

print('\n' + '=' * 88)
print('ITSG ENSEMBLE MEMBERSHIP VALIDATION')
print('=' * 88)
print(f'Best CV config            : {ensemble_membership_validation.iloc[0]["config"]} '
      f'({ensemble_membership_validation.iloc[0]["cv_mean_wape"]:.2f}% CV WAPE)')
print(f'Selected live config      : {selected_live["config"]} ({selected_live["cv_mean_wape"]:.2f}% CV WAPE)')
print(f'Selection rationale       : configs within {0.75:.2f} pp of the best combo-level walk-forward '
      f'CV WAPE are tie-broken by CV bias, then the validation-WAPE proxy, then member count')
print(f'Holdout WAPE (diagnostic) : {selected_live["holdout_wape"]:.2f}% — reported only, did NOT drive selection')
print(f'Max closed-qtr error (dx) : {selected_live["max_closed_q_error_pct"]:.2f}% — reported only, did NOT drive selection')
print(f'Authoritative live set    : {ENSEMBLE_COMPONENT_MODELS}')
print('=' * 88 + '\n')

display(ensemble_membership_validation[
    ['config', 'n_members', 'cv_mean_wape', 'cv_mpe', 'val_wape_proxy',
     'holdout_wape', 'holdout_mpe', 'max_closed_q_error_pct', 'weights']
])


### Step 13H - Evidence gates: LightGBM weight, quarter-end option, level-shift *(NEW)*

Each option is adopted only if it beats the incumbent on earlier-data-only evidence (walk-forward CV + holdout). Step 7 reads `LGB_WEIGHT_MULT_SELECTED`, `QE_METHOD_SELECTED`, `LEVEL_ADJ_ENABLED_SELECTED`. Thin evidence => incumbent kept.

In [ ]:
# =============================================================================
# STEP 13H-A: EVIDENCE GATE -- LIGHTGBM WEIGHT MULTIPLIER (walk-forward CV + holdout)
# =============================================================================
# Needs (earlier cells): cv_fold_predictions [model, fold, ds, actual, forecast],
# holdout_benchmark_predictions [model, ds, actual, forecast], cv_benchmark_summary [model, mean_wape],
# ENSEMBLE_COMPONENT_MODELS, GLOBAL_MODEL_NAME. Weights replicate Step 7 (inverse mean CV WAPE).
# The walk-forward CV folds ARE this notebook's rolling-origin backtest (origin = day before the fold's
# first test week; horizon_week = 1..13). Fail-soft: any problem => multiplier stays 1.0.
LGB_WEIGHT_MULT_SELECTED = 1.0
IG_BT_BASE = None
if globals().get('IMPROVEMENT_GATES_ENABLED', False):
    try:
        _need = {'cv_fold_predictions': ['model', 'fold', 'ds', 'forecast', 'actual'],
                 'holdout_benchmark_predictions': ['model', 'ds', 'forecast', 'actual'],
                 'cv_benchmark_summary': ['model', 'mean_wape']}
        for _v, _cols in _need.items():
            assert _v in globals(), f'{_v} not defined (run the earlier step first)'
            _miss = [c for c in _cols if c not in globals()[_v].columns]
            assert not _miss, f'{_v} lacks columns {_miss}; has {list(globals()[_v].columns)}'
        assert 'ENSEMBLE_COMPONENT_MODELS' in globals() and 'GLOBAL_MODEL_NAME' in globals()
        _comp = list(ENSEMBLE_COMPONENT_MODELS)
        _ws = cv_benchmark_summary[cv_benchmark_summary['model'].isin(_comp)].copy()
        IG_W = dict(zip(_ws['model'], (1.0 / _ws['mean_wape']) / (1.0 / _ws['mean_wape']).sum()))
        _lgb_models = [m for m in [GLOBAL_MODEL_NAME] if m in IG_W]
        _cv = cv_fold_predictions[cv_fold_predictions['actual'] > 0].copy()
        _cv['ds'] = pd.to_datetime(_cv['ds'])
        _cv = _cv.sort_values(['model', 'fold', 'ds'])
        _cv['horizon_week'] = _cv.groupby(['model', 'fold']).cumcount() + 1
        _org = _cv.groupby('fold')['ds'].transform('min') - pd.Timedelta(days=7)
        IG_BT = _cv.assign(origin_date=_org)
        IG_HO = holdout_benchmark_predictions.copy(); IG_HO['origin_date'] = 'holdout'
        _mults = tuple(globals().get('LGB_WEIGHT_MULT_OPTIONS', (1.0, 0.5, 0.0)))
        assert len(IG_W) >= 2, f'need >= 2 ensemble members, have {list(IG_W)}'
        _bt = {m: ig_ensemble_wide(IG_BT, IG_W, _lgb_models, m, ()) for m in _mults}
        _ho = {m: ig_ensemble_wide(IG_HO, IG_W, _lgb_models, m, ())[['ds', 'actual', 'ens']] for m in _mults}
        IG_BT_BASE, IG_HO_BASE = _bt, _ho
        if not _lgb_models:
            IG_BT_SEL, IG_HO_SEL = _bt[1.0], _ho[1.0]
            print('13H-A: LightGBM is not in the live ensemble (Step 6/6B gate) -> weight multiplier N/A, stays 1.0.')
        else:
            _res = ig_lgb_weight_eval(_bt, _ho, _mults, LGB_WEIGHT_MARGIN_PP, LGB_HOLDOUT_BIAS_GATE_PCT, min_prior_origins=2)
            IG_RESULTS['lgb'] = _res
            LGB_WEIGHT_MULT_SELECTED = float(_res['selected'])
            print('=== 13H-A  LightGBM weight multiplier (weekly WAPE %, bias % = fc vs actual) ===')
            print(_res['table'].round(2).to_string(index=False))
            if _res['oos']:
                print(f"  Expanding-window selector vs fixed 1.0 (OOS, {_res['oos']['n_origins']} folds): "
                      f"{_res['oos']['selector_wape']:.2f}% vs {_res['oos']['fixed_1_wape']:.2f}%")
            print(f"  DECISION: LGB_WEIGHT_MULT_SELECTED = {LGB_WEIGHT_MULT_SELECTED} -- {_res['reason']}")
            _k = LGB_WEIGHT_MULT_SELECTED if LGB_WEIGHT_MULT_SELECTED in _bt else 1.0
            IG_BT_SEL, IG_HO_SEL = _bt[_k], _ho[_k]
            _w2 = dict(IG_W)
            for _m in _lgb_models:
                _w2[_m] = _w2[_m] * LGB_WEIGHT_MULT_SELECTED
            _s = sum(_w2.values()) or 1.0
            print('  Weights before -> after (renormalised): ' + ', '.join(f'{m}: {IG_W[m]:.3f} -> {_w2[m] / _s:.3f}' for m in IG_W))
    except Exception as _exc_lgb:
        LGB_WEIGHT_MULT_SELECTED = 1.0
        for _n in ('IG_BT_SEL', 'IG_HO_SEL'):
            globals().pop(_n, None)
        print(f'13H-A skipped, LGB weight stays 1.0 (fail-soft): {type(_exc_lgb).__name__}: {str(_exc_lgb)[:250]}')
else:
    print('13H-A: IMPROVEMENT_GATES_ENABLED=False -> LGB weight multiplier 1.0 (previous behaviour).')

In [ ]:
# =============================================================================
# STEP 13H-B: EVIDENCE GATES -- QUARTER-END OPTION AND LEVEL-SHIFT CORRECTION
# =============================================================================
# OOS weekly series = walk-forward CV ensemble (one forecast per week, 1-13 weeks ahead) + holdout
# ensemble weeks not already covered, both built with the SELECTED LightGBM multiplier (13H-A).
# NOTE: the 13-week validation slice has no retained per-model predictions, so it is NOT in this
# series (no data is invented); quarters need >= 12 OOS weeks to be scored. If the evidence is too
# thin the gate says so and keeps today's behaviour (multiplicative ITSG QE calibration, no level shift).
# Expanding window, earlier quarters/weeks only. Fail-soft.
QE_METHOD_SELECTED = 'incumbent'; QE_DATA_ADD_USD = 0.0
LEVEL_ADJ_ENABLED_SELECTED = False; LEVEL_ADJ_DELTA_SELECTED = 0.0
IG_WK = None
if globals().get('IMPROVEMENT_GATES_ENABLED', False) and 'IG_BT_SEL' in globals():
    try:
        _wk_bt = ig_oos_weekly(IG_BT_SEL, QE_EVAL_MAX_HORIZON)
        _wk_ho = IG_HO_SEL.dropna().rename(columns={'ens': 'fc'})[['ds', 'actual', 'fc']]
        _wk_ho['ds'] = pd.to_datetime(_wk_ho['ds'])
        IG_WK = pd.concat([_wk_bt, _wk_ho[~_wk_ho['ds'].isin(_wk_bt['ds'])]]).sort_values('ds').reset_index(drop=True)
        print(f'13H-B OOS weekly series: {len(IG_WK)} weeks {IG_WK["ds"].min().date()} -> {IG_WK["ds"].max().date()} '
              f'({len(_wk_bt)} from walk-forward CV, {len(IG_WK) - len(_wk_bt)} from holdout)')
    except Exception as _exc_wk:
        print(f'13H-B setup failed (fail-soft, incumbent behaviour kept): {type(_exc_wk).__name__}: {str(_exc_wk)[:200]}')
if IG_WK is not None:
    try:
        _flat_on = float(ITSG_QE_FLAT_USD) > 0
        _q = ig_qe_options_eval(IG_WK, float(ITSG_QE_FLAT_USD) if _flat_on else 200_000.0, QE_EVAL_MIN_PRIOR_QUARTERS,
                                QE_EVAL_SHRINK_K, QE_EVAL_MULT_CAP, QE_EVAL_MARGIN_FRAC, _flat_on,
                                min_scored=QE_EVAL_MIN_SCORED_QUARTERS)
        _sel, _cf = ig_qe_confirm(_q.get('detail'), _q['selected'])
        if _sel != _q['selected']:
            _q['reason'] += f' | confirmation: {_cf}'; _q['selected'] = _sel
        elif _cf:
            _q['reason'] += f' | {_cf}'
        IG_RESULTS['qe'] = _q
        QE_METHOD_SELECTED = _q['selected']
        QE_DATA_ADD_USD = float(_q['data_add_usd']) if QE_METHOD_SELECTED == 'data_add' else 0.0
        print('=== 13H-B1  Quarter-end options, last week of quarter (USD; err = actual - forecast) ===')
        if len(_q['table']):
            print(_q['table'].round(0).to_string(index=False))
            _d2 = ig_qe2_diag(IG_WK, QE_EVAL_MIN_PRIOR_QUARTERS, QE_EVAL_MULT_CAP)
            if len(_d2):
                print('  last-2-weeks mean |error| (USD), none vs multiplicative (prior-quarter ratio):')
                print(_d2.round(2).to_string(index=False))
        print(f"  DECISION: QE_METHOD_SELECTED = {QE_METHOD_SELECTED} -- {_q['reason']}")
        if QE_METHOD_SELECTED == 'data_add':
            print(f'  data-driven add that WILL be applied to the forward last week: ${QE_DATA_ADD_USD:,.0f} '
                  f'(replaces the multiplicative QE calibration; never stacked)')
    except Exception as _exc_qe:
        QE_METHOD_SELECTED = 'incumbent'; QE_DATA_ADD_USD = 0.0
        print(f'13H-B1 QE gate failed (fail-soft, incumbent kept): {type(_exc_qe).__name__}: {str(_exc_qe)[:200]}')
    try:
        _lv = ig_level_eval(IG_BT_SEL, IG_WK, LEVEL_ADJ_K_WEEKS, LEVEL_ADJ_SHRINK_K, LEVEL_ADJ_MODE, LEVEL_ADJ_CLAMP,
                            LEVEL_ADJ_H_WEEKS, LEVEL_ADJ_HALFLIFE_WEEKS, LEVEL_ADJ_MARGIN_PP, LEVEL_ADJ_MIN_ORIGINS)
        IG_RESULTS['level'] = _lv
        LEVEL_ADJ_ENABLED_SELECTED = bool(_lv['adopt'])
        LEVEL_ADJ_DELTA_SELECTED = ig_level_delta(IG_WK, LEVEL_ADJ_K_WEEKS, LEVEL_ADJ_SHRINK_K, LEVEL_ADJ_MODE,
                                                  LEVEL_ADJ_CLAMP) if LEVEL_ADJ_ENABLED_SELECTED else 0.0
        print(f'=== 13H-B2  Level-shift correction (walk-forward origins, first {LEVEL_ADJ_H_WEEKS} weeks ahead, mode={LEVEL_ADJ_MODE}) ===')
        print(f"  origins={_lv.get('n_origins')}  {_lv['reason']}")
        print(f'  DECISION: LEVEL_ADJ_ENABLED_SELECTED = {LEVEL_ADJ_ENABLED_SELECTED}'
              + (f'; current delta = {LEVEL_ADJ_DELTA_SELECTED:+.4f} ({LEVEL_ADJ_MODE})' if LEVEL_ADJ_ENABLED_SELECTED else ''))
    except Exception as _exc_lv:
        LEVEL_ADJ_ENABLED_SELECTED = False; LEVEL_ADJ_DELTA_SELECTED = 0.0
        print(f'13H-B2 level gate failed (fail-soft, no adjustment): {type(_exc_lv).__name__}: {str(_exc_lv)[:200]}')
else:
    print('13H-B: not evaluated -> incumbent QE behaviour (multiplicative calibration), no level adjustment.')

### Step 7 — Production forward forecast **(CHANGED — no holdout-derived scaling, clean weights, coherent markets)**

Generates the 27-week forward forecast (week x market, p05/p25/p50/p75/p95) and writes it to
`itsg_forecast_v5` (primary) and `itsg_forecast_v4` (legacy mirror). Structural changes vs V4:

- **The 0.831 "MC calibration" multiplier is gone.** V4 estimated a bias factor from holdout errors and
  applied it to all production forecasts — a transform present in production but absent from every
  reported metric, tuned on the window used to rank models. In v2, nothing is applied in production that
  wasn't applied identically during evaluation. If the ensemble is biased, that shows up honestly in the
  holdout table (Step 8) instead of being patched after the fact.
- **Ensemble weights = inverse mean CV WAPE** over clean folds (recorded in the output). V4's path ended
  at inverse-*holdout*-WAPE, contaminating its own holdout claims. LightGBM participates only if it passed
  the Step 6 validation gate.
- **Statistical models' p05/p25/p75/p95 ratios come from CV residuals** (clean); the ensemble's forward
  interval is calibrated on its holdout residuals — acceptable for a *forward* interval and labeled as
  such, but never reported as out-of-sample coverage. LGB keeps its native 5-level quantiles.
- **LightGBM future features come from training-only week-of-quarter profiles** per market (V4 read
  future `forecast_prophet` rows when present, zeros when not).
- **MinT market reconciliation clips negatives and rescales** so markets are non-negative *and* still sum
  exactly to each model's total (V4 clipped without rescaling, silently breaking coherence).

In [ ]:
# STEP 7: PRODUCTION FORWARD FORECAST
# =============================================================================
# PURPOSE: Generate the forward weekly forecast from the week after the last
# closed week through the end of the forecast horizon (27 weeks), at
# (week x market) grain.  Each model is retrained on ALL available history.
# The output is written to Delta for FP&A consumption.
#
# Output tables:
#   itsg_forecast_v5 (primary) — consumed by dashboards
#   itsg_forecast_v4 (legacy mirror) — backward compatibility
#
# Outputs produced:
#   production_quarterly_forecasts — quarterly aggregated (all models)
#   ensemble_quarterly_forecasts   — Adaptive Ensemble P05/P25/P50/P75/P95
#   ensemble_weights               — model weights (inverse CV WAPE)
#   ensemble_holdout_history       — weekly holdout p50/actual/error
#
# CHANGED vs V4:
#   * REMOVED the holdout-derived 0.831 global scaling (production must equal
#     the evaluated pipeline).
#   * Ensemble weights = inverse mean CV WAPE (clean folds), method recorded.
#     LGB included only if it passed the Step 6 validation gate.
#   * Stat-model interval ratios from CV residuals (now at p05/p25/p75/p95);
#     ensemble forward interval from holdout residuals (forward-only, labeled).
#   * LGB future features from training-only WoQ profiles per market.
#   * MinT allocation clips negatives THEN rescales to preserve the total.
# UNCHANGED: output tables, overwrite mode + run_timestamp lineage, ZORDER
#   maintenance (wrapped), 27-week horizon.
# =============================================================================

PRIMARY_PRODUCTION_OUTPUT_TABLE = 'datagroup_mdl.mdl_sales_analytics.itsg_forecast_v5'
LEGACY_PRODUCTION_OUTPUT_TABLE = 'datagroup_mdl.mdl_sales_analytics.itsg_forecast_v4'
APP_FORECAST_OUTPUT_TABLE = 'datagroup_mdl.mdl_sales_analytics.arr_forecast_app_latest'
APP_FORECAST_PRODUCT_GROUP = 'ITSG'
APP_FORECAST_NOTEBOOK_NAME = 'ITSG Growth ARR Forecast Foundation V5'
MONITORING_BASELINE_TABLE = 'datagroup_mdl.mdl_sales_analytics.itsg_forecast_monitoring_baseline'
NOTEBOOK_VERSION = 'v5_ucc_aligned'
FORECAST_HORIZON_MONTHS = 18   # was a fixed 27 weeks; 18 months per management request 2026-10-27
FORECAST_HORIZON_WEEKS = int(round(FORECAST_HORIZON_MONTHS * 52 / 12))   # 78 weeks
RUN_TIMESTAMP_UTC = pd.Timestamp.utcnow().tz_localize(None)
RUN_DATE_UTC = RUN_TIMESTAMP_UTC.normalize()
HIERARCHY_MARKETS = list(ALL_MARKETS)
# Current fiscal year, derived from the data's actual max closed week rather than
# a hardcoded literal (e.g. 'FY2026') that would need a manual bump every year.
# Reused by Step 7B (Monte Carlo) and the Step 8 final summary table.
CURRENT_FY = int(weekly_itsg['ds'].max().year)

assert 'ENSEMBLE_COMPONENT_MODELS' in globals() and len(ENSEMBLE_COMPONENT_MODELS) >= 2, 'Run Step 6B before Step 7 so the live ensemble membership is defined.'
PRODUCTION_BENCHMARK_MODELS = list(ENSEMBLE_COMPONENT_MODELS)
print(f'Production ensemble members (validated in Step 6B): {PRODUCTION_BENCHMARK_MODELS}')

FORWARD_START = OBSERVED_END + pd.Timedelta(weeks=1)
future_weeks = pd.date_range(FORWARD_START, periods=FORECAST_HORIZON_WEEKS, freq='W-MON')

# --- Ensemble weights: inverse mean CV WAPE (clean folds; Fix 4 analogue) -------
weights = cv_benchmark_summary[cv_benchmark_summary['model'].isin(PRODUCTION_BENCHMARK_MODELS)][['model', 'mean_wape']].copy()
assert len(weights) == len(PRODUCTION_BENCHMARK_MODELS), f'Missing CV summaries: {weights["model"].tolist()}'
weights['raw_weight'] = 1.0 / weights['mean_wape']
weights['ensemble_weight'] = weights['raw_weight'] / weights['raw_weight'].sum()
# --- Evidence-gated LightGBM weight multiplier (Step 13H-A; default 1.0 = unchanged) ----
_lgb_mult_prod = float(globals().get('LGB_WEIGHT_MULT_SELECTED', 1.0)) if globals().get('IMPROVEMENT_GATES_ENABLED', False) else 1.0
if _lgb_mult_prod != 1.0 and GLOBAL_MODEL_NAME in set(weights['model']):
    _w_before = weights.set_index('model')['ensemble_weight'].copy()
    weights.loc[weights['model'] == GLOBAL_MODEL_NAME, 'ensemble_weight'] *= _lgb_mult_prod
    weights['ensemble_weight'] = weights['ensemble_weight'] / weights['ensemble_weight'].sum()
    _w_after = weights.set_index('model')['ensemble_weight']
    weights['weighting_method'] = f'inverse_cv_wape_lgb_x{_lgb_mult_prod:g}'
    print(f'LGB weight multiplier {_lgb_mult_prod} ADOPTED (Step 13H-A). Weights before -> after: '
          + ', '.join(f'{m}: {_w_before[m]:.3f} -> {_w_after[m]:.3f}' for m in _w_after.index))
else:
    print('LGB weight multiplier: 1.0 (unchanged weights).')
weights['weighting_method'] = 'inverse_cv_wape'
ensemble_weights = weights.sort_values('ensemble_weight', ascending=False).reset_index(drop=True)

# --- Interval ratios for stat models: from that model's CLEAN CV residuals -----
model_interval_adjustments = {}
for model_name in ['Prophet_trend', 'MSTL_v2', 'ETS', 'DHR_ARIMA']:
    cv_slice = cv_fold_predictions[cv_fold_predictions['model'] == model_name][['actual', 'forecast']].copy()
    # CHANGED: derive_interval_adjustments now returns FOUR relative adjustments
    # (p05/p25/p75/p95) instead of two (p10/p90).
    model_interval_adjustments[model_name] = derive_interval_adjustments(cv_slice)
print('Interval ratio adjustments (from CV residuals): ' +
      ', '.join(f'{m}: [{a05:+.2f}, {a25:+.2f}, {a75:+.2f}, {a95:+.2f}]'
                for m, (a05, a25, a75, a95) in model_interval_adjustments.items()))

# --- Forward forecasts: statistical models (tuned params, full history) ---------
production_predictions = []
for model_name, fit_fn, kwargs in benchmark_model_specs:
    pred = np.asarray(fit_fn(model_df[['ds', 'y']].copy(), horizon=FORECAST_HORIZON_WEEKS, **kwargs), dtype=float)
    frame = pd.DataFrame({'ds': future_weeks, 'model': model_name, 'forecast': pred})
    a05, a25, a75, a95 = model_interval_adjustments[model_name]
    frame = apply_interval_adjustments(frame, a05, a25, a75, a95)
    frame['model'] = model_name
    production_predictions.append(frame[['ds', 'model', 'forecast', 'p05', 'p25', 'p50', 'p75', 'p95']])

# DHR forward (training-only future profiles inside the fit function)
dhr_fwd = np.asarray(fit_predict_dhr_arima(model_df[['ds', 'y']].copy(), horizon=FORECAST_HORIZON_WEEKS,
                                           full_feature_df=dhr_df, **BEST_DHR_PARAMS), dtype=float)
dhr_frame = pd.DataFrame({'ds': future_weeks, 'model': 'DHR_ARIMA', 'forecast': dhr_fwd})
a05, a25, a75, a95 = model_interval_adjustments['DHR_ARIMA']
dhr_frame = apply_interval_adjustments(dhr_frame, a05, a25, a75, a95)
dhr_frame['model'] = 'DHR_ARIMA'
production_predictions.append(dhr_frame[['ds', 'model', 'forecast', 'p05', 'p25', 'p50', 'p75', 'p95']])

# --- LGB forward: complete future spine, training-only WoQ profile features -----
def build_forward_lgb_panel():
    fwd = pd.MultiIndex.from_product([future_weeks, HIERARCHY_MARKETS], names=['ds', 'sales_market']).to_frame(index=False)
    fwd['y'] = np.nan
    fwd['deal_count'] = 0.0
    fwd['slice_id'] = PRODUCT_SCOPE + '|' + fwd['sales_market']
    fwd = add_calendar_columns(fwd)
    # Training-only WoQ median profiles per market for the raw feature columns
    hist = global_panel.sort_values('ds').groupby('slice_id').tail(104)
    for col in FP_FEATURE_COLS:
        profile = hist.groupby(['sales_market', 'week_of_qtr'])[col].median().to_dict()
        market_default = hist.groupby('sales_market')[col].median().to_dict()
        fwd[col] = [profile.get((m, wk), market_default.get(m, 0.0)) for m, wk in zip(fwd['sales_market'], fwd['week_of_qtr'])]
    fwd['month_key'] = fwd['ds'].dt.to_period('M').astype(str)
    future_auth = load_authoritative_actuals(fwd['ds'].min(), fwd['ds'].max())
    fwd = fwd.merge(future_auth, on=['month_key', 'sales_market'], how='left')
    base_cols = ['ds', 'sales_market', 'slice_id', 'y', 'deal_count', 'month_key', 'authoritative_actuals'] + FP_FEATURE_COLS
    hist_base = global_panel[base_cols].copy()
    combined = pd.concat([hist_base, fwd[base_cols]], ignore_index=True)
    combined = add_calendar_columns(combined)
    return _engineer_panel_features(combined)


forward_lgb_panel = build_forward_lgb_panel()
production_lgb_models, production_feature_importance = fit_global_quantile_models(global_panel, params=BEST_LGB_PARAMS)
future_global_scored = recursive_lgb_predict(production_lgb_models, forward_lgb_panel, FORWARD_START, future_weeks.max())
production_predictions.append(
    future_global_scored.groupby('ds', as_index=False)
    .agg(forecast=('p50', 'sum'), p05=('p05', 'sum'), p25=('p25', 'sum'), p50=('p50', 'sum'),
         p75=('p75', 'sum'), p95=('p95', 'sum'))
    .assign(model=GLOBAL_MODEL_NAME)[['ds', 'model', 'forecast', 'p05', 'p25', 'p50', 'p75', 'p95']]
)

production_total_forecasts = pd.concat(production_predictions, ignore_index=True)
# NOTE (CHANGED): V4 multiplied everything by a holdout-derived 0.831 factor
# here. Removed — no production transform may differ from the evaluated one.

# --- Ensemble total + holdout residual history -----------------------------------
weight_map = ensemble_weights[['model', 'ensemble_weight']]
ensemble_holdout = (
    holdout_benchmark_predictions[holdout_benchmark_predictions['model'].isin(PRODUCTION_BENCHMARK_MODELS)]
    .merge(weight_map, on='model', how='inner')
)
ensemble_holdout = (
    ensemble_holdout.assign(wf=lambda d: d['forecast'] * d['ensemble_weight'])
    .groupby('ds', as_index=False)
    .agg(actual=('actual', 'first'), forecast=('wf', 'sum'))
)
ensemble_holdout['error'] = ensemble_holdout['actual'] - ensemble_holdout['forecast']
# Forward-only interval calibration from holdout residual ratios (labeled as such).
# CHANGED: 4 adjustments (p05/p25/p75/p95) instead of 2 (p10/p90).
ensemble_p05_adj, ensemble_p25_adj, ensemble_p75_adj, ensemble_p95_adj = derive_interval_adjustments(ensemble_holdout)

ensemble_total = (
    production_total_forecasts[production_total_forecasts['model'].isin(PRODUCTION_BENCHMARK_MODELS)]
    .merge(weight_map, on='model', how='inner')
    .assign(wf=lambda d: d['forecast'] * d['ensemble_weight'])
    .groupby('ds', as_index=False)
    .agg(forecast=('wf', 'sum'))
)
ensemble_total = apply_interval_adjustments(
    ensemble_total, ensemble_p05_adj, ensemble_p25_adj, ensemble_p75_adj, ensemble_p95_adj)
ensemble_total['model'] = 'Adaptive_Ensemble'
itsg_ensemble_total_raw = ensemble_total[['ds', 'forecast', 'p50']].copy()   # pre-calibration snapshot (Step 14F pace)

# --- Evidence-gated level-shift correction (Step 13H-B2; only if its gate passed) ----------
if (globals().get('IMPROVEMENT_GATES_ENABLED', False) and globals().get('LEVEL_ADJ_ENABLED_SELECTED', False)
        and float(globals().get('LEVEL_ADJ_DELTA_SELECTED', 0.0)) != 0.0):
    try:
        _lv_d = float(LEVEL_ADJ_DELTA_SELECTED)
        ensemble_total = ensemble_total.sort_values('ds').reset_index(drop=True)
        _lv_h = np.arange(1, len(ensemble_total) + 1, dtype=float)
        _lv_dec = np.where(_lv_h <= LEVEL_ADJ_H_WEEKS, 0.5 ** ((_lv_h - 1) / LEVEL_ADJ_HALFLIFE_WEEKS), 0.0)
        for _c_lv in ['forecast', 'p05', 'p25', 'p50', 'p75', 'p95']:
            if LEVEL_ADJ_MODE == 'add':
                ensemble_total[_c_lv] = ensemble_total[_c_lv] + _lv_d * _lv_dec
            else:
                ensemble_total[_c_lv] = ensemble_total[_c_lv] * (1.0 + _lv_d * _lv_dec)
        print(f'Level-shift correction ACTIVE ({LEVEL_ADJ_MODE}, delta={_lv_d:+.4f}, first {LEVEL_ADJ_H_WEEKS} weeks, '
              f'half-life {LEVEL_ADJ_HALFLIFE_WEEKS}w); week-1 change = {_lv_d * _lv_dec[0]:+.4f}')
    except Exception as _exc_lva:
        print(f'Level-shift correction skipped (fail-soft): {_exc_lva}')
else:
    print('Level-shift correction: not adopted (gate not passed or disabled).')

# ── QUARTER-END CALIBRATION (2026-10-05, ported from UCC) ───────────────────
# EVIDENCE: Step 8-QE's holdout bias diagnostic shows the Adaptive Ensemble
# systematically under-forecasts quarter-end weeks (-18.91% bias) while
# running positive elsewhere (+7.39%) -- a ~26pp swing, and EVERY individual
# model shows the same directional miss (not just one bad model, unlike
# UCC's LGB-specific issue). This lines up with Step 7B-QE's seasonal surge
# diagnostic showing real 1.4x-2.3x quarter-end ARR surges that the model
# family isn't capturing in its point forecasts. Fix: the same post-ensemble
# multiplicative calibration UCC uses -- corrects the POINT FORECAST only
# for forward quarter-end weeks, scaled by the live holdout actual/forecast
# ratio on those same weeks. Interval widths are intentionally left alone:
# Step 7B's QE_DISPERSION_RATIO genuinely came back ~1.00x (confirmed at the
# per-quarter level too in Step 7B-QE), so there's no evidence the SPREAD of
# errors is wider at quarter-end here -- only the center is biased.
ITSG_QE_CALIBRATION_ENABLED = True
ITSG_QE_CALIBRATION_WEEKS_FROM_END = 2
ITSG_QE_CALIBRATION_MAX_ADJUSTMENT = 0.50  # safety cap, same as UCC
ITSG_QE_CALIBRATION_MIN_OBS = 3            # minimum holdout QE weeks required

_itsg_qe_method = str(globals().get('QE_METHOD_SELECTED', 'incumbent')) if globals().get('IMPROVEMENT_GATES_ENABLED', False) else 'incumbent'
print(f'ITSG QE method in effect: {_itsg_qe_method} (incumbent/mult = multiplicative calibration below; '
      f'flat/data_add = additive on last week of quarter; none = no QE adjustment)')
if ITSG_QE_CALIBRATION_ENABLED and _itsg_qe_method in ('incumbent', 'mult'):
    _itsg_qe_ho = ensemble_holdout.copy()
    _itsg_qe_ho['ds'] = pd.to_datetime(_itsg_qe_ho['ds'])
    _itsg_qe_ho['quarter'] = _itsg_qe_ho['ds'].dt.quarter
    _itsg_qe_ho['year'] = _itsg_qe_ho['ds'].dt.year
    _itsg_qe_ho['week_of_qtr'] = _itsg_qe_ho.groupby(['year', 'quarter']).cumcount() + 1
    _itsg_max_woq = _itsg_qe_ho.groupby(['year', 'quarter'])['week_of_qtr'].transform('max')
    _itsg_qe_ho['is_qe'] = (_itsg_max_woq - _itsg_qe_ho['week_of_qtr'] < ITSG_QE_CALIBRATION_WEEKS_FROM_END)
    _itsg_qe_rows = _itsg_qe_ho[_itsg_qe_ho['is_qe'] & _itsg_qe_ho['actual'].notna() & (_itsg_qe_ho['forecast'] > 0)]
    if len(_itsg_qe_rows) >= ITSG_QE_CALIBRATION_MIN_OBS:
        _itsg_qe_factor = float(_itsg_qe_rows['actual'].sum() / _itsg_qe_rows['forecast'].sum())
        _itsg_qe_factor = min(_itsg_qe_factor, 1.0 + ITSG_QE_CALIBRATION_MAX_ADJUSTMENT)
        _itsg_qe_factor = max(_itsg_qe_factor, 1.0)  # only uplift, never suppress
        _itsg_fwd_q = ensemble_total.copy()
        _itsg_fwd_q['ds'] = pd.to_datetime(_itsg_fwd_q['ds'])
        _itsg_fwd_q['quarter'] = _itsg_fwd_q['ds'].dt.quarter
        _itsg_fwd_q['year'] = _itsg_fwd_q['ds'].dt.year
        _itsg_fwd_q['week_of_qtr'] = _itsg_fwd_q.groupby(['year', 'quarter']).cumcount() + 1
        _itsg_max_woq_fwd = _itsg_fwd_q.groupby(['year', 'quarter'])['week_of_qtr'].transform('max')
        _itsg_fwd_qe_mask = (_itsg_max_woq_fwd - _itsg_fwd_q['week_of_qtr'] < ITSG_QE_CALIBRATION_WEEKS_FROM_END).to_numpy()
        for _col_qe in ['forecast', 'p05', 'p25', 'p50', 'p75', 'p95']:
            ensemble_total.loc[_itsg_fwd_qe_mask, _col_qe] *= _itsg_qe_factor
        print(f'ITSG QE calibration ACTIVE: factor={_itsg_qe_factor:.3f} applied to '
              f'{int(_itsg_fwd_qe_mask.sum())} forward QE weeks (holdout had '
              f'{len(_itsg_qe_rows)} QE obs, actual/forecast ratio = {_itsg_qe_factor:.3f})')
    else:
        print(f'ITSG QE calibration SKIPPED: only {len(_itsg_qe_rows)} holdout QE weeks '
              f'(need >= {ITSG_QE_CALIBRATION_MIN_OBS}). Forward forecast unchanged.')
elif ITSG_QE_CALIBRATION_ENABLED:
    print(f'ITSG QE multiplicative calibration NOT applied (method={_itsg_qe_method}).')
else:
    print('ITSG QE calibration: disabled (ITSG_QE_CALIBRATION_ENABLED=False).')

# --- Additive QE options (evidence-gated; mutually exclusive with the multiplicative block) -------
_itsg_flat = float(globals().get('ITSG_QE_FLAT_USD', 0.0))
_itsg_add = 0.0
if _itsg_qe_method == 'flat':
    _itsg_add = _itsg_flat
elif _itsg_qe_method == 'data_add':
    _itsg_add = float(globals().get('QE_DATA_ADD_USD', 0.0))
elif _itsg_qe_method == 'incumbent' and ITSG_QE_CALIBRATION_ENABLED:
    _itsg_add = _itsg_flat          # 0 by default => today's behaviour exactly
if _itsg_add != 0.0:
    _itsg_ds = pd.to_datetime(ensemble_total['ds'])
    _itsg_last = ((_itsg_ds + pd.Timedelta(days=7)).dt.quarter != _itsg_ds.dt.quarter).to_numpy()
    for _col_qe in ['forecast', 'p05', 'p25', 'p50', 'p75', 'p95']:
        ensemble_total.loc[_itsg_last, _col_qe] += _itsg_add
    print(f'ITSG QE additive adjustment ACTIVE ({_itsg_qe_method}): +${_itsg_add:,.0f} on {int(_itsg_last.sum())} forward last-week-of-quarter rows')

production_total_forecasts = pd.concat([
    production_total_forecasts[['ds', 'model', 'forecast', 'p05', 'p25', 'p50', 'p75', 'p95']],
    ensemble_total[['ds', 'model', 'forecast', 'p05', 'p25', 'p50', 'p75', 'p95']],
], ignore_index=True)

# --- MinT market reconciliation (clip + rescale keeps coherence) ------------------
recent_market_shares = (
    weekly_itsg_by_market[weekly_itsg_by_market['ds'] >= OBSERVED_END - pd.Timedelta(weeks=13)]
    .groupby('sales_market', as_index=False).agg(y=('y', 'sum'))
)
recent_market_shares['share'] = recent_market_shares['y'] / max(recent_market_shares['y'].sum(), 1e-9)
share_map = dict(zip(recent_market_shares['sales_market'], recent_market_shares['share']))


def estimate_mint_covariances(cv_preds_df, history_market_df, lookback_weeks=26):
    market_hist = history_market_df.copy()
    market_hist['ds'] = pd.to_datetime(market_hist['ds'])
    actual_wide = (
        market_hist.pivot_table(index='ds', columns='sales_market', values='y', aggfunc='sum')
        .reindex(columns=HIERARCHY_MARKETS).fillna(0.0).sort_index()
    )
    total_actual = actual_wide.sum(axis=1)
    fb_top = float((total_actual.diff().dropna() ** 2).mean()) if total_actual.diff().dropna().shape[0] else 1.0
    fb_bottom = np.atleast_2d(np.cov(actual_wide.diff().dropna().to_numpy(dtype=float), rowvar=False))
    if fb_bottom.shape != (len(HIERARCHY_MARKETS), len(HIERARCHY_MARKETS)):
        fb_bottom = np.diag(np.maximum(actual_wide.var().reindex(HIERARCHY_MARKETS).fillna(1.0).to_numpy(dtype=float), 1.0))
    covs = {}
    for model_name in PRODUCTION_BENCHMARK_MODELS:
        preds = cv_preds_df[cv_preds_df['model'] == model_name].sort_values(['fold', 'ds'])
        rows = []
        for row in preds.itertuples(index=False):
            ds = pd.to_datetime(row.ds)
            if ds not in actual_wide.index:
                continue
            prior = actual_wide.loc[actual_wide.index < ds].tail(lookback_weeks)
            if prior.empty:
                continue
            shares = prior.sum(axis=0)
            st = float(shares.sum())
            shares = shares / st if st > 0 else pd.Series(1.0 / len(HIERARCHY_MARKETS), index=HIERARCHY_MARKETS)
            bottom_base = float(row.forecast) * shares.reindex(HIERARCHY_MARKETS).fillna(0.0).to_numpy(dtype=float)
            rows.append(np.concatenate([[float(row.actual) - float(row.forecast)],
                                        actual_wide.loc[ds, HIERARCHY_MARKETS].to_numpy(dtype=float) - bottom_base]))
        if len(rows) >= 6:
            cov = np.cov(np.vstack(rows), rowvar=False)
        else:
            cov = np.zeros((len(HIERARCHY_MARKETS) + 1, len(HIERARCHY_MARKETS) + 1))
            cov[0, 0] = fb_top
            cov[1:, 1:] = fb_bottom
        covs[model_name] = np.atleast_2d(cov) + np.eye(len(HIERARCHY_MARKETS) + 1) * 1e-6
    return covs


def mint_reconcile(top_value, market_values, covariance):
    bottom = np.asarray(list(market_values), dtype=float)
    S = np.vstack([np.ones(len(bottom)), np.eye(len(bottom))])
    yhat = np.concatenate([[float(top_value)], bottom])
    try:
        inv_w = np.linalg.pinv(np.asarray(covariance, dtype=float))
        G = np.linalg.pinv(S.T @ inv_w @ S) @ S.T @ inv_w
        reconciled = S @ (G @ yhat)
    except Exception:
        reconciled = S @ (np.linalg.pinv(S.T @ S) @ S.T @ yhat)
    return float(reconciled[0]), reconciled[1:]


# --- Per-market interval-width scaling (CHANGED — fixes uniform-ratio bands) --
# Previously every market's p10/p90 used the SAME relative ratio computed at
# the TOTAL level (p10r = row.p10/top, p90r = row.p90/top), applied uniformly
# regardless of that market's own volatility/error. That fix is preserved and
# extended here to the full 5-column convention (p05/p25/p50/p75/p95): the
# SAME market-volatility-ratio scale is now applied to both the outer
# (p05/p95) and inner (p25/p75) bands. LGB is unaffected (its market-level
# p05/p25/p50/p75/p95 already come from genuine native per-market recursive
# quantile predictions, from the Step 6 retrain at the 5 quantile levels —
# left untouched below).
#
# Ideally this would scale by each market's own HOLDOUT WAPE relative to the
# total's holdout WAPE (same approach as the UCC fix), but no per-market
# holdout predictions-vs-actuals exist in this session for the benchmark /
# ensemble models: holdout_benchmark_predictions and cv_fold_predictions are
# TOTAL-level series only (Prophet/MSTL/ETS/DHR/Ensemble are all fit on the
# total series, not per market) — only LGB has per-market holdout rows
# (global_test_predictions), and LGB's market path is intentionally left
# alone. So, per the documented fallback, we scale by each market's own
# historical week-over-week RELATIVE VOLATILITY of actuals (std of pct
# changes), normalized against the same statistic for the total series —
# a noisier market gets a wider interval, a calmer market gets a narrower one,
# clipped to [0.5, 3.0] so one sparse/noisy market can't blow up the interval.
_market_pct_changes = (
    weekly_itsg_by_market.sort_values(['sales_market', 'ds'])
    .groupby('sales_market')['y']
    .apply(lambda s: s.pct_change().replace([np.inf, -np.inf], np.nan).dropna())
)
_market_rel_vol = _market_pct_changes.groupby(level=0).std()
_total_rel_vol = float(
    weekly_itsg.sort_values('ds')['y'].pct_change().replace([np.inf, -np.inf], np.nan).dropna().std()
)
if not np.isfinite(_total_rel_vol) or _total_rel_vol <= 0:
    _total_rel_vol = 1.0
market_interval_scale = (
    (_market_rel_vol / _total_rel_vol).reindex(HIERARCHY_MARKETS).fillna(1.0).clip(0.5, 3.0).to_dict()
)
print('Per-market interval-width scale (volatility-relative-to-total, fallback method): '
      + ', '.join(f'{m}={s:.2f}x' for m, s in market_interval_scale.items()))


def coherent_allocate(model_name, weekly_total_df, mint_covs, interval_scale):
    """CHANGED: extended from the 2-band (p10/p90) per-market interval scaling
    to 4 bands (p05/p25/p75/p95). Reuses the SAME market-volatility-ratio
    scale (interval_scale) fixed in the prior pass — not a naive proportional
    split — for both the outer (p05/p95) and the new inner (p25/p75) band."""
    rows = []
    covariance = mint_covs.get(model_name)
    for row in weekly_total_df.itertuples(index=False):
        market_values = [float(row.forecast) * share_map.get(m, 0.0) for m in HIERARCHY_MARKETS]
        if covariance is not None:
            _, bottom = mint_reconcile(float(row.forecast), market_values, covariance)
        else:
            bottom = np.asarray(market_values, dtype=float)
        # CHANGED: clip negatives THEN rescale so markets still sum to the total
        bottom = np.clip(np.asarray(bottom, dtype=float), 0.0, None)
        bsum = float(bottom.sum())
        top = max(float(row.forecast), 0.0)
        if bsum > 0:
            bottom = bottom * (top / bsum)
        # Total-level relative half-widths (fraction below / above p50) — the
        # starting point before market-specific scaling. Outer (p05/p95) and
        # inner (p25/p75) bands each get their own fraction.
        lo05_frac = max(0.0, 1.0 - (float(row.p05) / top)) if top > 0 else 0.0
        hi95_frac = max(0.0, (float(row.p95) / top) - 1.0) if top > 0 else 0.0
        lo25_frac = max(0.0, 1.0 - (float(row.p25) / top)) if top > 0 else 0.0
        hi75_frac = max(0.0, (float(row.p75) / top) - 1.0) if top > 0 else 0.0
        for market, value in zip(HIERARCHY_MARKETS, bottom):
            scale = interval_scale.get(market, 1.0)
            p05_val = value * (1.0 - lo05_frac * scale)
            p25_val = value * (1.0 - lo25_frac * scale)
            p75_val = value * (1.0 + hi75_frac * scale)
            p95_val = value * (1.0 + hi95_frac * scale)
            rows.append({'model': model_name, 'ds': row.ds, 'sales_market': market,
                         'forecast': float(value), 'p05': float(p05_val), 'p25': float(p25_val),
                         'p50': float(value), 'p75': float(p75_val), 'p95': float(p95_val)})
    return enforce_quantile_order(pd.DataFrame(rows), ['p05', 'p25', 'p50', 'p75', 'p95'])


mint_covariances = estimate_mint_covariances(cv_fold_predictions, weekly_itsg_by_market)

market_frames = []
lgb_market = future_global_scored[['ds', 'sales_market', 'p05', 'p25', 'p50', 'p75', 'p95']].copy()
lgb_market['model'] = GLOBAL_MODEL_NAME
lgb_market['forecast'] = lgb_market['p50']
market_frames.append(lgb_market[['model', 'ds', 'sales_market', 'forecast', 'p05', 'p25', 'p50', 'p75', 'p95']])
for model_name in ['Prophet_trend', 'MSTL_v2', 'ETS', 'DHR_ARIMA', 'Adaptive_Ensemble']:
    totals = production_total_forecasts[production_total_forecasts['model'] == model_name][['ds', 'forecast', 'p05', 'p25', 'p50', 'p75', 'p95']]
    market_frames.append(coherent_allocate(model_name, totals, mint_covariances, market_interval_scale)[['model', 'ds', 'sales_market', 'forecast', 'p05', 'p25', 'p50', 'p75', 'p95']])
market_outputs = pd.concat(market_frames, ignore_index=True)

# Coherence check for the ensemble
_check = (market_outputs[market_outputs['model'] == 'Adaptive_Ensemble'].groupby('ds')['p50'].sum().sort_index()
          - ensemble_total.set_index('ds')['p50'].sort_index()).abs()
assert (_check < 1.0).all(), 'Ensemble market reconciliation drifted from the total.'

# --- Persist -----------------------------------------------------------------------
total_outputs = production_total_forecasts.copy()
total_outputs['grain_level'] = 'total'
total_outputs['sales_market'] = 'ALL'
market_export = market_outputs.assign(grain_level='market')
production_output = pd.concat([
    total_outputs[['ds', 'model', 'grain_level', 'sales_market', 'forecast', 'p05', 'p25', 'p50', 'p75', 'p95']],
    market_export[['ds', 'model', 'grain_level', 'sales_market', 'forecast', 'p05', 'p25', 'p50', 'p75', 'p95']],
], ignore_index=True)
production_output = enforce_quantile_order(production_output, ['p05', 'p25', 'p50', 'p75', 'p95'])

# --- REMOVED (v2.2 -> 5-column convention): the old "_v5_model_factors"
# enrichment used to synthesize an extra p20/p30/p40/p60/p70/p80 grid from
# calibrated holdout-residual-percentile factors, on top of the p10/p50/p90
# already present. That grid is now REDUNDANT: production_output already
# carries the full target convention (p05/p25/p50/p75/p95) natively —
# p05/p25/p75/p95 come from apply_interval_adjustments (stat models +
# ensemble, 4-quantile CV/holdout residual calibration) or directly from
# LightGBM's own retrained 5-quantile model (Step 6) — instead of a
# secondary round-number decile fan bolted on afterward. Simplifying here
# avoids maintaining two parallel (and potentially inconsistent) interval
# mechanisms for the same data.

production_output['product_group'] = PRODUCT_SCOPE
production_output['run_timestamp_utc'] = RUN_TIMESTAMP_UTC
production_output['run_date_utc'] = RUN_DATE_UTC
production_output['notebook_version'] = NOTEBOOK_VERSION
production_output['notebook_name'] = APP_FORECAST_NOTEBOOK_NAME
production_output['recommended_for_exec'] = (production_output['model'] == 'Adaptive_Ensemble').astype(int)
production_output['source_table'] = PRIMARY_PRODUCTION_OUTPUT_TABLE

for target_table in [PRIMARY_PRODUCTION_OUTPUT_TABLE, LEGACY_PRODUCTION_OUTPUT_TABLE]:
    spark.createDataFrame(production_output).write \
        .mode('overwrite') \
        .option('overwriteSchema', 'true') \
        .saveAsTable(target_table)

app_forecast_export = production_output[[
    'ds', 'product_group', 'model', 'grain_level', 'sales_market',
    'forecast', 'p05', 'p25', 'p50', 'p75', 'p95',
    'run_timestamp_utc', 'run_date_utc',
    'notebook_version', 'notebook_name', 'recommended_for_exec', 'source_table'
]].copy()
app_forecast_export['ds'] = pd.to_datetime(app_forecast_export['ds'])
app_forecast_sdf = spark.createDataFrame(app_forecast_export)
if spark.catalog.tableExists(APP_FORECAST_OUTPUT_TABLE):
    from pyspark.sql import functions as _F
    # Ensure table schema has the new p05/p25/p75/p95 quantile columns before
    # writing (CHANGED from the old p20/p30/p40/p60/p70/p80 decile-fan columns)
    # 2026-10-05 FIX: this used to always ISSUE the ALTER TABLE and swallow the
    # resulting "column already exists" exception every single run (which it
    # does on every run after the first, forever). The try/except itself is
    # harmless, but Spark Connect logs the exception as a full structured
    # ERROR-level stack trace to the driver logs BEFORE Python catches it --
    # so every run permanently printed a scary-looking GRPC/JVM error that
    # looked like a real failure but wasn't. Checking the existing schema
    # first means ALTER TABLE is only actually issued on the one run where a
    # column is genuinely missing; every later run is silent.
    _existing_app_cols = set(spark.table(APP_FORECAST_OUTPUT_TABLE).columns)
    _missing_app_cols = [c for c in ['p05', 'p25', 'p75', 'p95'] if c not in _existing_app_cols]
    if _missing_app_cols:
        _add_clause = ', '.join(f'{c} DOUBLE' for c in _missing_app_cols)
        spark.sql(f"ALTER TABLE {APP_FORECAST_OUTPUT_TABLE} ADD COLUMNS ({_add_clause})")
        print(f'  Added missing column(s) to {APP_FORECAST_OUTPUT_TABLE}: {_missing_app_cols}')
    app_forecast_sdf.writeTo(APP_FORECAST_OUTPUT_TABLE).overwrite(
        _F.col("product_group") == APP_FORECAST_PRODUCT_GROUP
    )
else:
    app_forecast_sdf.writeTo(APP_FORECAST_OUTPUT_TABLE).partitionedBy('product_group').createOrReplace()
try:
    spark.sql(f'OPTIMIZE {PRIMARY_PRODUCTION_OUTPUT_TABLE} ZORDER BY (run_date_utc)')
    spark.sql(f'OPTIMIZE {LEGACY_PRODUCTION_OUTPUT_TABLE} ZORDER BY (run_date_utc)')
except Exception as exc:
    print(f'OPTIMIZE skipped ({type(exc).__name__}) — data written successfully.')

production_quarterly = production_total_forecasts.copy()
production_quarterly['quarter'] = production_quarterly['ds'].dt.to_period('Q').astype(str)
production_quarterly_forecasts = (
    production_quarterly.groupby(['model', 'quarter'], as_index=False)
    .agg(weeks=('ds', 'nunique'),
         p05_arr_m=('p05', lambda s: round(float(np.nansum(s)) / 1e6, 3)),
         p25_arr_m=('p25', lambda s: round(float(np.nansum(s)) / 1e6, 3)),
         forecast_arr_m=('p50', lambda s: round(float(np.nansum(s)) / 1e6, 3)),
         p75_arr_m=('p75', lambda s: round(float(np.nansum(s)) / 1e6, 3)),
         p95_arr_m=('p95', lambda s: round(float(np.nansum(s)) / 1e6, 3)))
    .sort_values(['quarter', 'model']).reset_index(drop=True)
)

print(f'Forward production horizon: {FORWARD_START.date()} -> {future_weeks.max().date()} ({FORECAST_HORIZON_WEEKS} weeks)')
print(f"\nAdaptive ensemble weights — method: {ensemble_weights['weighting_method'].iloc[0]}:")
display(ensemble_weights)
print('\nRecent market shares (MinT top-down baseline):')
display(recent_market_shares[['sales_market', 'share']])
print('\nQuarterly production forecast summary ($M):')
display(production_quarterly_forecasts)
print(f'\nPersisted: {PRIMARY_PRODUCTION_OUTPUT_TABLE} (primary), {LEGACY_PRODUCTION_OUTPUT_TABLE} (legacy), {APP_FORECAST_OUTPUT_TABLE} (app)')

### Step 14F / 13M - Pace outlook and Miradore outlook *(NEW)*

Pace outlook (evidence-gated) -> `itsg_forecast_pace_outlook`. Miradore outlook is a SEPARATE line (not in the Salesforce ITSG target) -> `itsg_miradore_outlook`, `itsg_forecast_incl_miradore`.

In [ ]:
# =============================================================================
# STEP 14F: IN-QUARTER PACE OUTLOOK (evidence-gated; writes itsg_forecast_pace_outlook)
# =============================================================================
# Outlook = QTD actual + remaining-week forecast x pace factor (+ learned quarter-end uplift on the remaining
# last week). Backtested for every complete OOS quarter at as-of weeks PACE_ASOF_WEEKS using only earlier quarters
# for the uplift, vs the static model quarter forecast (sum of weekly forecasts). Marked validated ONLY at as-of
# weeks where outlook MAPE < model MAPE - PACE_MARGIN_PP over >= PACE_MIN_QUARTERS quarters (thin ITSG history =>
# usually NOT validated, and the row says so). Live basis = itsg_ensemble_total_raw (pre-calibration snapshot from
# Step 7). Existing tables are not touched. Fail-soft.
IG_PACE_SUMMARY = None
if globals().get('IMPROVEMENT_GATES_ENABLED', False) and globals().get('IG_WK') is not None:
    try:
        _pe = ig_pace_eval(IG_WK, PACE_ASOF_WEEKS, PACE_SHRINK_K, PACE_CLAMP[0], PACE_CLAMP[1], PACE_UPLIFT_CAP,
                           PACE_MARGIN_PP, 2, PACE_MIN_QUARTERS)
        IG_PACE_SUMMARY = _pe['summary']; IG_RESULTS['pace'] = _pe
        print('=== 14F  Pace outlook backtest (landing error % vs actual quarter total) ===')
        if len(_pe['summary']):
            print(_pe['summary'].round(2).to_string(index=False))
            print(_pe['detail'].assign(outlook_err_pct=lambda x: x['outlook_err_pct'].round(1),
                                       model_err_pct=lambda x: x['model_err_pct'].round(1))
                  [['quarter', 'asof_week', 'outlook_err_pct', 'model_err_pct']].to_string(index=False))
        else:
            print('  not enough complete OOS quarters to backtest (need >= 3: 2 earlier + 1 scored); outlook rows will be NOT validated.')
        _wa = weekly_itsg.assign(ds=pd.to_datetime(weekly_itsg['ds']))
        _asof = _wa['ds'].max()
        _cq = str(_asof.to_period('Q'))
        _qs_hist = [(q, g) for q, g in ig_complete_quarters(IG_WK) if q < _cq]
        _upl = ig_qe_uplift([g for _, g in _qs_hist], cap=PACE_UPLIFT_CAP)
        _act = _wa[_wa['ds'].dt.to_period('Q').astype(str) == _cq]
        _fw = itsg_ensemble_total_raw.assign(ds=pd.to_datetime(itsg_ensemble_total_raw['ds']))
        _fw = _fw[_fw['ds'].dt.to_period('Q').astype(str) == _cq].sort_values('ds')
        _fc_closed = IG_WK[IG_WK['ds'].isin(_act['ds'])].sort_values('ds')
        if len(_act) == 0 or len(_fw) == 0:
            print(f'  Current quarter {_cq}: no closed weeks or no remaining forward weeks -> no pace row written.')
        else:
            _c = ig_pace_calc(_fc_closed['actual'].values if len(_fc_closed) else [1.0],
                              _fc_closed['fc'].values if len(_fc_closed) else [1.0],
                              _fw['p50'].values, ig_is_last_week(_fw['ds']), _upl,
                              PACE_SHRINK_K, PACE_CLAMP[0], PACE_CLAMP[1])
            _qtd = float(_act['y'].sum()); _w = len(_act)
            _land = _qtd + _c['remaining_adj']
            _ef = ensemble_total.assign(ds=pd.to_datetime(ensemble_total['ds']))
            _model_land = _qtd + float(_ef.loc[_ef['ds'].dt.to_period('Q').astype(str) == _cq, 'p50'].sum())
            _sm = IG_PACE_SUMMARY[IG_PACE_SUMMARY['asof_week'] <= _w].tail(1) if IG_PACE_SUMMARY is not None and len(IG_PACE_SUMMARY) else []
            _val = bool(_sm['validated'].iloc[0]) if len(_sm) else False
            _row = {'run_date': pd.Timestamp.today().date(), 'asof_date': _asof.date(), 'quarter_label': _cq,
                    'weeks_closed': int(_w), 'qtd_actual': _qtd, 'remaining_forecast': float(_c['remaining_forecast']),
                    'pace_factor': float(_c['pace']), 'outlook_landing': float(_land), 'model_landing': float(_model_land),
                    'validated': _val,
                    'backtest_mape_pct': float(_sm['outlook_mape'].iloc[0]) if len(_sm) else np.nan,
                    'model_mape_pct': float(_sm['model_mape'].iloc[0]) if len(_sm) else np.nan,
                    'n_quarters': int(_sm['n_quarters'].iloc[0]) if len(_sm) else 0,
                    'notes': (f'qe_uplift={_upl:.3f}; pace weeks matched={len(_fc_closed)}/{_w}; model_landing = QTD + calibrated '
                              f'production p50; backtest compares with static quarter forecast; '
                              + ('validated at this as-of week' if _val else 'NOT validated at this as-of week - indicative only'))}
            _pdf = pd.DataFrame([_row]); IG_RESULTS['pace_row'] = _row
            print(_pdf.T.to_string(header=False))
            ig_write_table(_pdf, PACE_OUTLOOK_TABLE)
    except Exception as _exc_pace:
        print(f'14F pace outlook skipped (fail-soft): {type(_exc_pace).__name__}: {str(_exc_pace)[:250]}')
else:
    print('14F pace outlook: not run (IMPROVEMENT_GATES_ENABLED=False or 13H-B did not produce the OOS series).')

In [ ]:
# =============================================================================
# STEP 13M: MIRADORE OUTLOOK -- A SEPARATE LINE (NOT part of the Salesforce ITSG forecast/target)
# =============================================================================
# Salesforce ITSG product group has no Miradore genus, so Miradore is outside the target and the models.
# Inputs: miradore_plan (Version, Month of Data Month, Product Genus, value col) and mc_actuals (same schema).
# Method: forecast = monthly plan x shrunk attainment learned from PRIOR KNOWN COMPLETE months only; chosen by
# expanding-window MAE vs raw plan and trailing-3-month mean actual (raw plan wins ties). A month with plan
# but NO actual rows is UNKNOWN (excluded + warning, never zero). Writes itsg_miradore_outlook and the finance
# view itsg_forecast_incl_miradore (clearly labelled). Existing tables untouched. Soft-skips on any mismatch.
MIRADORE_RESULT = None
if globals().get('MIRADORE_OUTLOOK_ENABLED', False):
    try:
        def _mira_cols(tbl, need):
            _c = spark.table(tbl).columns
            _m = [n for n in need if n not in _c]
            assert not _m, f'{tbl} lacks columns {_m}. Actual columns: {_c}'
            return _c
        _pcols = spark.table(MIRADORE_PLAN_TABLE).columns
        _vc = next((c for c in MIRADORE_VALUE_COLS if c in _pcols), None)
        assert _vc, f'{MIRADORE_PLAN_TABLE}: none of {MIRADORE_VALUE_COLS} present. Columns: {_pcols}'
        _mira_cols(MIRADORE_PLAN_TABLE, ['Version', 'Month of Data Month', 'Product Genus', _vc])
        _ca = _mira_cols(MIRADORE_ACTUALS_TABLE, ['Month of Data Month', 'Product Genus', _vc])
        _gen = f"upper(trim(`Product Genus`)) = '{MIRADORE_GENUS.upper()}'"
        _p = (spark.table(MIRADORE_PLAN_TABLE).where(_gen)
              .selectExpr('`Version` AS ver', '`Month of Data Month` AS mo', f'CAST(`{_vc}` AS DOUBLE) AS v').toPandas())
        _bu = "AND upper(trim(`Business Unit`)) = 'ITSG'" if 'Business Unit' in _ca else ''
        if not _bu:
            print("  (mc_actuals has no 'Business Unit' column -> genus filter only)")
        _a = (spark.table(MIRADORE_ACTUALS_TABLE).where(f'{_gen} {_bu}')
              .selectExpr('`Month of Data Month` AS mo', f'CAST(`{_vc}` AS DOUBLE) AS v').toPandas())
        for _d in (_p, _a):
            _d['month'] = pd.to_datetime(_d['mo'], errors='coerce').dt.to_period('M')
            assert _d['month'].notna().mean() > 0.9, f"cannot parse Month of Data Month; sample={_d['mo'].head(3).tolist()}"
        _p = _p.dropna(subset=['month']); _a = _a.dropna(subset=['month'])
        _want = _p['month'].dt.year.map(lambda y: MIRADORE_PLAN_VERSION_FMT.format(yy=y % 100))
        _p = _p[_p['ver'].astype(str).str.strip() == _want]
        assert len(_p) > 0 and len(_a) > 0, f'no Miradore rows (plan={len(_p)}, actual={len(_a)}); plan Version must look like {MIRADORE_PLAN_VERSION_FMT}'
        _pm = _p.groupby('month', as_index=False).agg(plan=('v', 'sum'))
        _am = _a.groupby('month', as_index=False).agg(actual=('v', 'sum'), n_rows=('v', 'size'))
        _cur = pd.Timestamp.today().to_period('M')
        _m = _pm.merge(_am, on='month', how='left').sort_values('month').reset_index(drop=True)
        _m['known'] = _m['n_rows'].fillna(0).gt(0) & (_m['month'] < _cur) & (_m['plan'] > 0)
        _m.loc[~_m['known'], 'actual'] = np.nan
        _unk = _m[(_m['month'] < _cur) & ~_m['known']]['month'].astype(str).tolist()
        if _unk:
            print(f'  WARNING: months with plan but NO Miradore actual rows in mc_actuals: {_unk} -> UNKNOWN (excluded, NOT zero).')
        _m['attain'] = _m['actual'] / _m['plan'].replace(0, np.nan)
        print(f'=== 13M  Miradore plan vs actual ({_vc}; $K; attain = actual/plan %) ===')
        _show = _m.assign(plan=(_m['plan'] / 1e3).round(0), actual=(_m['actual'] / 1e3).round(0), attain=(_m['attain'] * 100).round(0))
        print(_show[['month', 'plan', 'actual', 'attain', 'known']].to_string(index=False))
        _N, _K = MIRADORE_ATT_WINDOW, MIRADORE_ATT_SHRINK_K
        _lo, _hi = MIRADORE_ATT_CLAMP
        _bd, _bs = mira_backtest(_m, _N, _K, _lo, _hi, MIRADORE_MIN_HIST)
        _meth, _valid, _why = mira_choose(_bs, len(_bd), MIRADORE_MIN_BACKTEST)
        print(f'  backtest months={len(_bd)}; MAE ($K): ' + ', '.join(f'{k}={v / 1e3:,.1f}' for k, v in _bs.items()))
        print(f'  CHOSEN METHOD: {_meth} (validated={_valid}) -- {_why}')
        _att, _n_att = mira_attainment(_m['plan'], _m['actual'], _m['known'], _N, _K, _lo, _hi)
        _tr3 = float(_m.loc[_m['known'], 'actual'].tail(3).mean()) if _m['known'].any() else np.nan
        _fwd_end = pd.to_datetime(ensemble_total['ds']).max().to_period('M')
        _fc_m = _m[(~_m['known']) & (_m['plan'] > 0) & (_m['month'] <= _fwd_end)]
        _fcv = {}
        for _r in _fc_m.itertuples():
            _fcv[_r.month] = _r.plan if _meth == 'raw_plan' else (_r.plan * _att if _meth == 'attainment' else _tr3)
        print(f'  attainment estimate x{_att:.3f} from {_n_att} known months; forecast months: {[str(k) for k in _fcv]}')
        _ext_months = []
        if globals().get('MIRADORE_EXTEND_ENABLED', True) and np.isfinite(_tr3):
            _ext_end = pd.Period(str(globals().get('MIRADORE_EXTEND_THROUGH', '2027-12')), 'M')
            _pp = _m['month'].max() + 1
            while _pp <= _ext_end:
                _fcv[_pp] = _tr3
                _ext_months.append(_pp)
                _pp += 1
            if _ext_months:
                print(f"  EXTENSION: no FY plan after {_m['month'].max()} -> flat trailing-3-month mean "
                      f"${_tr3 / 1e3:,.0f}K/month for {_ext_months[0]}..{_ext_months[-1]} (labelled 'extended')")
        _bmap = dict(zip(_bd['month'], _bd[_meth])) if len(_bd) else {}
        _out = pd.DataFrame({'run_date': pd.Timestamp.today().date(), 'month': _m['month'].dt.to_timestamp().dt.date,
                             'plan': _m['plan'], 'actual': _m['actual'], 'attainment': _m['attain'],
                             'outlook': [_fcv.get(x, _bmap.get(x, np.nan)) for x in _m['month']],
                             'method': _meth, 'validated': bool(_valid)})
        _out['notes'] = np.where(_m['known'], 'actual known; outlook = out-of-sample backtest value (NaN if too little history)',
                                 np.where(_m['month'].isin(list(_fcv.keys())),
                                          'actual UNKNOWN or incomplete (excluded from attainment); outlook = forecast', 'no outlook'))
        _out['extended'] = False
        if _ext_months:
            _ext_df = pd.DataFrame({'run_date': pd.Timestamp.today().date(),
                                    'month': [_p.to_timestamp().date() for _p in _ext_months],
                                    'plan': np.nan, 'actual': np.nan, 'attainment': np.nan, 'outlook': float(_tr3),
                                    'method': 'trail3 (extension: no FY plan)', 'validated': bool(_valid and _meth == 'trail3'),
                                    'notes': 'EXTENSION: no plan for this month; flat trailing-3-month mean of known actuals (estimate)',
                                    'extended': True})
            _out = pd.concat([_out, _ext_df], ignore_index=True)
        ig_write_table(_out, MIRADORE_OUTLOOK_TABLE)
        _sf = ensemble_total[['ds', 'p50']].copy(); _sf['ds'] = pd.to_datetime(_sf['ds'])
        _mw = mira_spread_weeks(_fcv, _sf['ds'])
        _tot = pd.DataFrame({'run_date': pd.Timestamp.today().date(), 'ds': _sf['ds'].dt.date,
                             'salesforce_itsg_p50': _sf['p50'].values, 'miradore_outlook': _mw,
                             'miradore_available': np.isfinite(_mw)})
        _tot['total'] = _tot['salesforce_itsg_p50'] + np.nan_to_num(_tot['miradore_outlook'])
        _tot['view_label'] = ('SEPARATE finance view: Salesforce ITSG P50 + Miradore outlook. Miradore is NOT part of the '
                              f'Salesforce ITSG forecast; method={_meth}, validated={_valid}')
        print(_tot.drop(columns=['view_label', 'run_date']).head(8).round(0).to_string(index=False))
        ig_write_table(_tot, MIRADORE_TOTAL_TABLE)
        MIRADORE_RESULT = {'method': _meth, 'validated': _valid, 'mae': _bs, 'n_backtest': len(_bd),
                           'unknown_months': _unk, 'att': _att}
        IG_RESULTS['miradore'] = MIRADORE_RESULT
    except AssertionError as _exc_ma:
        print(f'13M Miradore outlook SKIPPED (assumption mismatch, soft): {str(_exc_ma)[:600]}')
    except Exception as _exc_m:
        print(f'13M Miradore outlook skipped (fail-soft): {type(_exc_m).__name__}: {str(_exc_m)[:300]}')
else:
    print('13M: MIRADORE_OUTLOOK_ENABLED=False -> skipped.')

In [ ]:
# STEP 7A: HINDCAST — RETROACTIVE Q1/Q2 2026 MODEL PREDICTIONS
# =============================================================================
# PURPOSE: Generate what each V5 model WOULD have predicted for Q1 and Q2 2026
# had the full ITSG model pipeline been running at the start of those quarters.
#
# WHY THIS IS NEEDED:
#   The V5 pipeline launched mid-2026.  The PowerBI wide export backfills
#   closed-quarter model columns with Prophet-only values (model_fill_source =
#   'prophet_proxy'), making all models appear identical for Q1/Q2.
#   This cell retroactively trains every model at the BOQ cutoff and produces
#   differentiated forecasts that can replace the proxy values.
#
# APPROACH:
#   Two "hindcast origins" simulate beginning-of-quarter predictions:
#     Origin 1: 2025-12-28 -> predict Q1 2026 (Jan 5 - Mar 29, 13 weeks)
#     Origin 2: 2026-03-29 -> predict Q2 2026 (Apr 5 - Jun 28, 13 weeks)
#
#   At each origin every retained model is retrained on data available up to
#   that date, then predicts 13 weeks forward.  Results are compared to actual
#   outcomes and APPENDED to itsg_backtest_results.
#
# MODELS:
#   Prophet_trend, MSTL_v2, ETS   -- via benchmark_model_specs (tuned params)
#   DHR_ARIMA                      -- total-level with external regressors
#   Global_LGB_Q50_ITSG            -- global panel with market-level features
#   Adaptive_Ensemble              -- weighted combination via ensemble_weights
#
# OUTPUT TABLE (APPEND):
#   datagroup_mdl.mdl_sales_analytics.itsg_backtest_results
#
# RUNTIME: ~15-20 min on serverless (5 models x 2 origins + LGB feature build)
# =============================================================================

import numpy as np
import pandas as pd

# BOQ origins derived programmatically from CURRENT_FY (Fix, 2026-09-28 cleanup
# pass): the last closed week of the quarter immediately BEFORE each target
# quarter, read directly off weekly_itsg's actual Monday-bucketed spine. This
# replaces the previous hardcoded '2025-12-28' / '2026-03-29' literals, which
# would silently mislabel once the notebook runs past the current fiscal year.
_q1_first_ds = weekly_itsg.loc[
    weekly_itsg['ds'].dt.to_period('Q').astype(str) == CURRENT_FY_Q1_LABEL, 'ds'
].min()
_q2_first_ds = weekly_itsg.loc[
    weekly_itsg['ds'].dt.to_period('Q').astype(str) == CURRENT_FY_Q2_LABEL, 'ds'
].min()
assert pd.notna(_q1_first_ds) and pd.notna(_q2_first_ds), (
    f'Could not locate {CURRENT_FY_Q1_LABEL}/{CURRENT_FY_Q2_LABEL} in weekly_itsg -- '
    f'check CURRENT_FY / data coverage before running the hindcast.'
)
HINDCAST_ORIGINS = [
    _q1_first_ds - pd.Timedelta(weeks=1),  # BOQ for CURRENT_FY_Q1_LABEL
    _q2_first_ds - pd.Timedelta(weeks=1),  # BOQ for CURRENT_FY_Q2_LABEL
]
HINDCAST_HORIZON = 13
HINDCAST_TABLE = 'datagroup_mdl.mdl_sales_analytics.itsg_backtest_results'
HINDCAST_VERSION = NOTEBOOK_VERSION + '_hindcast'
_LGB_MODEL = globals().get('GLOBAL_MODEL_NAME', 'Global_LGB_Q50_ITSG')

hindcast_rows = []

for origin in HINDCAST_ORIGINS:
    origin_label = origin.date()
    forecast_weeks = pd.date_range(
        origin + pd.Timedelta(weeks=1),
        periods=HINDCAST_HORIZON,
        freq='W-MON',
    )

    # Actuals from model_df (total ITSG weekly ARR)
    actuals_map = dict(zip(model_df['ds'], model_df['y']))
    actual_arr = np.array([actuals_map.get(w, np.nan) for w in forecast_weeks])
    q_actual = float(np.nansum(actual_arr))

    print(f"\n{'=' * 72}")
    print(
        f"HINDCAST ORIGIN: {origin_label}  ->  "
        f"{forecast_weeks[0].date()} to {forecast_weeks[-1].date()}"
    )
    print(f"Actual quarter total: ${q_actual / 1e6:.2f}M")
    print('=' * 72)

    all_preds: dict[str, np.ndarray] = {}

    # ---- 1. Benchmark models: Prophet, MSTL, ETS (tuned params) --------------
    train_simple = model_df[model_df['ds'] <= origin][['ds', 'y']].copy()
    print(f"Training rows (simple models): {len(train_simple)}")

    for name, fn, kwargs in benchmark_model_specs:
        pred = np.asarray(
            fn(train_simple, horizon=HINDCAST_HORIZON, **kwargs), dtype=float
        )
        all_preds[name] = pred

    # ---- 2. DHR-ARIMA (total-level with external regressors) -----------------
    try:
        dhr_train = model_df[model_df['ds'] <= origin][['ds', 'y']].copy()
        _dhr_params = globals().get('BEST_DHR_PARAMS', {})
        dhr_pred = np.asarray(
            fit_predict_dhr_arima(
                dhr_train,
                horizon=HINDCAST_HORIZON,
                full_feature_df=dhr_df,
                **_dhr_params,
            ),
            dtype=float,
        )
        all_preds['DHR_ARIMA'] = dhr_pred
    except Exception as exc:
        print(f"  WARNING: DHR-ARIMA hindcast failed ({exc}); skipping")
        all_preds['DHR_ARIMA'] = np.full(HINDCAST_HORIZON, np.nan)

    # ---- 3. Global LGB (market-level panel, recursive prediction) ------------
    try:
        lgb_train_panel = global_panel[global_panel['ds'] <= origin].copy()
        if len(lgb_train_panel) >= 40:
            lgb_models, _ = fit_global_quantile_models(
                lgb_train_panel, params=BEST_LGB_PARAMS
            )
            # Build prediction panel: history + future from existing panel data
            lgb_pred_end = forecast_weeks.max()
            lgb_pred_panel = global_panel[
                global_panel['ds'] <= lgb_pred_end
            ].copy()
            # Mask future y so recursive prediction fills it in
            lgb_pred_panel.loc[
                lgb_pred_panel['ds'] > origin, 'y'
            ] = np.nan
            lgb_scored = recursive_lgb_predict(
                lgb_models,
                lgb_pred_panel,
                origin + pd.Timedelta(weeks=1),
                lgb_pred_end,
            )
            lgb_total = (
                lgb_scored.groupby('ds', as_index=False)
                .agg(forecast=('p50', 'sum'))
            )
            lgb_arr = (
                lgb_total.set_index('ds')
                .reindex(forecast_weeks)['forecast']
                .fillna(0)
                .values.astype(float)
            )
        else:
            print(f"  WARNING: LGB training rows ({len(lgb_train_panel)}) below minimum; skipping")
            lgb_arr = np.full(HINDCAST_HORIZON, np.nan)
        all_preds[_LGB_MODEL] = lgb_arr
    except Exception as exc:
        print(f"  WARNING: LGB hindcast failed ({exc}); skipping")
        all_preds[_LGB_MODEL] = np.full(HINDCAST_HORIZON, np.nan)

    # ---- 4. Adaptive Ensemble (weighted average of components) ----------------
    ew = ensemble_weights.set_index('model')['ensemble_weight']
    ensemble_pred = np.zeros(HINDCAST_HORIZON)
    total_weight = 0.0
    for m in ew.index:
        if m in all_preds and np.isfinite(all_preds[m]).all():
            w = float(ew[m])
            ensemble_pred += w * all_preds[m]
            total_weight += w
    if total_weight > 0:
        ensemble_pred /= total_weight
    else:
        ensemble_pred = np.full(HINDCAST_HORIZON, np.nan)
    all_preds['Adaptive_Ensemble'] = ensemble_pred

    # ---- Collect rows and print summary --------------------------------------
    for model_name, pred in all_preds.items():
        q_forecast = float(np.nansum(pred))
        pct_err = (
            100 * (q_forecast - q_actual) / q_actual
            if q_actual > 0
            else np.nan
        )
        print(f"  {model_name:<25} Q forecast: ${q_forecast / 1e6:.2f}M  (error: {pct_err:+.1f}%)")
        for i, (ds_val, fcast) in enumerate(zip(forecast_weeks, pred)):
            act = actual_arr[i]
            hindcast_rows.append({
                'model': model_name,
                'origin_date': origin_label,
                'ds': ds_val,
                'horizon_week': i + 1,
                'forecast': float(fcast),
                'actual': float(act) if np.isfinite(act) else None,
                'pipeline_version': HINDCAST_VERSION,
                'error': (
                    float(fcast - act) if np.isfinite(act) else None
                ),
                'rel_error': (
                    float((fcast - act) / act)
                    if np.isfinite(act) and abs(act) > 1e-9
                    else None
                ),
            })

# ---- Persist to Delta (idempotent: DELETE-then-APPEND scoped to this run's origins) ---------
# Fix (2026-09-28 cleanup pass): this cell always recomputes the SAME two BOQ
# origins for the current fiscal year's Q1/Q2 (HINDCAST_ORIGINS is derived
# from CURRENT_FY_Q1_LABEL/CURRENT_FY_Q2_LABEL, not from "today"), so every
# weekly rerun of this notebook was re-appending an identical batch of rows
# with a plain .mode('append') and no dedup -- duplicates accumulated forever.
# We now delete any existing rows for exactly these origin_date(s) immediately
# before appending the freshly computed batch. This keeps prior weeks' rows
# for OTHER origins (e.g. a future fiscal year's Q1/Q2) untouched, makes a
# same-day/same-week rerun produce zero duplicates, and is guarded with a
# table-existence check so the very first run ever (table not created yet)
# just skips the DELETE.
hindcast_df = pd.DataFrame(hindcast_rows)
hindcast_df['ds'] = pd.to_datetime(hindcast_df['ds'])

_hindcast_origin_labels = sorted({str(o.date()) for o in HINDCAST_ORIGINS})
if spark.catalog.tableExists(HINDCAST_TABLE):
    _origin_list_sql = ', '.join(f"'{d}'" for d in _hindcast_origin_labels)
    spark.sql(
        f"DELETE FROM {HINDCAST_TABLE} WHERE origin_date IN ({_origin_list_sql})"
    )
    print(f"Deleted existing rows for origin(s) {_hindcast_origin_labels} before re-appending (idempotent rerun).")
else:
    print(f"{HINDCAST_TABLE} does not exist yet; skipping pre-delete on first-ever run.")

hindcast_sdf = spark.createDataFrame(hindcast_df)
hindcast_sdf.write.mode('append').saveAsTable(HINDCAST_TABLE)

print(f"\n{'=' * 72}")
print(f"HINDCAST RESULTS APPENDED TO {HINDCAST_TABLE}")
print(f"{'=' * 72}")
print(f"Rows appended: {len(hindcast_df)}")
print(f"Models: {sorted(hindcast_df['model'].unique().tolist())}")
print(f"Origins: {sorted(str(o) for o in hindcast_df['origin_date'].unique())}")

# ---- Quarterly summary comparison --------------------------------------------
hindcast_df['quarter'] = hindcast_df['ds'].dt.to_period('Q').astype(str)
quarterly_hindcast = (
    hindcast_df.groupby(['quarter', 'model'], as_index=False)
    .agg(
        forecast_M=('forecast', lambda s: round(s.sum() / 1e6, 2)),
        actual_M=('actual', lambda s: round(s.sum() / 1e6, 2)),
    )
)
quarterly_hindcast['error_pct'] = round(
    100
    * (quarterly_hindcast['forecast_M'] - quarterly_hindcast['actual_M'])
    / quarterly_hindcast['actual_M'],
    1,
)
print("\nQuarterly hindcast vs actuals:")
display(
    quarterly_hindcast.sort_values(['quarter', 'error_pct'], key=lambda c: c.abs() if c.name == 'error_pct' else c)
    .reset_index(drop=True)
)

### Step 7B — Monte Carlo simulation of quarterly outcomes **(NEW)**

With average weekly ARR around $0.35M and best-model weekly WAPE north of 30%, ITSG's quarterly *point*
forecast is far less informative than its distribution — this step is arguably more decision-relevant here
than for larger product lines. Method: moving-block bootstrap (4-week blocks) of the **Adaptive
Ensemble's holdout relative errors** — strictly out-of-sample — applied to the forward weekly p50 path,
10,000 paths, clipped at zero, aggregated to quarterly and FY totals. Outputs: percentile tables,
P(total ≥ threshold) at $0.5M steps for planning, and a comparison against the ratio-based intervals from
Step 7. Caveat: ~27 holdout residuals is a modest bootstrap base — trust p05-p95, be humble beyond it.

In [ ]:
# STEP 7B: MONTE CARLO SIMULATION OF QUARTERLY OUTCOMES (NEW)
# =============================================================================
# PURPOSE: Generate statistically correct quarterly and current-FY distributions
# by simulating 10,000 paths of weekly forecast errors.  For a noisy series
# like ITSG (~30% weekly WAPE), the quarterly DISTRIBUTION is arguably more
# decision-relevant than the point forecast.
#
# Method:
#   1. Extract ensemble holdout relative errors (actual/p50 - 1)
#   2. Moving-block bootstrap (block=4 weeks, preserves autocorrelation)
#   3. Apply simulated errors to the forward p50 path -> 10K weekly paths
#   4. Aggregate to quarterly and current-FY (CURRENT_FY) totals per path
#   5. Extract percentiles and P(total >= target) for planning
#
# Outputs:
#   mc_quarterly_distribution — P05/P25/P50/P75/P95 for each quarter
#   mc_fy_distribution        — current-FY total distribution (closed + simulated)
#   mc_threshold_table        — P(total >= $XM) for planning thresholds
#
# Moving-block bootstrap of ensemble HOLDOUT relative errors -> quarterly / FY
# distributions and P(total >= target). Residuals are out-of-sample only.
# =============================================================================

N_SIMS = 10_000
BLOCK_SIZE = 4
MC_SEED = 42
rng = np.random.default_rng(MC_SEED)

mc_resid = ensemble_holdout.dropna(subset=['error']).sort_values('ds').copy()
assert len(mc_resid) >= 2 * BLOCK_SIZE, 'Too few holdout residuals for block bootstrap.'
mc_resid['rel_error'] = mc_resid['error'] / mc_resid['forecast'].clip(lower=1.0)
rel_errors = mc_resid['rel_error'].to_numpy(dtype=float)
print(f'Residual pool: {len(rel_errors)} holdout weeks | mean rel error {rel_errors.mean():+.3f} | std {rel_errors.std(ddof=1):.3f}')

# NEW (2026-10-05): QUARTER-END INTERVAL WIDENING -- ported from UCC's Step 14B
# fix (same date). UCC's Q3 2026 close showed the last 1-2 weeks of a quarter
# run materially more volatile than an average week (large deals concentrate
# at quarter-end); this bootstrap draws every week from ONE undifferentiated
# residual pool, so a quarter-end week's P05-P95 band is no wider than any
# other week's, understating risk exactly where the point forecast is least
# reliable. ITSG has no qe_surge_index/is_last_2_weeks_of_qtr feature (unlike
# UCC), so this computes "quarter-end week" directly from the date (last 13
# days of the calendar quarter) rather than from a precomputed column.
# Widens only the SPREAD of simulated quarter-end weeks; the P50 path
# (mc_p50) is untouched. Falls back to no widening if too few historical
# quarter-end holdout weeks exist to estimate a ratio reliably.
def _is_qtr_end_week(ds_series: pd.Series, weeks_from_end: int = 2) -> pd.Series:
    _d = pd.to_datetime(ds_series)
    _qtr_end = _d.dt.to_period('Q').dt.end_time.dt.normalize()
    _days_to_end = (_qtr_end - _d).dt.days
    return (_days_to_end <= 7 * weeks_from_end - 1)

mc_resid['is_qe_week'] = _is_qtr_end_week(mc_resid['ds'])
_qe_err = mc_resid.loc[mc_resid['is_qe_week'], 'rel_error']
_non_qe_err = mc_resid.loc[~mc_resid['is_qe_week'], 'rel_error']
MIN_QE_SAMPLES_FOR_WIDENING = 4
if len(_qe_err) >= MIN_QE_SAMPLES_FOR_WIDENING and _non_qe_err.std() > 1e-9:
    QE_DISPERSION_RATIO = float(np.clip(_qe_err.std() / _non_qe_err.std(), 1.0, 3.0))
    print(f'Quarter-end dispersion ratio: {QE_DISPERSION_RATIO:.2f}x '
          f'(from {len(_qe_err)} historical quarter-end holdout weeks vs '
          f'{len(_non_qe_err)} other holdout weeks). Widening QE-week simulated '
          f'intervals by this factor; point forecast (P50) is unchanged.')
else:
    QE_DISPERSION_RATIO = 1.0
    print(f'Quarter-end dispersion ratio: only {len(_qe_err)} historical quarter-end '
          f'holdout weeks available (< {MIN_QE_SAMPLES_FOR_WIDENING}) -- skipping '
          f'QE-specific interval widening this run (ratio left at 1.0).')

mc_fw = ensemble_total.sort_values('ds').reset_index(drop=True)
mc_h = len(mc_fw)
mc_p50 = mc_fw['p50'].to_numpy(dtype=float)
mc_quarters = pd.to_datetime(mc_fw['ds']).dt.to_period('Q').astype(str).to_numpy()

n_blocks = int(np.ceil(mc_h / BLOCK_SIZE))
starts = rng.integers(0, len(rel_errors) - BLOCK_SIZE + 1, size=(N_SIMS, n_blocks))
idx = starts[:, :, None] + np.arange(BLOCK_SIZE)[None, None, :]
sim_rel = rel_errors[idx].reshape(N_SIMS, -1)[:, :mc_h]

# Apply the quarter-end widening to the forward horizon's own quarter-end weeks.
_fwd_is_qe = _is_qtr_end_week(mc_fw['ds']).to_numpy()
_qe_widen_vec = np.where(_fwd_is_qe, QE_DISPERSION_RATIO, 1.0)
print(f'Forward horizon: {int(_fwd_is_qe.sum())} of {mc_h} weeks flagged as '
      f'quarter-end and widened by {QE_DISPERSION_RATIO:.2f}x.')
sim_rel = sim_rel * _qe_widen_vec[None, :]

sim_weekly = np.clip(mc_p50[None, :] * (1.0 + sim_rel), 0.0, None)

# CHANGED: trimmed from the 7-level superset [5,10,25,50,75,90,95] to the
# notebook-wide 5-column convention (drop 10 and 90, keep the rest).
PCTS = [5, 25, 50, 75, 95]
quarter_sims = {}
mc_rows = []
for q in pd.unique(mc_quarters):
    q_sum = sim_weekly[:, mc_quarters == q].sum(axis=1)
    quarter_sims[q] = q_sum
    row = {'quarter': q, 'weeks_simulated': int((mc_quarters == q).sum()),
           'point_p50_sum_m': round(float(mc_p50[mc_quarters == q].sum()) / 1e6, 3)}
    for p in PCTS:
        row[f'p{p:02d}_m'] = round(float(np.percentile(q_sum, p)) / 1e6, 3)
    mc_rows.append(row)
mc_quarterly_distribution = pd.DataFrame(mc_rows)

# CHANGED: CURRENT_FY (defined in Step 7 from weekly_itsg's actual max closed
# week) replaces the hardcoded 2026 literal so this doesn't need a manual bump.
closed_fy_actual = float(weekly_itsg.loc[weekly_itsg['ds'].dt.year == CURRENT_FY, 'y'].sum())
fy_weeks_mask = pd.to_datetime(mc_fw['ds']).dt.year == CURRENT_FY
fy_sims = closed_fy_actual + sim_weekly[:, fy_weeks_mask.to_numpy()].sum(axis=1)
fy_row = {'scope': f'FY{CURRENT_FY} (closed actuals + simulated remaining FY{CURRENT_FY} weeks)',
          'closed_actual_m': round(closed_fy_actual / 1e6, 3)}
for p in PCTS:
    fy_row[f'p{p:02d}_m'] = round(float(np.percentile(fy_sims, p)) / 1e6, 3)
mc_fy_distribution = pd.DataFrame([fy_row])

threshold_rows = []
for scope, sims in list(quarter_sims.items()) + [(f'FY{CURRENT_FY}', fy_sims)]:
    med = np.median(sims)
    for t in np.arange(max(0.5, np.floor(med / 5e5) * 0.5 - 1.0), np.floor(med / 5e5) * 0.5 + 1.51, 0.5):
        threshold_rows.append({'scope': scope, 'threshold_m': round(t, 2),
                               'prob_at_least_pct': round(100.0 * float(np.mean(sims >= t * 1e6)), 1)})
mc_threshold_table = pd.DataFrame(threshold_rows)

interval_compare = (
    production_quarterly_forecasts[production_quarterly_forecasts['model'] == 'Adaptive_Ensemble']
    [['quarter', 'p05_arr_m', 'forecast_arr_m', 'p95_arr_m']]
    .rename(columns={'p05_arr_m': 'ratio_p05_m', 'forecast_arr_m': 'p50_m', 'p95_arr_m': 'ratio_p95_m'})
    .merge(mc_quarterly_distribution[['quarter', 'p05_m', 'p95_m']].rename(columns={'p05_m': 'mc_p05_m', 'p95_m': 'mc_p95_m'}),
           on='quarter', how='inner')
)

print('\nMonte Carlo quarterly distributions (Adaptive Ensemble forward weeks, $M):')
display(mc_quarterly_distribution)
print(f'\nFY{CURRENT_FY} distribution ($M):')
display(mc_fy_distribution)
print('\nProbability of reaching planning thresholds:')
display(mc_threshold_table)
print('\nRatio-based (Step 7) vs Monte Carlo quarterly intervals (MC aggregates correctly across weeks):')
display(interval_compare)

import matplotlib.pyplot as plt
panels = list(quarter_sims.items()) + [(f'FY{CURRENT_FY}', fy_sims)]
fig, axes = plt.subplots(1, len(panels), figsize=(5.5 * len(panels), 4.2))
axes = np.atleast_1d(axes)
for ax, (label, sims) in zip(axes, panels):
    ax.hist(np.asarray(sims) / 1e6, bins=60, color='steelblue', alpha=0.8, edgecolor='white')
    for p, ls in [(5, '--'), (50, '-'), (95, '--')]:
        ax.axvline(np.percentile(sims, p) / 1e6, color='crimson', ls=ls, lw=1.3,
                   label=f'p{p}: ${np.percentile(sims, p)/1e6:,.2f}M')
    ax.set_title(f'{label} ({N_SIMS:,} paths)')
    ax.set_xlabel('Growth ARR ($M)')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()
print(f'Method: moving-block bootstrap (block={BLOCK_SIZE}) of holdout relative residuals; seed={MC_SEED}; residuals out-of-sample only.')

In [ ]:
# STEP 7B-QE: PER-QUARTER SEASONAL SURGE DIAGNOSTIC (NEW, 2026-10-05, ported from UCC)
# =============================================================================
# PURPOSE: Step 7B's QE_DISPERSION_RATIO (above) checks quarter-end weeks as
# ONE pooled group across all four quarters and came back at exactly 1.00x --
# i.e. no quarter-end volatility premium for ITSG, unlike UCC. That is an
# AGGREGATE check, though, and could in principle mask a quarter-specific
# effect that cancels out when pooled (e.g. Q4 running hot while Q1 runs
# cool). This cell checks each calendar quarter separately against ITSG's
# full history to confirm the "no surge" finding actually holds quarter by
# quarter, not just on average.
#
# PURELY DIAGNOSTIC -- does not change QE_DISPERSION_RATIO, the Monte Carlo
# widening, or any forecast/calibration. ITSG intentionally has no
# QE_CALIBRATION-style point-forecast correction (UCC-only), so this cell
# only reports; there is nothing here for it to feed into even if a
# quarter-specific effect did show up. If one does show up, it's a trigger
# to look closer before assuming ITSG truly has no quarter-end seasonality.
# =============================================================================

try:
    def _is_qtr_end_week_diag(ds_series: pd.Series, weeks_from_end: int = 2) -> pd.Series:
        d = pd.to_datetime(ds_series)
        qtr_end = d.dt.to_period('Q').dt.end_time.dt.normalize()
        days_to_end = (qtr_end - d).dt.days
        return days_to_end <= (7 * weeks_from_end - 1)

    _seas = weekly_itsg[['ds', 'y']].copy()
    _seas['ds'] = pd.to_datetime(_seas['ds'])
    _seas['quarter'] = _seas['ds'].dt.quarter
    _seas['year'] = _seas['ds'].dt.year
    _seas['is_qe_week'] = _is_qtr_end_week_diag(_seas['ds'])

    _rows = []
    for _q, _g in _seas.groupby('quarter'):
        _qe_wks = _g[_g['is_qe_week']]
        _other_wks = _g[~_g['is_qe_week']]
        _qe_avg = float(_qe_wks['y'].mean()) if len(_qe_wks) else np.nan
        _other_avg = float(_other_wks['y'].mean()) if len(_other_wks) else np.nan
        _ratio = (_qe_avg / _other_avg) if (_other_avg and _other_avg > 0) else np.nan
        _rows.append({
            'quarter': f'Q{int(_q)}',
            'n_years_observed': int(_g['year'].nunique()),
            'n_qe_weeks': int(len(_qe_wks)),
            'n_other_weeks': int(len(_other_wks)),
            'qe_avg_weekly_arr': round(_qe_avg, 0) if pd.notna(_qe_avg) else None,
            'other_avg_weekly_arr': round(_other_avg, 0) if pd.notna(_other_avg) else None,
            'surge_ratio': round(_ratio, 3) if pd.notna(_ratio) else None,
        })

    itsg_seasonal_qe_surge_diagnostic = pd.DataFrame(_rows).sort_values('quarter').reset_index(drop=True)
    print('=' * 72)
    print('ITSG PER-QUARTER SEASONAL SURGE DIAGNOSTIC (reporting only -- no calibration change)')
    print('  surge_ratio = avg weekly ARR in last-2-weeks-of-quarter / avg weekly ARR other weeks')
    print('  Step 7B\'s pooled QE_DISPERSION_RATIO came back 1.00x -- this checks that holds per-quarter.')
    print('=' * 72)
    display(itsg_seasonal_qe_surge_diagnostic)

    _ratios = itsg_seasonal_qe_surge_diagnostic['surge_ratio'].dropna()
    if len(_ratios):
        _rng = float(_ratios.max() - _ratios.min())
        if _rng > 0.30:
            print(f"\n  NOTE: surge ratios vary meaningfully across quarters (range {_rng:.2f}) "
                  f"even though the pooled ratio is ~1.00x -- a quarter-specific effect may be "
                  f"canceling out in the aggregate. Worth a closer look before concluding ITSG "
                  f"has no quarter-end seasonality at all.")
        else:
            print(f"\n  All four quarters are consistent with each other (range {_rng:.2f}) and "
                  f"with the pooled 1.00x finding -- ITSG genuinely shows no quarter-end surge, "
                  f"confirmed at the per-quarter level too.")
except Exception as e:
    itsg_seasonal_qe_surge_diagnostic = None
    print(f'ITSG per-quarter seasonal surge diagnostic skipped (non-fatal): {e}')


### Step 8 — Holdout scoreboard, generated executive summary, and monitoring baseline **(CHANGED)**

Everything here is computed from live DataFrames. The closed-quarter scoreboard covers **all** holdout
quarters for all five models plus the ensemble, with quarters defined by `to_period('Q')` (V4 had three
conflicting hand-set Q1 definitions and a duplicated DHR row). The executive-summary markdown is generated
and printed — nothing hand-typed. The monitoring baseline now includes the Adaptive Ensemble row, records
the weighting method, split boundaries, and the LGB gate decision, and flags the ensemble as the
recommended production model (V4 hardcoded DHR as recommended while its own tables showed DHR mid-pack).

In [ ]:
# STEP 8: HOLDOUT SCOREBOARD, GENERATED EXECUTIVE SUMMARY, MONITORING BASELINE
# =============================================================================
# PURPOSE: Produce the definitive model ranking, executive summary, and
# monitoring baseline — all from live DataFrames (no hand-typed numbers).
#
# Outputs:
#   * Holdout leaderboard (WAPE, MAPE, Bias, Directional Accuracy)
#   * Closed-quarter forecast audit (Q1/Q2 2026 actual vs each model)
#   * Generated executive summary with all accuracy numbers
#   * Monitoring baseline written to Delta (drift thresholds = WAPE x 1.5)
#
# Decision rules:
#   * Primary sort: holdout WAPE (lower = better)
#   * Flag any model with |Bias| > 10% for ITSG (noisier threshold than UCC)
#   * Monitoring alert: if future run's rolling WAPE > baseline x 1.5
#
# CHANGED vs V4: single quarter definition (to_period), ensemble included,
# no duplicated rows, summary generated from live frames, monitoring baseline
# records methodology and includes the ensemble.
# =============================================================================

# --- Ensemble holdout metrics -----------------------------------------------------
ens_hm = forecast_metrics(ensemble_holdout['actual'].values, ensemble_holdout['forecast'].values)
holdout_leaderboard = pd.concat([
    holdout_benchmark_summary,
    pd.DataFrame([{'model': 'Adaptive_Ensemble', 'test_start': str(ensemble_holdout['ds'].min().date()),
                   'test_end': str(ensemble_holdout['ds'].max().date()), 'test_rows': len(ensemble_holdout),
                   'wape': round(ens_hm['wape'], 3), 'mape': round(ens_hm['mape'], 3), 'mpe': round(ens_hm['mpe'], 3),
                   'smape': round(ens_hm['smape'], 3), 'rmse': round(ens_hm['rmse'], 3)}]),
], ignore_index=True).sort_values('wape').reset_index(drop=True)

# --- Closed-quarter scoreboard (all models + ensemble, consistent quarters) --------
scoreboard_src = pd.concat([
    holdout_benchmark_predictions[['model', 'ds', 'actual', 'forecast']],
    ensemble_holdout.assign(model='Adaptive_Ensemble')[['model', 'ds', 'actual', 'forecast']],
], ignore_index=True)
scoreboard_src['quarter'] = pd.to_datetime(scoreboard_src['ds']).dt.to_period('Q').astype(str)
holdout_quarter_scoreboard = (
    scoreboard_src.groupby(['model', 'quarter'], as_index=False)
    .agg(weeks=('ds', 'nunique'), actual_arr=('actual', 'sum'), forecast_arr=('forecast', 'sum'))
)
holdout_quarter_scoreboard['actual_arr_m'] = (holdout_quarter_scoreboard['actual_arr'] / 1e6).round(3)
holdout_quarter_scoreboard['forecast_arr_m'] = (holdout_quarter_scoreboard['forecast_arr'] / 1e6).round(3)
holdout_quarter_scoreboard['pct_error'] = np.where(
    holdout_quarter_scoreboard['actual_arr'].abs() > 1e-9,
    (100.0 * (holdout_quarter_scoreboard['forecast_arr'] - holdout_quarter_scoreboard['actual_arr']) / holdout_quarter_scoreboard['actual_arr']).round(2),
    np.nan,
)
holdout_quarter_scoreboard = holdout_quarter_scoreboard.sort_values(['quarter', 'model']).reset_index(drop=True)

print(f'Holdout leaderboard (weekly WAPE, {CURRENT_FY_Q1_LABEL} onward, incl. ensemble):')
display(holdout_leaderboard)
print('\nClosed-quarter scoreboard (holdout quarters):')
display(holdout_quarter_scoreboard)

# --- Generated executive summary ----------------------------------------------------
def _fmt(df):
    try:
        return df.to_markdown(index=False)
    except Exception:
        return df.to_string(index=False)

_lgb_cov = global_holdout_summary['ci_coverage'].iloc[0]
_ens_q = holdout_quarter_scoreboard[holdout_quarter_scoreboard['model'] == 'Adaptive_Ensemble']
_w_view = ensemble_weights[['model', 'mean_wape', 'ensemble_weight', 'weighting_method']].copy()
_w_view['ensemble_weight'] = (_w_view['ensemble_weight'] * 100).round(1)

lines = []
lines.append(f'## ITSG Growth ARR Forecast — Executive Summary ({NOTEBOOK_VERSION}, generated {pd.Timestamp.today().date()})')
lines.append('')
lines.append('*All numbers generated from this run — nothing hand-typed.*')
lines.append('')
lines.append(f'- Data through **{OBSERVED_END.date()}** ({len(weekly_itsg)} weekly obs; total ${weekly_itsg["y"].sum()/1e6:.1f}M; avg ${weekly_itsg["y"].mean()/1e3:.0f}K/week)')
lines.append(f'- Splits: CV -> {cv_region_df["ds"].max().date()} | validation (tuning only) {VALIDATION_START.date()} -> {validation_df["ds"].max().date()} | holdout from {BACKTEST_START.date()}')
lines.append(f'- Walk-forward CV: **{CV_FOLDS} folds x {CV_HORIZON} weeks**, all strictly pre-holdout')
lines.append(f'- Ensemble weighting: **{ensemble_weights["weighting_method"].iloc[0]}** | live components: **{", ".join(ENSEMBLE_COMPONENT_MODELS)}**')
lines.append(f'- LGB gate: {"included" if LGB_IN_ENSEMBLE else "excluded"} (validation WAPE {LGB_VALIDATION_WAPE:.1f} vs threshold {LGB_GATE_THRESHOLD:.1f})')
lines.append(f'- Ensemble membership selection: **{ENSEMBLE_SELECTION_METHOD}**')
lines.append(f'- Global LGB holdout 90% CI coverage (weekly total, p05-p95): **{_lgb_cov if pd.notna(_lgb_cov) else "n/a"}%**')
lines.append('')
lines.append('### CV leaderboard (clean folds)')
lines.append(_fmt(cv_benchmark_summary[['model', 'folds', 'mean_wape', 'std_wape', 'mean_mape', 'mean_mpe']]))
lines.append('')
lines.append('### Holdout leaderboard (incl. ensemble)')
lines.append(_fmt(holdout_leaderboard[['model', 'test_start', 'test_end', 'wape', 'mape', 'mpe']]))
lines.append('')
lines.append('### Ensemble weights')
lines.append(_fmt(_w_view))
lines.append('')
lines.append('### Adaptive Ensemble — closed-quarter accuracy (holdout)')
lines.append(_fmt(_ens_q[['quarter', 'weeks', 'actual_arr_m', 'forecast_arr_m', 'pct_error']]))
lines.append('')
lines.append('### Forward quarters (production, $M)')
lines.append(_fmt(production_quarterly_forecasts[production_quarterly_forecasts['model'].isin(['Adaptive_Ensemble'] + PRODUCTION_BENCHMARK_MODELS)]))
lines.append('')
lines.append('### Monte Carlo quarterly distributions ($M)')
lines.append(_fmt(mc_quarterly_distribution[['quarter', 'p05_m', 'p25_m', 'p50_m', 'p75_m', 'p95_m']]))
lines.append('')
lines.append('**Recommended production forecast: Adaptive_Ensemble.** ITSG is a small, volatile series; '
             'quarterly Monte Carlo ranges (Step 7B) should accompany any point number shared with leadership.')
generated_exec_summary_md = '\n'.join(lines)
print('\n' + '=' * 90)
print('GENERATED EXECUTIVE SUMMARY (copy from below — do not hand-edit numbers):')
print('=' * 90 + '\n')
print(generated_exec_summary_md)
try:
    displayHTML('<pre style="white-space: pre-wrap; font-size: 12px;">' + generated_exec_summary_md.replace('<', '&lt;') + '</pre>')
except NameError:
    pass

# --- Monitoring baseline --------------------------------------------------------------
monitoring_baseline = holdout_leaderboard[['model', 'test_start', 'test_end', 'test_rows', 'wape', 'mape', 'mpe', 'smape', 'rmse']].copy()
cv_wape_map = cv_benchmark_summary.set_index('model')['mean_wape'].to_dict()
monitoring_baseline['cv_wape_baseline'] = monitoring_baseline['model'].map(cv_wape_map)
monitoring_baseline = monitoring_baseline.rename(columns={'wape': 'holdout_wape_baseline'})
monitoring_baseline['rolling_4w_wape_alert_threshold'] = (
    monitoring_baseline['cv_wape_baseline'].fillna(monitoring_baseline['holdout_wape_baseline']) * 1.5
).round(3)
monitoring_baseline['recommended_for_production'] = (monitoring_baseline['model'] == 'Adaptive_Ensemble').astype(int)
monitoring_baseline['lgb_in_ensemble'] = bool(LGB_IN_ENSEMBLE)
monitoring_baseline['ensemble_weighting_method'] = ensemble_weights['weighting_method'].iloc[0]
monitoring_baseline['cv_folds'] = CV_FOLDS
monitoring_baseline['cv_region_end'] = str(cv_region_df['ds'].max().date())
monitoring_baseline['validation_start'] = str(VALIDATION_START.date())
monitoring_baseline['holdout_start'] = str(BACKTEST_START.date())
monitoring_baseline['product_group'] = PRODUCT_SCOPE
monitoring_baseline['run_timestamp_utc'] = RUN_TIMESTAMP_UTC
monitoring_baseline['run_date_utc'] = RUN_DATE_UTC
monitoring_baseline['primary_output_table'] = PRIMARY_PRODUCTION_OUTPUT_TABLE
monitoring_baseline['legacy_output_table'] = LEGACY_PRODUCTION_OUTPUT_TABLE
monitoring_baseline['notebook_version'] = NOTEBOOK_VERSION

# -- Per-quarter holdout error (closed-quarter accuracy for executive display) --
# The publish pipeline reads these columns to display quarterly error in the app.
# A quarter is "closed" if the holdout has all 13 weeks for that quarter.
try:
    _hbp = pd.concat([
        holdout_benchmark_predictions[['model', 'ds', 'actual', 'forecast']],
        ensemble_holdout.assign(model='Adaptive_Ensemble')[['model', 'ds', 'actual', 'forecast']],
    ], ignore_index=True)
    _hbp['ds'] = pd.to_datetime(_hbp['ds'])
    _hbp['quarter'] = _hbp['ds'].dt.to_period('Q').astype(str)
    _q_agg = (
        _hbp.groupby(['model', 'quarter'], as_index=False)
        .agg(weeks=('ds', 'nunique'), forecast_sum=('forecast', 'sum'), actual_sum=('actual', 'sum'))
    )
    _q_agg['qtr_error_pct'] = ((_q_agg['forecast_sum'] / _q_agg['actual_sum'] - 1) * 100).abs().round(3)
    _q_agg['is_closed'] = _q_agg['weeks'] >= 12  # 12+ of 13 = closed
    # Last closed quarter per model (prefer closed, then latest)
    _q_agg = _q_agg.sort_values(['is_closed', 'quarter'], ascending=[False, False])
    _last_closed = _q_agg.groupby('model', as_index=False).first()
    _last_closed = _last_closed[['model', 'quarter', 'qtr_error_pct', 'is_closed', 'weeks']]
    _last_closed = _last_closed.rename(columns={
        'quarter': 'last_closed_quarter',
        'qtr_error_pct': 'last_closed_quarter_error_pct',
        'is_closed': 'last_closed_quarter_is_closed',
        'weeks': 'last_closed_quarter_weeks',
    })
    monitoring_baseline = monitoring_baseline.merge(_last_closed, on='model', how='left')
    print(f'  Added closed-quarter error:')
    print(_last_closed[['model', 'last_closed_quarter', 'last_closed_quarter_error_pct']].to_string(index=False))
except Exception as exc:
    print(f'  WARN: Could not compute closed-quarter error: {exc}')
    for col in ['last_closed_quarter', 'last_closed_quarter_error_pct', 'last_closed_quarter_is_closed', 'last_closed_quarter_weeks']:
        monitoring_baseline[col] = None

spark.createDataFrame(monitoring_baseline).write \
    .mode('append') \
    .option('mergeSchema', 'true') \
    .saveAsTable(MONITORING_BASELINE_TABLE)

print('\nMonitoring baseline written:')
display(monitoring_baseline)
print(f'Persisted monitoring baseline table: {MONITORING_BASELINE_TABLE}')

In [ ]:
# STEP 8-QE: QUARTER-END HOLDOUT BIAS DIAGNOSTIC (NEW, 2026-10-05)
# =============================================================================
# PURPOSE: Step 7B-QE just showed ITSG's raw weekly ARR DOES surge at quarter-
# end (1.4x-2.3x depending on quarter) -- contrary to how the Step 7B
# QE_DISPERSION_RATIO=1.00x finding had been read ("ITSG has no quarter-end
# effect"). That 1.00x only means the forecast ERROR SPREAD isn't wider at
# quarter-end -- it says nothing about whether the point forecast is
# systematically biased (too high or too low) at quarter-end specifically.
# This is exactly the gap that, for UCC, hid a real problem (Global_LGB_Q50_UCC
# under-forecasting QE weeks by -35.6% while its overall dispersion looked
# unremarkable). This cell checks the same thing for ITSG directly: per-model
# holdout bias (MPE) split quarter-end vs. other weeks.
#
# PURELY DIAGNOSTIC -- does not change any forecast, weight, or calibration.
# =============================================================================

try:
    def _is_qtr_end_week_bias(ds_series: pd.Series, weeks_from_end: int = 2) -> pd.Series:
        d = pd.to_datetime(ds_series)
        qtr_end = d.dt.to_period('Q').dt.end_time.dt.normalize()
        days_to_end = (qtr_end - d).dt.days
        return days_to_end <= (7 * weeks_from_end - 1)

    _bias_src = scoreboard_src.copy()
    _bias_src['ds'] = pd.to_datetime(_bias_src['ds'])
    _bias_src['is_qe'] = _is_qtr_end_week_bias(_bias_src['ds'])

    _rows = []
    for _mdl, _mg in _bias_src.groupby('model'):
        _row = {'model': _mdl}
        for _label, _mask in [('qtr_end_WAPE_pct', _mg['is_qe']), ('other_WAPE_pct', ~_mg['is_qe'])]:
            _sub = _mg[_mask]
            _row[_label] = (
                round(float((_sub['forecast'] - _sub['actual']).abs().sum() / _sub['actual'].sum() * 100), 2)
                if len(_sub) > 0 and _sub['actual'].sum() > 0 else None
            )
        for _label, _mask in [('qtr_end_bias_pct', _mg['is_qe']), ('other_bias_pct', ~_mg['is_qe'])]:
            _sub = _mg[_mask]
            _row[_label] = (
                round(float((_sub['forecast'] - _sub['actual']).sum() / _sub['actual'].sum() * 100), 2)
                if len(_sub) > 0 and _sub['actual'].sum() > 0 else None
            )
        _row['n_qtr_end_obs'] = int(_mg['is_qe'].sum())
        _rows.append(_row)

    itsg_qe_bias_diagnostic = pd.DataFrame(_rows).sort_values('qtr_end_WAPE_pct').reset_index(drop=True)
    print('=' * 72)
    print('ITSG QUARTER-END HOLDOUT BIAS DIAGNOSTIC (reporting only -- no calibration change)')
    print(f'  Holdout quarter-end sample: {int(_bias_src["is_qe"].sum())} (model, week) rows '
          f'across {_bias_src.loc[_bias_src["is_qe"], "ds"].nunique()} distinct calendar weeks.')
    print('  qtr_end_bias_pct < 0 = under-forecasts quarter-end (forecast below actual).')
    print('=' * 72)
    display(itsg_qe_bias_diagnostic)

    _ens_row = itsg_qe_bias_diagnostic[itsg_qe_bias_diagnostic['model'] == 'Adaptive_Ensemble']
    if not _ens_row.empty and pd.notna(_ens_row['qtr_end_bias_pct'].iloc[0]) and pd.notna(_ens_row['other_bias_pct'].iloc[0]):
        _qe_b = _ens_row['qtr_end_bias_pct'].iloc[0]
        _other_b = _ens_row['other_bias_pct'].iloc[0]
        if abs(_qe_b - _other_b) > 8.0:
            print(f"\n  NOTE: Adaptive_Ensemble's quarter-end bias ({_qe_b:+.2f}%) differs meaningfully "
                  f"from its other-week bias ({_other_b:+.2f}%) -- i.e. despite QE_DISPERSION_RATIO=1.00x, "
                  f"the ensemble IS systematically mis-forecasting quarter-end specifically. Worth "
                  f"considering a QE-specific correction (UCC-style), not just wider intervals.")
        else:
            print(f"\n  Adaptive_Ensemble's quarter-end bias ({_qe_b:+.2f}%) is close to its other-week "
                  f"bias ({_other_b:+.2f}%) -- despite the raw ARR surge (Step 7B-QE), the ensemble's "
                  f"POINT FORECAST already tracks it reasonably well. No QE-specific bias correction "
                  f"needed at this time; QE_DISPERSION_RATIO=1.00x looks correct, not misleading.")
except Exception as e:
    itsg_qe_bias_diagnostic = None
    print(f'ITSG quarter-end holdout bias diagnostic skipped (non-fatal): {e}')


### Step 8B — Complete run summary dashboard **(NEW — team-share packet)**

Run this after Step 8 when preparing the notebook for team review. It mirrors the UCC notebook's end-of-run snapshot and consolidates the key outputs produced in this run:

* Run metadata and output table destinations
* Live ensemble composition and weights
* Holdout leaderboard and closed-quarter audit
* Forward quarterly outlook plus Monte Carlo FY distribution
* Planning-threshold probabilities that can be copied directly into readouts

Every section below is rendered from live DataFrames generated in this run, so the notebook stays consistent with the actual pipeline outputs.

In [ ]:
# STEP 8B: COMPLETE RUN SUMMARY DASHBOARD
# =============================================================================
# PURPOSE: Provide a compact end-of-run operational dashboard similar to the
# UCC notebook, so reviewers can inspect the live setup and core outputs from
# one place without reading the full notebook.
#
# TEAM-SHARE SECTIONS:
#   1. Run metadata and published-table destinations
#   2. Live ensemble weights and governance outcome
#   3. Holdout leaderboard and closed-quarter audit
#   4. Forward quarterly outlook
#   5. Monte Carlo quarterly / FY distribution and planning thresholds
# =============================================================================

required_objects = [
    'weekly_itsg', 'holdout_leaderboard', 'holdout_quarter_scoreboard',
    'production_quarterly_forecasts', 'ensemble_weights',
    'mc_quarterly_distribution', 'mc_fy_distribution'
]
missing_objects = [name for name in required_objects if name not in globals()]
if missing_objects:
    raise NameError(f"Step 8B prerequisites missing: {missing_objects}. Run Steps 7, 7B, and 8 first.")

adaptive_row = holdout_leaderboard.loc[holdout_leaderboard['model'] == 'Adaptive_Ensemble'].iloc[0]
summary_rows = [
    ('Notebook version', NOTEBOOK_VERSION),
    ('Observed data through', str(OBSERVED_END.date())),
    ('Weekly observations', int(len(weekly_itsg))),
    ('Live ensemble components', ', '.join(ENSEMBLE_COMPONENT_MODELS)),
    ('Selection method', ENSEMBLE_SELECTION_METHOD),
    ('Ensemble weighting', ensemble_weights['weighting_method'].iloc[0]),
    ('Adaptive holdout WAPE %', round(float(adaptive_row['wape']), 3)),
    ('Adaptive holdout bias %', round(float(adaptive_row['mpe']), 3)),
    ('Primary output table', PRIMARY_PRODUCTION_OUTPUT_TABLE),
    ('Legacy output table', LEGACY_PRODUCTION_OUTPUT_TABLE),
    ('Monitoring baseline table', MONITORING_BASELINE_TABLE),
]
run_summary_dashboard = pd.DataFrame(summary_rows, columns=['metric', 'value'])
run_summary_dashboard['value'] = run_summary_dashboard['value'].map(
    lambda x: '' if pd.isna(x) else str(x)
)

ensemble_weight_view = ensemble_weights[['model', 'mean_wape', 'ensemble_weight', 'weighting_method']].copy()
ensemble_weight_view['ensemble_weight_pct'] = (ensemble_weight_view['ensemble_weight'] * 100).round(1)
ensemble_weight_view = ensemble_weight_view[[
    'model', 'mean_wape', 'ensemble_weight_pct', 'weighting_method'
]].sort_values('ensemble_weight_pct', ascending=False).reset_index(drop=True)

leaderboard_view = holdout_leaderboard.copy().sort_values('wape').reset_index(drop=True)
adaptive_quarters = holdout_quarter_scoreboard[
    holdout_quarter_scoreboard['model'] == 'Adaptive_Ensemble'
].copy()
forward_focus = production_quarterly_forecasts[
    production_quarterly_forecasts['model'].isin(['Adaptive_Ensemble'] + ENSEMBLE_COMPONENT_MODELS)
].copy()
forward_focus = forward_focus.sort_values(['quarter', 'forecast_arr_m'], ascending=[True, False]).reset_index(drop=True)

mc_quarter_view = mc_quarterly_distribution.copy()
fy_distribution_view = mc_fy_distribution.copy()
fy_threshold_view = (
    mc_threshold_table[mc_threshold_table['scope'] == CURRENT_FY_LABEL].copy().sort_values('threshold_m')
    if 'mc_threshold_table' in globals() else pd.DataFrame()
)

print('=' * 88)
print('ITSG FORECAST COMPLETE RUN SUMMARY')
print('=' * 88)

print('\n1. Run metadata and destinations:')
display(run_summary_dashboard)

print('\n2. Live ensemble weights:')
display(ensemble_weight_view)

print('\n3. Holdout leaderboard:')
display(leaderboard_view)

print('\n4. Adaptive Ensemble closed-quarter audit:')
display(adaptive_quarters[['quarter', 'weeks', 'actual_arr_m', 'forecast_arr_m', 'pct_error']])

print('\n5. Forward quarterly outlook ($M):')
display(forward_focus)

print('\n6. Monte Carlo quarterly distribution ($M):')
display(mc_quarter_view)

print(f'\n7. Monte Carlo {CURRENT_FY_LABEL} distribution ($M):')
display(fy_distribution_view)

if not fy_threshold_view.empty:
    print(f'\n8. {CURRENT_FY_LABEL} planning thresholds:')
    display(fy_threshold_view)

print('\nShare note: pair any point forecast with the Monte Carlo P05 / P25 / P50 / P75 / P95 range when sending ITSG updates to leadership or FP&A.')


# ITSG Forecast Foundation V5 — Team Runbook

## 1. Purpose

This notebook is the canonical ITSG Growth weekly forecast pipeline for team sharing. It starts from raw
Salesforce OLI data, runs benchmark and global models under UCC-aligned validation rules, selects the live
ensemble explicitly, publishes forward forecast tables, and produces the end-of-run summary packet used in
reviews with FP&A and leadership.

## 2. Start here

1. Run Step 0 once per fresh compute session, then re-run the notebook from the top.
2. For a complete production refresh, run all code cells sequentially through Step 8B and the final
   summary cell below.
3. For team review, share Step 8's generated executive summary, Step 8B's complete run summary dashboard,
   and the final FY2026 quarter table together.
4. `datagroup_mdl.mdl_sales_analytics.forecast_prophet` remains a read-only feature source and is never
   written by this notebook.

## 3. Source and output tables

| Table | Access | Purpose |
| --- | --- | --- |
| `datagroup.datalake_transform.cds_sfdc_opp_products_latest` | read | Target: closed-won ITSG Growth `amount_towards_plan` on the complete weekly spine |
| `datagroup_mdl.mdl_sales_analytics.forecast_prophet` | **read-only** | Features only (pipeline, headcount, activity, plan). `Actuals` is diagnostic-only and never used as target |
| `datagroup.uc_forecast.actuals` (product `RSG-IT`) | read | Authoritative Magna Carta actuals used for `mc_actuals_lag4` |
| `itsg_forecast_v5` | write | Primary production forward-forecast table |
| `itsg_forecast_v4` | write | Legacy mirror for backward compatibility |
| `itsg_forecast_monitoring_baseline` | write (append) | Monitoring baseline and drift thresholds |

## 4. Validation and governance design

CV region (through 13 weeks before the holdout; computed fold count x 13-week expanding walk-forward,
boundaries asserted) -> validation slice (final 13 pre-holdout weeks; all hyperparameter, window, and
model-inclusion decisions; never reported as accuracy) -> holdout (2026-01-05 onward; never trained on or
tuned against). Final models retrain on all pre-holdout data before holdout scoring, and on all closed
weeks before the production forecast. Non-negativity is applied inside every model's predict path, so CV,
holdout, and production receive identical post-processing. Ensemble membership is now governed explicitly in
Step 6B and then consumed authoritatively in Step 7.

## 5. What to share with the team

* Step 8 generated executive summary for the narrative readout
* Step 8B complete run summary dashboard for operations, weights, outputs, and Monte Carlo ranges
* Final FY2026 quarter table for closed-quarter audit plus forward-quarter outlook
* `itsg_forecast_monitoring_baseline` for the persisted benchmark used in future drift checks

## 6. Key V5 upgrades vs V4

| # | Step(s) | Change | Why |
| --- | --- | --- | --- |
| 1 | 2, 3, 4, 5, 6 | Three-way split; CV folds computed and constrained to the CV region; boundaries asserted. | V4's hardcoded later folds trained inside the 2026 holdout. |
| 2 | 6 | LGB panel rebuilt on the complete week x market spine; SFDC ATP target; `forecast_prophet` features-only and forward-filled. | The prior partial spine created holes in the target and lag features. |
| 3 | 6B, 7 | UCC-style ensemble membership validation added and made authoritative for production. | Live membership now comes from explicit evidence, not implicit pass-through. |
| 4 | 7 | Removed the holdout-derived production scaling and standardized weights on inverse CV WAPE. | Production now matches the evaluated pipeline and reported metrics. |
| 5 | 7 | MinT market allocation clips negatives then rescales to preserve totals. | Market forecasts stay coherent with the published total forecast. |
| 6 | 8, 8B | Generated executive summary, monitoring baseline, and complete run summary dashboard all render from live DataFrames. | Team-facing outputs stay synchronized with the actual run artifacts. |
| 7 | 7B | Monte Carlo quarterly/FY distributions added from out-of-sample residuals. | For a small noisy series, the distribution is as important as the point estimate. |

## 7. Operational notes

Run weekly after the SFDC refresh. The 27-week horizon crosses into 2027Q1, so quarterly summaries can
include a partial quarter at the tail. Expect ITSG accuracy to remain worse than larger product lines:
with roughly $0.35M average weekly bookings, a few deals can move a week by double-digit percentages. For
leadership sharing, pair every point forecast with the Step 7B Monte Carlo range rather than quoting a
single weekly number in isolation.

In [ ]:
# =============================================================================
# FINAL SUMMARY TABLE: CURRENT-FY Q1-Q4 ACTUALS vs FORECAST (ALL MODELS)
# =============================================================================
# PURPOSE: Produce the definitive quarter-level summary for team review.
# Closed quarters come from holdout actuals vs each model; forward quarters come
# from the Monte Carlo distribution built on the live Adaptive Ensemble.
# =============================================================================

import numpy as np
import pandas as pd

required_objects = [
    'holdout_quarter_scoreboard', 'mc_quarterly_distribution',
    'mc_fy_distribution', 'weekly_itsg'
]
missing_objects = [name for name in required_objects if name not in globals()]
if missing_objects:
    raise NameError(f"Final summary prerequisites missing: {missing_objects}. Run Steps 7B, 8, and 8B first.")

closed_q = holdout_quarter_scoreboard[['model', 'quarter', 'weeks', 'actual_arr_m', 'forecast_arr_m', 'pct_error']].copy()
closed_q.columns = ['Model', 'Quarter', 'Weeks', 'Actual ($M)', 'Forecast ($M)', 'Error %']
closed_q = closed_q.sort_values(
    ['Quarter', 'Error %'],
    key=lambda s: s.abs() if s.name == 'Error %' else s
).reset_index(drop=True)

quarter_cols = ['quarter', 'p05_m', 'p25_m', 'p50_m', 'p75_m', 'p95_m']
if 'weeks_simulated' in mc_quarterly_distribution.columns:
    quarter_cols.insert(1, 'weeks_simulated')
forward_q = mc_quarterly_distribution[quarter_cols].copy()
forward_q = forward_q.rename(columns={
    'quarter': 'Quarter',
    'weeks_simulated': 'Weeks',
    'p05_m': 'P05 ($M)',
    'p25_m': 'P25 ($M)',
    'p50_m': 'P50 ($M)',
    'p75_m': 'P75 ($M)',
    'p95_m': 'P95 ($M)',
})

# 2026-09-30 FIX: forward_q previously included EVERY quarter present in
# mc_quarterly_distribution, with no filter to the current fiscal year --
# since ITSG's forward horizon runs longer than one FY (27 weeks), this
# silently pulled in a quarter from the NEXT fiscal year (e.g. 2027Q1) once
# the forward window crossed the FY boundary. That extra quarter's P50 was
# getting summed into `fy_forward_p50` below, inflating "Forward <quarters>
# P50" -- previously invisible because the print label was hardcoded to
# "Q3+Q4" regardless of what was actually in forward_q/summed. Filter to
# CURRENT_FY quarters only so the displayed table and the sum agree with
# what "forward for this fiscal year" actually means.
forward_q = forward_q[forward_q['Quarter'].astype(str).str.startswith(str(CURRENT_FY))].reset_index(drop=True)

# CHANGED: CURRENT_FY (from Step 7, derived from weekly_itsg's actual max
# closed week) replaces the hardcoded 2026 literal.
fy_closed = float(weekly_itsg.loc[weekly_itsg['ds'].dt.year == CURRENT_FY, 'y'].sum()) / 1e6
fy_forward_p50 = float(forward_q['P50 ($M)'].sum())
fy_distribution = mc_fy_distribution.iloc[0]
fy_threshold_view = (
    mc_threshold_table[mc_threshold_table['scope'] == f'FY{CURRENT_FY}'].copy().sort_values('threshold_m')
    if 'mc_threshold_table' in globals() else pd.DataFrame()
)

print('\n' + '=' * 88)
print(f'   FY{CURRENT_FY} ITSG GROWTH ARR — QUARTERLY ACTUALS vs FORECAST (FINAL)')
print('=' * 88)

print('\nCLOSED QUARTERS (actuals available)')
print('Best model per quarter is the row with the lowest absolute Error %.\n')
display(closed_q)

print('\nFORWARD QUARTERS (Monte Carlo ensemble forecast)')
print('Based on the Adaptive Ensemble residual distribution from the holdout period.\n')
display(forward_q)

print(f'\nFY{CURRENT_FY} FULL-YEAR OUTLOOK')
print(f'  Closed FY{CURRENT_FY} actuals    : ${fy_closed:.3f}M')
_fwd_quarters_label = '+'.join(sorted(forward_q['Quarter'].astype(str).unique()))  # 2026-09-30 FIX: was hardcoded 'Q3+Q4' -- wrong once run after Q3 closes (forward = Q4 only) or after an FY turn (forward = next FY's Q1/Q2). Build the label from the quarters actually present in forward_q instead.
print(f'  Forward {_fwd_quarters_label} P50        : ${fy_forward_p50:.3f}M')
print(f'  FY{CURRENT_FY} P50 total         : ${fy_distribution["p50_m"]:.3f}M')
print(f'  FY{CURRENT_FY} P05 to P95 range  : ${fy_distribution["p05_m"]:.3f}M to ${fy_distribution["p95_m"]:.3f}M')
print(f'  FY{CURRENT_FY} P25 to P75 range  : ${fy_distribution["p25_m"]:.3f}M to ${fy_distribution["p75_m"]:.3f}M')

if not fy_threshold_view.empty:
    print(f'\nFY{CURRENT_FY} planning thresholds')
    display(fy_threshold_view)

print('=' * 88)


In [ ]:
# STEP 8C: ROLLING MULTI-YEAR FORECAST — TWO EXECUTIVE VIEWS (ITSG)
# =============================================================================
# PURPOSE: Two complementary planning views that roll forward indefinitely,
# adapted from UCC Forecast Foundation Step 15C for the ITSG model stack
# (Prophet_trend, MSTL_v2, ETS, DHR_ARIMA, Global_LGB_Q50_ITSG, Adaptive_Ensemble).
#
# VIEW A — "BOQ PURE MODEL FORECAST" (Sona-consistent)
#   Full-quarter model predictions with NO actuals blended in. This is the
#   model's best estimate for each complete quarter, locked at run time.
#   Use this as the plan target / baseline. Shows all retained models +
#   Adaptive Ensemble.
#
# VIEW B — "ADAPTIVE PLANNING VIEW" (live trajectory)
#   As weeks close, actuals replace model predictions. Shows real-time pace:
#   - If actuals > model pace -> planning total RISES above BOQ -> ahead of plan
#   - If actuals < model pace -> planning total FALLS below BOQ -> behind plan
#   Includes pace_vs_boq_pct (%) so executives see at a glance if the quarter
#   is tracking hot or cold relative to the pure model prediction.
#
# ROLLING LOGIC: Always shows FORECAST_HORIZON_MONTHS // 3 (= 6) quarters ahead from the current quarter.
#   Window auto-extends every run as the current quarter changes.
#
# CRITICAL DESIGN DECISION — WHY NOT REFIT FOR THE CURRENT PRODUCTION WINDOW:
#   Do NOT re-call fit_predict_prophet_trend() (or any model fit) for weeks
#   already covered by Step 7's production forecast. Prophet's MAP fit is
#   stochastic (cmdstanpy), so refitting drifts the quarterly number between
#   runs, and Step 7 already applies interval-ratio calibration (from CV /
#   holdout residuals via apply_interval_adjustments) that a raw refit loses.
#   For weeks through PRODUCTION_END, reuse production_total_forecasts
#   (Step 7's output, which already includes the calibrated Adaptive_Ensemble
#   rows — unlike UCC this notebook keeps model + ensemble in ONE frame, so no
#   separate ensemble variable is needed). Only refit stat models + LGB for
#   weeks BEYOND PRODUCTION_END (the extension into the following year).
# =============================================================================

import pandas as pd
import numpy as np

# ── Section 1: Dynamic rolling window — 4 quarters ahead from current quarter ──
_today = pd.Timestamp.today().normalize()
_current_quarter = _today.to_period('Q')
_target_end_quarter = _current_quarter + FORECAST_HORIZON_MONTHS // 3  # 6 quarters ahead
_target_end_date = pd.Timestamp(_target_end_quarter.end_time.date())
_extended_end = _target_end_date - pd.Timedelta(days=_target_end_date.weekday())
if _extended_end > _target_end_date:
    _extended_end -= pd.Timedelta(weeks=1)

_data_end = weekly_itsg['ds'].max()
_extended_start = _data_end + pd.Timedelta(weeks=1)
_extended_forecast_weeks = pd.date_range(_extended_start, _extended_end, freq='W-MON')
_extended_horizon = len(_extended_forecast_weeks)

# PRODUCTION_END: last week already covered by Step 7's calibrated production
# forecast. This notebook has no separate named constant for it (Step 7 just
# prints future_weeks.max()), so derive it directly.
PRODUCTION_END = future_weeks.max()

print(f'Rolling forecast window: {_current_quarter} -> {_target_end_quarter} ({FORECAST_HORIZON_MONTHS // 3} quarters ahead)')
print(f'  Data end: {_data_end.date()} | Extended horizon: {_extended_horizon} weeks -> {_extended_end.date()}')
print(f'  Current production end: {PRODUCTION_END.date()}')
print()

_beyond_production_weeks = _extended_forecast_weeks[_extended_forecast_weeks > PRODUCTION_END]
_beyond_horizon = len(_beyond_production_weeks)

print(f'Using Step 7 production forecast for weeks through {PRODUCTION_END.date()} (deterministic, calibrated)')
print(f'Extending {_beyond_horizon} weeks beyond into the following year...')

extended_total_rows = []

# ── Section 2a: current production window — reuse Step 7 output directly ──────
_prod_total = production_total_forecasts[['model', 'ds', 'forecast', 'p50']].copy()
_prod_total['ds'] = pd.to_datetime(_prod_total['ds'])
extended_total_rows.append(_prod_total)

# ── Section 2b: extension beyond PRODUCTION_END — refit over the FULL extended
# horizon (so recursive/lag features stay continuous), then slice to only the
# weeks beyond PRODUCTION_END ──────────────────────────────────────────────────
if _beyond_horizon > 0:
    for model_name, fit_fn, kwargs in benchmark_model_specs:
        pred_full = np.asarray(fit_fn(model_df[['ds', 'y']].copy(), horizon=_extended_horizon, **kwargs), dtype=float)
        _all_df = pd.DataFrame({'ds': _extended_forecast_weeks, 'forecast': pred_full})
        _beyond = _all_df[_all_df['ds'] > PRODUCTION_END].copy()
        _beyond['model'] = model_name
        _beyond['p50'] = _beyond['forecast']
        extended_total_rows.append(_beyond[['model', 'ds', 'forecast', 'p50']])
        print(f'  {model_name}: {len(_beyond)} extension weeks')

    # DHR-ARIMA extension
    dhr_ext_pred = np.asarray(
        fit_predict_dhr_arima(model_df[['ds', 'y']].copy(), horizon=_extended_horizon,
                               full_feature_df=dhr_df, **BEST_DHR_PARAMS),
        dtype=float,
    )
    _dhr_all = pd.DataFrame({'ds': _extended_forecast_weeks, 'forecast': dhr_ext_pred})
    _dhr_beyond = _dhr_all[_dhr_all['ds'] > PRODUCTION_END].copy()
    _dhr_beyond['model'] = DHR_MODEL_NAME
    _dhr_beyond['p50'] = _dhr_beyond['forecast']
    extended_total_rows.append(_dhr_beyond[['model', 'ds', 'forecast', 'p50']])
    print(f'  {DHR_MODEL_NAME}: {len(_dhr_beyond)} extension weeks')

    # LGB extension — build the forward panel over the FULL extended horizon
    # (mirrors Step 7's build_forward_lgb_panel(), parameterized by weeks so it
    # can reach past the 27-week production horizon). Reuse the already-fit
    # production_lgb_models (same training data/params as Step 7 — deterministic,
    # so no refit needed, just a longer recursive prediction), then slice.
    try:
        def _build_extended_lgb_panel(weeks_index):
            fwd = pd.MultiIndex.from_product(
                [weeks_index, HIERARCHY_MARKETS], names=['ds', 'sales_market']
            ).to_frame(index=False)
            fwd['y'] = np.nan
            fwd['deal_count'] = 0.0
            fwd['slice_id'] = PRODUCT_SCOPE + '|' + fwd['sales_market']
            fwd = add_calendar_columns(fwd)
            hist = global_panel.sort_values('ds').groupby('slice_id').tail(104)
            for col in FP_FEATURE_COLS:
                profile = hist.groupby(['sales_market', 'week_of_qtr'])[col].median().to_dict()
                market_default = hist.groupby('sales_market')[col].median().to_dict()
                fwd[col] = [profile.get((m, wk), market_default.get(m, 0.0))
                            for m, wk in zip(fwd['sales_market'], fwd['week_of_qtr'])]
            fwd['month_key'] = fwd['ds'].dt.to_period('M').astype(str)
            future_auth = load_authoritative_actuals(fwd['ds'].min(), fwd['ds'].max())
            fwd = fwd.merge(future_auth, on=['month_key', 'sales_market'], how='left')
            base_cols = ['ds', 'sales_market', 'slice_id', 'y', 'deal_count', 'month_key',
                         'authoritative_actuals'] + FP_FEATURE_COLS
            hist_base = global_panel[base_cols].copy()
            combined = pd.concat([hist_base, fwd[base_cols]], ignore_index=True)
            combined = add_calendar_columns(combined)
            return _engineer_panel_features(combined)

        _ext_lgb_panel = _build_extended_lgb_panel(_extended_forecast_weeks)
        _ext_lgb_scored = recursive_lgb_predict(production_lgb_models, _ext_lgb_panel, _extended_start, _extended_end)
        _lgb_all = (
            _ext_lgb_scored.groupby('ds', as_index=False)
            .agg(forecast=('p50', 'sum'), p50=('p50', 'sum'))
            .assign(model=GLOBAL_MODEL_NAME)
        )
        _lgb_beyond = _lgb_all[_lgb_all['ds'] > PRODUCTION_END].copy()
        extended_total_rows.append(_lgb_beyond[['model', 'ds', 'forecast', 'p50']])
        print(f'  {GLOBAL_MODEL_NAME}: {len(_lgb_beyond)} extension weeks')
    except Exception as e:
        print(f'  {GLOBAL_MODEL_NAME}: SKIPPED for extension ({type(e).__name__}: {e})')

    # Equal-weight ensemble for the extension weeks only
    _ext_parts = pd.concat(extended_total_rows, ignore_index=True)
    _ext_parts['ds'] = pd.to_datetime(_ext_parts['ds'])
    _beyond_only = _ext_parts[_ext_parts['ds'] > PRODUCTION_END].copy()
    _ens_models_avail = [m for m in ENSEMBLE_COMPONENT_MODELS if m in _beyond_only['model'].unique()]
    _ens_beyond = (
        _beyond_only[_beyond_only['model'].isin(_ens_models_avail)]
        .groupby('ds', as_index=False).agg(forecast=('p50', 'mean'))
        .assign(model='Adaptive_Ensemble', p50=lambda d: d['forecast'])
    )
    extended_total_rows.append(_ens_beyond[['model', 'ds', 'forecast', 'p50']])
    print(f'  Ensemble ({len(_ens_models_avail)} models) for extension weeks')

print()

extended_model_forecasts = pd.concat(extended_total_rows, ignore_index=True)
extended_model_forecasts['ds'] = pd.to_datetime(extended_model_forecasts['ds'])
extended_model_forecasts = extended_model_forecasts.drop_duplicates(subset=['model', 'ds'], keep='first')

# ── Section 3: BOQ quarterly aggregation — holdout weekly + extended weekly ────
extended_model_forecasts['quarter'] = extended_model_forecasts['ds'].dt.to_period('Q').astype(str)
ext_weekly = extended_model_forecasts[['model', 'ds', 'quarter', 'forecast']].copy()

holdout_weekly = holdout_benchmark_predictions[['model', 'ds', 'forecast']].copy()
holdout_weekly['ds'] = pd.to_datetime(holdout_weekly['ds'])
holdout_weekly['quarter'] = holdout_weekly['ds'].dt.to_period('Q').astype(str)

_hb_ens = holdout_benchmark_predictions.merge(
    ensemble_weights[['model', 'ensemble_weight']], on='model', how='inner'
)
_ens_holdout_weekly = (
    _hb_ens.groupby('ds', as_index=False)
    .agg(forecast=('forecast', lambda s: float(np.average(s, weights=_hb_ens.loc[s.index, 'ensemble_weight']))))
    .assign(model='Adaptive_Ensemble')
)
_ens_holdout_weekly['ds'] = pd.to_datetime(_ens_holdout_weekly['ds'])
_ens_holdout_weekly['quarter'] = _ens_holdout_weekly['ds'].dt.to_period('Q').astype(str)
holdout_weekly = pd.concat(
    [holdout_weekly, _ens_holdout_weekly[['model', 'ds', 'quarter', 'forecast']]], ignore_index=True
)

# Combine holdout + extended weekly -> full 13-week quarterly BOQ predictions
all_model_weekly = pd.concat([holdout_weekly, ext_weekly], ignore_index=True)
all_model_weekly = all_model_weekly.drop_duplicates(subset=['model', 'ds'], keep='first')

boq_quarterly = (
    all_model_weekly.groupby(['model', 'quarter'], as_index=False)
    .agg(forecast_weeks=('ds', 'nunique'), forecast_arr=('forecast', 'sum'))
)
boq_quarterly['forecast_arr_m'] = (boq_quarterly['forecast_arr'] / 1e6).round(3)

_all_quarters = sorted(boq_quarterly[boq_quarterly['model'] == 'Adaptive_Ensemble']['quarter'].unique())
_fy_this, _fy_next = _current_quarter.year, _current_quarter.year + 1

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# VIEW A: BOQ PURE MODEL FORECAST (no actuals — Sona-consistent)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
_exec_models = ['Adaptive_Ensemble'] + PRODUCTION_BENCHMARK_MODELS
boq_pivot = boq_quarterly[boq_quarterly['model'].isin(_exec_models)].pivot_table(
    index='model', columns='quarter', values='forecast_arr_m', aggfunc='first'
)
boq_pivot = boq_pivot.reindex([m for m in _exec_models if m in boq_pivot.index])

_fy_this_cols = [c for c in boq_pivot.columns if c.startswith(str(_fy_this))]
_fy_next_cols = [c for c in boq_pivot.columns if c.startswith(str(_fy_next))]
boq_pivot[f'FY{_fy_this}_m'] = boq_pivot[_fy_this_cols].sum(axis=1).round(2)
if _fy_next_cols:
    boq_pivot[f'FY{_fy_next}_m'] = boq_pivot[_fy_next_cols].sum(axis=1).round(2)
boq_pivot = boq_pivot.reset_index()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# VIEW B: ADAPTIVE PLANNING VIEW (actuals + forward, with pace tracking vs BOQ)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
actual_all = (
    weekly_itsg.assign(quarter=lambda d: d['ds'].dt.to_period('Q').astype(str))
    .groupby('quarter', as_index=False)
    .agg(actual_arr=('y', 'sum'), closed_weeks=('ds', 'nunique'))
)
actual_all['actual_arr_m'] = (actual_all['actual_arr'] / 1e6).round(3)
actual_dict = dict(zip(actual_all['quarter'], zip(actual_all['actual_arr_m'], actual_all['closed_weeks'])))

_ens_ext_weekly = extended_model_forecasts[
    extended_model_forecasts['model'] == 'Adaptive_Ensemble'
][['ds', 'quarter', 'forecast']].copy()

boq_ens_dict = dict(zip(
    boq_quarterly[boq_quarterly['model'] == 'Adaptive_Ensemble']['quarter'],
    boq_quarterly[boq_quarterly['model'] == 'Adaptive_Ensemble']['forecast_arr_m']
))

adaptive_rows = []
for q in _all_quarters:
    actual_m, actual_weeks = actual_dict.get(q, (0.0, 0))
    boq_m = boq_ens_dict.get(q, 0.0)

    if actual_weeks >= 13:
        pace = round(((actual_m / boq_m) - 1) * 100, 1) if boq_m > 0 else 0.0
        adaptive_rows.append({
            'quarter': q, 'actual_m': actual_m, 'fwd_forecast_m': 0.0,
            'adaptive_total_m': actual_m, 'boq_model_m': boq_m,
            'pace_vs_boq_pct': pace, 'status': '\u2713 complete',
        })
    elif actual_weeks > 0:
        remaining_m = round(_ens_ext_weekly[_ens_ext_weekly['quarter'] == q]['forecast'].sum() / 1e6, 3)
        remaining_wks = len(_ens_ext_weekly[_ens_ext_weekly['quarter'] == q])
        adaptive_total = round(actual_m + remaining_m, 3)
        pace = round(((adaptive_total / boq_m) - 1) * 100, 1) if boq_m > 0 else 0.0
        adaptive_rows.append({
            'quarter': q, 'actual_m': actual_m, 'fwd_forecast_m': remaining_m,
            'adaptive_total_m': adaptive_total, 'boq_model_m': boq_m,
            'pace_vs_boq_pct': pace, 'status': f'{actual_weeks}wk actual + {remaining_wks}wk fwd',
        })
    else:
        fwd_m = round(_ens_ext_weekly[_ens_ext_weekly['quarter'] == q]['forecast'].sum() / 1e6, 3)
        fwd_wks = len(_ens_ext_weekly[_ens_ext_weekly['quarter'] == q])
        adaptive_rows.append({
            'quarter': q, 'actual_m': 0.0, 'fwd_forecast_m': fwd_m,
            'adaptive_total_m': fwd_m, 'boq_model_m': boq_m,
            'pace_vs_boq_pct': 0.0, 'status': f'forecast ({fwd_wks}wk)',
        })

adaptive_view = pd.DataFrame(adaptive_rows)
adaptive_view['fiscal_year'] = 'FY' + adaptive_view['quarter'].str[:4]

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# DISPLAY BOTH VIEWS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print('=' * 100)
print(f'ROLLING MULTI-YEAR ITSG FORECAST — {_current_quarter} through {_target_end_quarter}')
print(f'  Window auto-rolls each run. Today: {_today.date()}')
print('=' * 100)

print('\n' + '\u2501' * 100)
print('VIEW A: BOQ PURE MODEL FORECAST ($M) — Full 13-week model prediction, no actuals blended.')
print('  This is the plan target (Sona-consistent). Stays fixed once a quarter starts.')
print('  Compare against actuals to measure model quality.')
print('\u2501' * 100)
display(boq_pivot)

_ens_boq = boq_pivot[boq_pivot['model'] == 'Adaptive_Ensemble'].iloc[0]
print(f'\n  -> Ensemble BOQ: ', end='')
for q in _all_quarters:
    if q in _ens_boq.index:
        print(f'{q}=${_ens_boq[q]:.1f}M  ', end='')
print(f'| FY{_fy_this}=${_ens_boq.get(f"FY{_fy_this}_m", 0):.1f}M', end='')
if f'FY{_fy_next}_m' in _ens_boq.index:
    print(f'  FY{_fy_next}=${_ens_boq[f"FY{_fy_next}_m"]:.1f}M', end='')
print()

print('\n' + '\u2501' * 100)
print('VIEW B: ADAPTIVE PLANNING VIEW ($M) — Actuals replace model as weeks close. Tracks pace vs BOQ.')
print('  pace_vs_boq > 0% = AHEAD of model (upside); < 0% = BEHIND model (downside risk).')
print('  This view updates each run. When ahead -> model is conservative; behind -> flag for exec review.')
print('\u2501' * 100)
display(adaptive_view[['quarter', 'actual_m', 'fwd_forecast_m', 'adaptive_total_m', 'boq_model_m', 'pace_vs_boq_pct', 'status']])

print('\nFiscal Year Summaries:')
for fy in sorted(adaptive_view['fiscal_year'].unique()):
    fy_rows = adaptive_view[adaptive_view['fiscal_year'] == fy]
    adapt_total = fy_rows['adaptive_total_m'].sum()
    boq_total = fy_rows['boq_model_m'].sum()
    pace = ((adapt_total / boq_total) - 1) * 100 if boq_total > 0 else 0
    n_complete = (fy_rows['status'].str.contains('complete')).sum()
    n_partial = (fy_rows['status'].str.contains('actual')).sum()
    n_fwd = len(fy_rows) - n_complete - n_partial
    print(f'  {fy}: Adaptive=${adapt_total:.2f}M | BOQ Model=${boq_total:.2f}M | '
          f'Pace={pace:+.1f}% | ({n_complete} complete, {n_partial} partial, {n_fwd} forecast)')

print('\n' + '\u2500' * 100)
print('ACTUALS REFERENCE:')
_actuals_display = actual_all[actual_all['quarter'].isin(_all_quarters)].copy()
_actuals_display['status'] = np.where(_actuals_display['closed_weeks'] >= 13, 'complete', 'partial')
if len(_actuals_display) > 0:
    display(_actuals_display[['quarter', 'actual_arr_m', 'closed_weeks', 'status']])

print(f'\n\u2713 Both views generated successfully. Window auto-rolls each run.')
print(f'  VIEW A = Model prediction (plan target, doesn\'t change mid-quarter) — for Sona alignment')
print(f'  VIEW B = Live trajectory (actuals + forecast) — shows if ahead/behind model, adjusts weekly')
print(f'  Next run after {_current_quarter} closes: window extends to {_target_end_quarter + 1}.')


In [ ]:
# STEP 8D: PERSIST BOQ + CBE PLANNING VIEWS TO DELTA (ITSG)
# =============================================================================
# PURPOSE: Write two production planning tables with p05/p25/p50/p75/p95
# quantiles (CHANGED from the old p10-p90-by-10s decile fan, to match the
# notebook-wide 5-column convention), mirroring UCC's Step 14E. These feed
# the executive app and forecast_unify_publish.
#
# TABLE 1 — itsg_forecast_boq (Forecast Only)
#   Rolling 1-year BOQ plan. All models, all weeks from the start of last closed
#   quarter through today + 1 year. Pure model predictions, no actuals blended.
#   source = 'forecast' everywhere.
#
# TABLE 2 — itsg_forecast_cbe (Current Best Estimate = Forecast + Actuals)
#   Actuals for closed weeks + forecasts for open weeks.
#   Includes: cumulative_qtd, boq_weekly, boq_cumulative_qtd, pace_vs_boq_pct.
#   Actuals rows get p05 = p25 = p50 = p75 = p95 = actual value (no uncertainty on realized).
#
# NOTE: CHANGED — this table used to carry the full p10-p90-by-10s decile fan.
# That fan was the OPERATIVE convention here (no p05/p25/p75/p95 existed
# anywhere in this cell), so it is converted rather than left as a superset
# (contrast with a genuinely wider grid, which would be left alone). MC
# simulation quantiles for Adaptive_Ensemble; individual models use
# holdout-residual relative-error factors at the 5 target quantile levels.
# =============================================================================

import pandas as pd
import numpy as np
from datetime import datetime, timezone

SCHEMA_PREFIX = 'datagroup_mdl.mdl_sales_analytics'
BOQ_TABLE = f'{SCHEMA_PREFIX}.itsg_forecast_boq'
CBE_TABLE = f'{SCHEMA_PREFIX}.itsg_forecast_cbe'
PIPELINE_VERSION = PIPELINE_VERSION if 'PIPELINE_VERSION' in dir() else 'V5'

print(f'Persisting ITSG planning tables (pipeline_version={PIPELINE_VERSION})')
print(f'  BOQ -> {BOQ_TABLE}')
print(f'  CBE -> {CBE_TABLE}')
print()

# ── Quantile generation from holdout residuals ───────────────────────────────
# For individual models: use holdout relative errors to generate p05/p25/p75/p95
# For Adaptive_Ensemble: use MC simulation paths (sim_weekly from Step 7B)

# ── Interval calibration target (2026-08-25 recalibration) ──────────────────
# Previous: 80% nominal (percentile 10/90) gave 100% holdout coverage (too wide).
# New: 60% nominal (percentile 20/80) targets ~85-90% holdout coverage.
# Rationale: model accuracy improved ~20pp from training era to 2026 production;
# tighter bands give narrower, more actionable uncertainty and higher confidence.
INTERVAL_LO_PCTILE = 80  # was 90 (upper error bound for the outer-low quantile)
INTERVAL_HI_PCTILE = 20  # was 10 (lower error bound for the outer-high quantile)

# CHANGED: 5-column production convention (was the p10-p90-by-10s decile fan).
QUANTILES = [5, 25, 50, 75, 95]
QUANTILE_COLS = [f'p{q:02d}' for q in QUANTILES]

def _model_quantile_factors(holdout_preds, model_name):
    """Compute p05/p25/p50/p75/p95 relative factors from holdout residuals for
    a model. Calibrated: p05 uses INTERVAL_LO_PCTILE, p95 uses
    INTERVAL_HI_PCTILE, intermediate quantiles (p25/p75) are linearly
    interpolated between them — same calibration intent as the old p10/p90
    version, just re-anchored to the new outer quantile levels (5/95 instead
    of 10/90)."""
    m = holdout_preds[holdout_preds['model'] == model_name].copy()
    fallback = {q: (0.7 + (q - 5) * (0.6 / 90)) for q in QUANTILES}  # linear 0.7-1.3
    fallback[50] = 1.0
    if m.empty or 'actual' not in m.columns:
        return fallback
    rel_errors = (m['forecast'] - m['actual']) / m['actual'].replace(0, np.nan)
    rel_errors = rel_errors.dropna()
    if len(rel_errors) < 5:
        return fallback
    # Calibrated percentile mapping: p05->pctile 80, p95->pctile 20, linear between
    factors = {}
    for q in QUANTILES:
        pctile = INTERVAL_LO_PCTILE - (q - 5) * (INTERVAL_LO_PCTILE - INTERVAL_HI_PCTILE) / 90
        raw = 1 - np.percentile(rel_errors, pctile)
        if q < 50:
            factors[q] = max(0.3, min(raw, 0.99))
        elif q == 50:
            factors[q] = 1.0  # p50 = point forecast unchanged
        else:
            factors[q] = max(1.01, min(raw, 2.0))
    return factors

# Build quantile factors for each model
model_factors = {}
for m_name in all_model_weekly['model'].unique():
    if m_name == 'Adaptive_Ensemble':
        continue  # handled separately via MC
    factors = _model_quantile_factors(holdout_benchmark_predictions, m_name)
    model_factors[m_name] = factors
    print(f'  {m_name}: p05={factors[5]:.3f}, p50={factors[50]:.3f}, p95={factors[95]:.3f}')

# Ensemble quantiles from MC simulation (Step 7B).
# CHANGED / SIMPLIFIED (judgment call): the old version applied a
# recalibration remap (mc_pctiles) that translated nominal decile labels
# (p10..p90) into DIFFERENT empirical percentiles (15th..85th) of the MC
# distribution, compensating for the old raw p10/p90 MC band being too wide.
# Step 7B's Monte Carlo simulation now directly targets the 5 production
# quantile levels (PCTS = [5, 25, 50, 75, 95] in that cell), so re-deriving
# via a recalibrated mc_pctiles mapping here would double-apply a correction
# that no longer applies at the new (already-narrower, already-correct)
# quantile levels. We take the RAW empirical percentile of sim_weekly at each
# of QUANTILES directly instead — simpler, and consistent with Step 7B's own
# percentiles for the same simulated paths.
try:
    # sim_weekly shape: (10000, n_production_weeks)
    _mc_weeks = pd.to_datetime(mc_fw['ds'].values) if 'mc_fw' in dir() else future_weeks
    mc_quantile_map = {}  # ds -> {p05: val, p25: val, ..., p95: val}
    for i, w in enumerate(_mc_weeks[:sim_weekly.shape[1]]):
        week_quantiles = {q: float(np.percentile(sim_weekly[:, i], q)) for q in QUANTILES}
        mc_quantile_map[pd.Timestamp(w)] = week_quantiles
    print(f'  Adaptive_Ensemble: MC quantiles (p05-p95, direct empirical percentiles) from {len(mc_quantile_map)} production weeks')
    _use_mc = True
except Exception as e:
    print(f'  Adaptive_Ensemble: MC quantiles unavailable ({e}), using residual fallback')
    model_factors['Adaptive_Ensemble'] = _model_quantile_factors(holdout_benchmark_predictions, 'Adaptive_Ensemble')
    _use_mc = False

def apply_quantiles(row):
    """Apply p05/p25/p50/p75/p95 (5-column convention) to a forecast row."""
    model = row['model']
    fc = row['forecast']
    ds = pd.Timestamp(row['ds'])
    result = {}

    if model == 'Adaptive_Ensemble' and _use_mc and ds in mc_quantile_map:
        wq = mc_quantile_map[ds]
        for q in QUANTILES:
            result[f'p{q:02d}'] = max(0, wq[q])
    elif model in model_factors:
        factors = model_factors[model]
        for q in QUANTILES:
            result[f'p{q:02d}'] = max(0, fc * factors[q])
    else:
        # Fallback
        for q in QUANTILES:
            factor = 0.7 + (q - 5) * (0.6 / 90)
            if q == 50:
                factor = 1.0
            result[f'p{q:02d}'] = max(0, fc * factor)
    return pd.Series(result)

print()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TABLE 1: BOQ — Rolling 18-month pure model forecast
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Start from FY start (Jan 1) to include full Q1 actuals in the app
# 2026-09-30 FIX: was pd.Timestamp.today().year -- every other FY-scoping
# construct in this notebook (Cells 4, 19, 22, 28) derives the fiscal year
# from weekly_itsg['ds'].max().year (CURRENT_FY) specifically so it tracks
# the DATA's current period, not the wall clock -- if this notebook is ever
# run against stale/lagging source data, today() could silently pick the
# wrong Jan 1 while CURRENT_FY would still reflect what the actuals frame
# actually contains. Reuse CURRENT_FY here too for consistency.
_fy_year = CURRENT_FY
_window_start = pd.Timestamp(f'{_fy_year}-01-01')
_window_start = _window_start - pd.Timedelta(days=_window_start.weekday())  # align to prior Monday
_window_end = pd.Timestamp.today() + pd.DateOffset(months=FORECAST_HORIZON_MONTHS)

boq_df = all_model_weekly.copy()
boq_df['ds'] = pd.to_datetime(boq_df['ds'])
boq_df = boq_df[(boq_df['ds'] >= _window_start) & (boq_df['ds'] <= _window_end)].copy()

# Apply quantiles
quantiles_boq = boq_df.apply(apply_quantiles, axis=1)
boq_df = pd.concat([boq_df, quantiles_boq], axis=1)
boq_df['quarter'] = boq_df['ds'].dt.to_period('Q').astype(str)
boq_df['fiscal_year'] = 'FY' + boq_df['quarter'].str[:4]
boq_df['source'] = 'forecast'
boq_df['grain_level'] = 'total'
boq_df['sales_market'] = 'ALL'  # 2026-09-30: standardized to match itsg_forecast_v5's own 'ALL' sentinel for total-grain rows (was 'Total', inconsistent within this notebook's own tables)
boq_df['run_date_utc'] = datetime.now(timezone.utc).strftime('%Y-%m-%d')
boq_df['pipeline_version'] = PIPELINE_VERSION

boq_out = boq_df[['model', 'ds', 'quarter', 'forecast'] + QUANTILE_COLS +
                  ['fiscal_year', 'source', 'grain_level', 'sales_market',
                  'run_date_utc', 'pipeline_version']].copy()

# Write BOQ
spark_boq = spark.createDataFrame(boq_out)
spark_boq.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(BOQ_TABLE)
print(f'TABLE 1 written: {BOQ_TABLE} ({len(boq_out)} rows, {boq_out["model"].nunique()} models)')
print(f'  Window: {boq_df["ds"].min().date()} -> {boq_df["ds"].max().date()}')
print()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TABLE 2: CBE — Actuals (closed weeks) + Forecast (open weeks)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
_last_actual_date = weekly_itsg['ds'].max()

# Actuals rows — from weekly_itsg for weeks in the rolling window
actuals_weekly = weekly_itsg[['ds', 'y']].copy()
actuals_weekly['ds'] = pd.to_datetime(actuals_weekly['ds'])
actuals_weekly = actuals_weekly[(actuals_weekly['ds'] >= _window_start) & (actuals_weekly['ds'] <= _last_actual_date)].copy()

cbe_rows = []

# Add actual rows for each model (actual = realized, no uncertainty)
for m_name in boq_out['model'].unique():
    m_actuals = actuals_weekly.copy()
    m_actuals['model'] = m_name
    m_actuals['forecast'] = m_actuals['y']
    m_actuals['source'] = 'actual'
    for qc in QUANTILE_COLS:
        m_actuals[qc] = m_actuals['y']
    cbe_rows.append(m_actuals[['ds', 'model', 'forecast', 'source'] + QUANTILE_COLS])

# Add forecast rows for open weeks
fwd_for_cbe = boq_df[boq_df['ds'] > _last_actual_date][['ds', 'model', 'forecast'] + QUANTILE_COLS].copy()
fwd_for_cbe['source'] = 'forecast'
cbe_rows.append(fwd_for_cbe)

cbe_df = pd.concat(cbe_rows, ignore_index=True)
cbe_df['ds'] = pd.to_datetime(cbe_df['ds'])
cbe_df = cbe_df.sort_values(['model', 'ds']).reset_index(drop=True)
cbe_df['quarter'] = cbe_df['ds'].dt.to_period('Q').astype(str)
cbe_df['fiscal_year'] = 'FY' + cbe_df['quarter'].str[:4]
cbe_df['grain_level'] = 'total'
cbe_df['sales_market'] = 'ALL'  # 2026-09-30: standardized to match itsg_forecast_v5's own 'ALL' sentinel for total-grain rows (was 'Total', inconsistent within this notebook's own tables)

# Cumulative QTD and pace vs BOQ
cbe_df['cumulative_qtd'] = cbe_df.groupby(['model', 'quarter'])['forecast'].cumsum()

# BOQ weekly reference (what the model predicted for each week at BOQ time)
boq_ref = boq_out.set_index(['model', 'ds'])['forecast'].to_dict()
cbe_df['boq_weekly'] = [boq_ref.get((m, d), np.nan) for m, d in zip(cbe_df['model'], cbe_df['ds'])]
cbe_df['boq_cumulative_qtd'] = cbe_df.groupby(['model', 'quarter'])['boq_weekly'].cumsum()
cbe_df['pace_vs_boq_pct'] = np.where(
    cbe_df['boq_cumulative_qtd'] > 0,
    ((cbe_df['cumulative_qtd'] / cbe_df['boq_cumulative_qtd']) - 1) * 100,
    0.0
)
cbe_df['run_date_utc'] = datetime.now(timezone.utc).strftime('%Y-%m-%d')
cbe_df['pipeline_version'] = PIPELINE_VERSION

cbe_out = cbe_df[['ds', 'model', 'forecast', 'source', 'quarter'] + QUANTILE_COLS +
                  ['fiscal_year', 'grain_level', 'sales_market', 'cumulative_qtd',
                  'boq_weekly', 'boq_cumulative_qtd', 'pace_vs_boq_pct',
                  'run_date_utc', 'pipeline_version']].copy()

# Write CBE
spark_cbe = spark.createDataFrame(cbe_out)
spark_cbe.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(CBE_TABLE)
print(f'TABLE 2 written: {CBE_TABLE} ({len(cbe_out)} rows, {cbe_out["model"].nunique()} models)')
print(f'  Actuals through: {_last_actual_date.date()} | Forecast from: {(_last_actual_date + pd.Timedelta(weeks=1)).date()}')
print(f'  Window: {cbe_df["ds"].min().date()} -> {cbe_df["ds"].max().date()}')
print()

# Summary
print('=' * 80)
print('ITSG PLANNING TABLES PERSISTED')
print('=' * 80)
print(f'  {BOQ_TABLE}: {len(boq_out)} rows (forecast only, all models, rolling 1yr)')
print(f'  {CBE_TABLE}: {len(cbe_out)} rows (actuals + forecast, cumulative QTD, pace tracking)')
print(f'  Quantiles: p05/p25/p50/p75/p95 (MC-based for Ensemble, residual-based for individual models)')
print(f'  Pipeline version: {PIPELINE_VERSION}')
print(f'  Run date: {datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")}')

In [ ]:
# =============================================================================
# STEP 8E: APPEND FORECAST VINTAGE (daily drift tracking)
# =============================================================================
# PURPOSE: Append the current run's forward forecasts to an append-only
# vintages table, mirroring UCC's Step 14C. This enables:
#   - Forecast vs Reality backtesting (what did we predict for now-closed weeks?)
#   - Vintage drift analysis (how stable are forecasts across daily runs?)
#   - Model comparison over time
#
# TABLE: datagroup_mdl.mdl_sales_analytics.itsg_forecast_vintages
#   Append-only (no overwrites). Each run adds its full forward prediction set.
#   Key columns: ds, model, p05, p25, p50, p75, p95, grain_level, sales_market,
#                run_date_utc, run_timestamp_utc, pipeline_version
# CHANGED: p10/p90 -> p05/p25/p75/p95, matching the notebook-wide move to the
# 5-column production interval convention (itsg_forecast_v5 now carries
# these columns instead of p10/p90 — see Step 7).
# =============================================================================

from pyspark.sql import functions as F

CATALOG = "datagroup_mdl"
SCHEMA = "mdl_sales_analytics"
ITSG_VINTAGES_TABLE = f"{CATALOG}.{SCHEMA}.itsg_forecast_vintages"

try:
    # Read the itsg_forecast_v5 output we just wrote
    itsg_v5 = spark.table(f"{CATALOG}.{SCHEMA}.itsg_forecast_v5")
    
    # Add vintage metadata columns (consistent with ucc_forecast_vintages schema)
    vintage_df = (itsg_v5
        .select(
            F.col("ds"),
            F.col("p50"),
            F.col("p05"),
            F.col("p25"),
            F.col("p75"),
            F.col("p95"),
            F.col("model"),
            F.col("run_date_utc").cast("string").alias("run_date_utc"),
            F.col("run_timestamp_utc").cast("string").alias("run_ts_utc"),
            F.lit("V5").alias("pipeline_version"),
            F.lit(False).alias("partial_week_dropped"),
            F.col("forecast"),
            F.col("grain_level"),
            F.col("sales_market"),
            F.col("run_timestamp_utc"),
            F.col("recommended_for_exec"),
            F.lit("ITSG_Growth_ARR_Forecasting_v5").alias("notebook_name"),
            F.lit("V5").alias("notebook_version"),
            F.lit("ITSG").alias("product_group"),
            F.lit(f"{CATALOG}.{SCHEMA}.itsg_forecast_v5").alias("source_table"),
        )
    )
    
    n_rows = vintage_df.count()
    
    # Append to vintages table (create if first run)
    (vintage_df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(ITSG_VINTAGES_TABLE)
    )
    
    total_vintages = spark.table(ITSG_VINTAGES_TABLE).count()
    distinct_dates = (spark.table(ITSG_VINTAGES_TABLE)
        .select(F.to_date("run_date_utc").alias("rd"))
        .distinct().count())
    
    print(f"\u2713 Appended {n_rows} rows to {ITSG_VINTAGES_TABLE}")
    print(f"  Total vintages: {total_vintages} rows across {distinct_dates} run dates")
    
except Exception as exc:
    print(f"  WARN: ITSG vintage append failed (non-blocking): {exc}")
    print(f"  The main outputs (itsg_forecast_v5, itsg_forecast_boq, itsg_forecast_cbe) are unaffected.")

### Step 18H - Validation scorecard for the evidence-gated improvements *(NEW)*

In [ ]:
# =============================================================================
# STEP 18H: VALIDATION SCORECARD FOR THE EVIDENCE-GATED IMPROVEMENTS (ITSG)
# =============================================================================
# Documents, in the run output, whether each change was adopted and what the out-of-sample evidence says.
# Every number comes from the cells above (no invented values).
print('=' * 78)
print(f"VALIDATION SCORECARD (IMPROVEMENT_GATES_ENABLED={globals().get('IMPROVEMENT_GATES_ENABLED')}, "
      f"ITSG_INCLUDE_CARE_SALES_OTHER={globals().get('ITSG_INCLUDE_CARE_SALES_OTHER')})")
print('=' * 78)
try:
    _R = globals().get('IG_RESULTS', {})
    _ab = _R.get('ablation')
    if _ab:
        _t = _ab['table'].set_index('variant'); _s = _ab['selected']
        print(f"A. LGB features, variant '{_s}': ADOPTED={_s != 'full'} (dropped {_ab['dropped']}) | pooled WAPE full {_t.loc['full', 'pool_wape']:.2f}% -> "
              f"{_t.loc[_s, 'pool_wape']:.2f}% | pooled bias {_t.loc['full', 'pool_bias']:+.1f}% -> {_t.loc[_s, 'pool_bias']:+.1f}% | {_ab['reason']}")
    else:
        print('A. LGB feature ablation: not evaluated (full set kept).')
    _mr = _R.get('miradore')
    if _mr:
        print(f"B. Miradore outlook (separate line): method={_mr['method']} validated={_mr['validated']} backtest months={_mr['n_backtest']} "
              f"MAE $K {({k: round(v / 1e3, 1) for k, v in _mr['mae'].items()})} unknown months={_mr['unknown_months']}")
    else:
        print('B. Miradore outlook: not produced (see the 13M cell output for the reason).')
    _qe = _R.get('qe')
    if _qe and len(_qe['table']):
        _t = _qe['table'].set_index('method'); _m = _qe['selected']
        print(f"C. QE method '{_m}': ADOPTED_CHANGE={_m != 'incumbent'} | last-week MAE incumbent ${_t.loc['incumbent', 'mae']:,.0f} -> "
              f"${_t.loc[_m, 'mae']:,.0f} | quarter-total MAE ${_t.loc['incumbent', 'qtr_total_mae']:,.0f} -> ${_t.loc[_m, 'qtr_total_mae']:,.0f} | n_quarters={_qe['n_quarters']}")
    else:
        print(f"C. QE method: incumbent kept ({_qe['reason'] if _qe else 'not evaluated'}).")
    _lg = _R.get('lgb')
    if _lg:
        _t = _lg['table'].set_index('mult'); _s = _lg['selected']
        print(f"D1. LGB weight x{_s}: ADOPTED={_s != 1.0} | CV WAPE 1.0={_t.loc[1.0, 'bt_wape']:.2f}% -> {_t.loc[_s, 'bt_wape']:.2f}%"
              + (f" | holdout bias {_t.loc[1.0, 'ho_bias_pct']:+.1f}% -> {_t.loc[_s, 'ho_bias_pct']:+.1f}%" if 'ho_bias_pct' in _t else ''))
    else:
        print('D1. LGB weight: not evaluated or LGB not in ensemble (kept 1.0).')
    _lv = _R.get('level')
    if _lv and 'wape_none' in _lv:
        print(f"D2. Level-shift ({_lv['mode']}): ADOPTED={_lv['adopt']} | non-QE WAPE {_lv['wape_none']:.2f}% -> {_lv['wape_adj']:.2f}% | "
              f"quarter-total abs err {_lv['qtr_abs_none']:,.0f} -> {_lv['qtr_abs_adj']:,.0f} | origins={_lv['n_origins']}")
    else:
        print(f"D2. Level-shift: not adopted ({_lv['reason'] if _lv else 'not evaluated'}).")
    print("E. Scope: Care/Sales Other " + ('INCLUDED' if globals().get('ITSG_INCLUDE_CARE_SALES_OTHER') else 'EXCLUDED (default)') + ' - see the Step 1C diagnostic for the quarterly difference.')
    _pc = _R.get('pace')
    if _pc and len(_pc['summary']):
        for _, r in _pc['summary'].iterrows():
            print(f"F. Pace as-of wk {int(r['asof_week']):>2}: validated={bool(r['validated'])} | outlook MAPE {r['outlook_mape']:.1f}% vs model {r['model_mape']:.1f}% "
                  f"| n_quarters={int(r['n_quarters'])}")
    else:
        print('F. Pace outlook: not backtested (too few complete OOS quarters) - outlook row, if written, is NOT validated.')
    _h = globals().get('ensemble_holdout')
    if _h is not None and len(_h):
        _h = _h.dropna(subset=['actual', 'forecast'])
        print(f"Overall holdout (Step 7 ensemble, pre-QE, {len(_h)} weeks): WAPE {ig_wape(_h['actual'], _h['forecast']):.2f}% | bias {ig_bias_pct(_h['actual'], _h['forecast']):+.2f}%")
    _wk = globals().get('IG_WK')
    if _wk is not None:
        print('Closed-quarter total error (OOS weekly forecasts; partial quarters excluded):')
        for _q, _g in ig_complete_quarters(_wk):
            print(f"   {_q}: actual ${_g['actual'].sum():,.0f}  forecast ${_g['fc'].sum():,.0f}  error {(_g['fc'].sum() / _g['actual'].sum() - 1) * 100:+.1f}%")
    print('Reading guide: ADOPTED=False means the evidence did not clear the gate and the previous behaviour is unchanged.')
except Exception as _exc_sc:
    print(f'Scorecard skipped (fail-soft): {type(_exc_sc).__name__}: {str(_exc_sc)[:250]}')

## ITSG Forecast Foundation V5 — Current Status for Team Sharing

Following your preferences, this section is the concise status note to keep with the final outputs.

* UCC-style governance is now in place: explicit ensemble membership validation in Step 6B and authoritative propagation into Step 7.
* Team-facing outputs now align more closely with the UCC notebook: generated executive summary, complete run summary dashboard, team runbook, and final FY2026 quarter table.
* The live ensemble and all published metrics shown in the notebook are rendered from current run DataFrames rather than hand-entered narrative.
* For leadership discussions, use quarterly totals and the Monte Carlo P05 / P25 / P50 / P75 / P95 range, not isolated weekly point values.
* Remaining work is routine operational follow-through rather than structural remediation: scheduled reruns, monitoring-baseline review, and future ensemble re-evaluation only if accuracy evidence changes.


## VS Code / Cursor Prompt — Forward 2027 Outlook + Actuals Refresh

> **STALE NOTICE (2026-09-30):** the table schemas and SQL below were written
> before the pipeline moved from a P10/P50/P90 convention to the current
> P05/P25/P50/P75/P95 convention. The schema list and SQL snippets have been
> updated in place to match what these tables actually look like now, but
> this whole cell is otherwise historical scratch (a one-time prompt for an
> app-side IDE agent, not something this notebook runs) -- treat the rest of
> it as a point-in-time snapshot of a task, not living documentation.

Copy everything below into your IDE agent:

---

### TASK: Add 2027 Forward Outlook as First Visual + Fix Stale Actuals

**Context**: The `forecast_unify_publish` notebook now writes rows with:
- `forecast_type = 'forward_1yr'` (weeks beyond FY2026-12-28, i.e. all of 2027)
- `forecast_view = 'adaptive'` or `'boq'` (two planning perspectives)
- The tables `forecast_results` and `forecast_results_v2compat` already contain these rows.
- Combined FY2027 forecast: UCC=$43.49M + ITSG=$9.92M = $53.41M (Q1-Q3 2027)
- FY2026 actuals through Jul 27: UCC Q3=$4.578M (4 closed weeks), ITSG Q3=$0.809M
- App currently shows Q3 actual = $2.6M which is STALE — correct combined value is ~$5.4M

**Tables the app reads from** (all in `datagroup_mdl.mdl_sales_analytics`):
- `arr_forecast_v2` — wide format: ds, product, sales_market, Actuals, Most_Likely, Worst_Case, Best_Case, arr_ets, arr_prophet, arr_lightgbm, arr_mstl_v2, arr_dhr_arima, forecast_type, forecast_view, run_date
- `forecast_results_v2compat` — view over forecast_results (long format: model, p05, p25, p50, p75, p95, actual, forecast_type, forecast_view) -- was p10/p50/p90 until the 2026-09-28/29 quantile-convention conversion
- `ucc_forecast_v5` — UCC weekly forecasts (model, ds, full p05..p95 quantile grid incl. legacy p10-p90-by-10s, forecast_version, run_date)
- `itsg_forecast_v5` — ITSG weekly forecasts (model, ds, p05, p25, p50, p75, p95 -- no longer the same schema as UCC's: ITSG dropped the legacy p10-by-10s grid entirely, UCC kept it as an internal superset)
- `arr_forecast_app_latest` — UCC V5 app-facing output

---

### BACKEND CHANGES (`backend/routes/forecast_v2_impl.py`)

**1. Add `forward_1yr` to valid forecast types (line 73):**
```python
VALID_FORECAST_TYPES = {"actuals", "rolling", "roy", "forward_1yr"}
```

**2. New endpoint — `GET /api/forecast/v2/forward-outlook`**

Returns quarterly-aggregated 2027+ forecast data for the Planning View:
```python
@router.get("/forward-outlook")
async def forward_outlook(
    product: Optional[str] = Query(None),
    sales_market: Optional[str] = Query(None),
    forecast_view: str = Query("adaptive"),  # 'adaptive' or 'boq'
    _user=Depends(require_authenticated_user),
    live: bool = Depends(_live_mode_dependency),
):
    """
    Quarterly-aggregated forward outlook (2027+).
    Returns BOQ vs Adaptive comparison for executive planning.
    """
    if not live:
        return {"source": "demo", "quarters": _demo_forward_outlook()}

    prod_filter = f"AND product = '{product}'" if product else ""
    geo_filter = f"AND sales_market = '{sales_market}'" if sales_market else ""

    sql = f"""
    SELECT
        date_trunc('quarter', ds) AS quarter_start,
        CONCAT('Q', QUARTER(ds), ' ', YEAR(ds)) AS quarter_label,
        forecast_view,
        SUM(COALESCE(actual, 0)) AS actual_total,
        SUM(COALESCE(p50, Most_Likely)) AS forecast_p50,
        SUM(COALESCE(p05, Worst_Case)) AS forecast_p05,
        SUM(COALESCE(p95, Best_Case)) AS forecast_p95,
        COUNT(DISTINCT ds) AS weeks_in_quarter
    FROM {FC_TABLE}
    WHERE forecast_type = 'forward_1yr'
      AND run_date = (SELECT MAX(run_date) FROM {FC_TABLE} WHERE forecast_type = 'forward_1yr')
      {prod_filter} {geo_filter}
    GROUP BY 1, 2, 3
    ORDER BY quarter_start
    """
    rows = await _cached_query(sql, "forward_outlook", product=product, geo=sales_market, view=forecast_view)
    return {"source": "live", "quarters": rows}
```

**3. New endpoint — `GET /api/forecast/v2/planning-view`**

BOQ vs Adaptive quarterly comparison (the "are we ahead or behind plan?" view):
```python
@router.get("/planning-view")
async def planning_view(
    product: Optional[str] = Query(None),
    sales_market: Optional[str] = Query(None),
    _user=Depends(require_authenticated_user),
    live: bool = Depends(_live_mode_dependency),
):
    """
    BOQ vs Adaptive quarterly comparison — FY2026 + FY2027.
    Shows actual progress against the beginning-of-quarter model prediction.
    """
    if not live:
        return {"source": "demo", "data": _demo_planning_view()}

    prod_filter = f"AND product = '{product}'" if product else ""
    geo_filter = f"AND sales_market = '{sales_market}'" if sales_market else ""

    sql = f"""
    WITH latest AS (
        SELECT MAX(run_date) AS max_rd FROM {FC_TABLE}
        WHERE forecast_type IN ('rolling', 'roy', 'forward_1yr')
    ),
    combined AS (
        SELECT
            CONCAT('Q', QUARTER(ds), ' ', YEAR(ds)) AS quarter_label,
            YEAR(ds) AS yr,
            QUARTER(ds) AS qtr,
            forecast_view,
            SUM(CASE WHEN actual IS NOT NULL AND actual > 0 THEN actual ELSE 0 END) AS actual_total,
            SUM(COALESCE(p50, Most_Likely, 0)) AS forecast_total,
            COUNT(DISTINCT CASE WHEN actual IS NOT NULL AND actual > 0 THEN ds END) AS actual_weeks,
            COUNT(DISTINCT ds) AS total_weeks
        FROM {FC_TABLE}, latest
        WHERE run_date = latest.max_rd
          AND forecast_type IN ('actuals', 'rolling', 'roy', 'forward_1yr')
          {prod_filter} {geo_filter}
        GROUP BY 1, 2, 3, 4
    )
    SELECT
        quarter_label, yr, qtr, forecast_view,
        actual_total, forecast_total, actual_weeks, total_weeks,
        CASE
            WHEN actual_weeks >= 13 THEN 'complete'
            WHEN actual_weeks > 0 THEN 'in_progress'
            ELSE 'forecast'
        END AS status
    FROM combined
    ORDER BY yr, qtr, forecast_view
    """
    rows = await _cached_query(sql, "planning_view", product=product, geo=sales_market)

    # Reshape: for each quarter, compute adaptive_total (actual + remaining forecast)
    # and boq_total (pure model prediction)
    quarters = {}
    for r in rows:
        qlabel = r['quarter_label']
        view = r.get('forecast_view', 'adaptive')
        if qlabel not in quarters:
            quarters[qlabel] = {'quarter': qlabel, 'yr': r['yr'], 'qtr': r['qtr']}
        if view == 'adaptive':
            actual = _f(r['actual_total'])
            fwd = _f(r['forecast_total']) - actual
            quarters[qlabel]['actual'] = actual
            quarters[qlabel]['fwd_forecast'] = max(0, fwd)
            quarters[qlabel]['adaptive_total'] = _f(r['forecast_total'])
            quarters[qlabel]['actual_weeks'] = int(r.get('actual_weeks', 0))
            quarters[qlabel]['status'] = r.get('status', 'forecast')
        elif view == 'boq':
            quarters[qlabel]['boq_total'] = _f(r['forecast_total'])

    result = sorted(quarters.values(), key=lambda x: (x.get('yr', 0), x.get('qtr', 0)))
    for q in result:
        boq = q.get('boq_total', 0)
        adapt = q.get('adaptive_total', 0)
        q['pace_vs_boq_pct'] = round(((adapt / boq) - 1) * 100, 1) if boq > 0 else 0.0

    return {"source": "live", "data": result}
```

**4. Fix actuals freshness** — The existing `/weekly` endpoint reads from `arr_forecast_v2` which gets actuals from the `forecast_unify_publish` job. Ensure the weekly endpoint includes the latest actuals week. No code change needed — the stale data is because the job was broken (now fixed). Once the job runs, actuals will update to $4.58M for UCC Q3 and $0.81M for ITSG Q3.

---

### FRONTEND CHANGES

**1. `frontend/src/components/forecast/constants.js` — Add forward_1yr FC type:**
```javascript
export const FC_TYPES = [
  { key: 'rolling', label: '13-Week Quarter' },
  { key: 'roy', label: 'Rest of Year' },
  { key: 'forward_1yr', label: 'Next Year (2027)' },
];
```

**2. New file: `frontend/src/components/forecast/tabs/PlanningViewTab.jsx`**

This is the BOQ vs Adaptive quarterly bars visual with pace badges — should be the **FIRST section on the Overview tab** (insert ABOVE the existing hero cards):

```jsx
import { BarChart, Bar, XAxis, YAxis, Tooltip, ResponsiveContainer, Cell, ReferenceLine, Legend } from 'recharts';
import { CardWrap, SectionTitle, fmtM } from '../common';

const COLORS = {
  actual: '#f59e0b',      // gold for closed actuals
  forecast: '#3b82f6',    // blue for forecast portion
  boq: 'rgba(148,163,184,0.5)', // muted gray for BOQ reference
};

const PaceBadge = ({ pct }) => {
  const color = pct >= 0 ? '#10b981' : '#ef4444';
  const arrow = pct >= 0 ? '▲' : '▼';
  return (
    <span style={{
      fontSize: 10, fontWeight: 700, padding: '2px 8px', borderRadius: 12,
      color, background: `${color}18`, border: `1px solid ${color}40`,
    }}>
      {arrow} {pct >= 0 ? '+' : ''}{pct.toFixed(1)}%
    </span>
  );
};

const PlanningViewSection = ({ data, loading, runDate }) => {
  if (loading || !data || data.length === 0) return null;

  // Split into current FY and next FY
  const currentYear = new Date().getFullYear();
  const fyThis = data.filter(q => q.yr === currentYear);
  const fyNext = data.filter(q => q.yr === currentYear + 1);

  const renderQuarterBars = (quarters, title) => (
    <div style={{ marginBottom: 16 }}>
      <div style={{ fontSize: 11, color: '#94a3b8', fontWeight: 600, marginBottom: 8, textTransform: 'uppercase', letterSpacing: '0.05em' }}>{title}</div>
      <div style={{ display: 'grid', gridTemplateColumns: `repeat(${quarters.length}, 1fr)`, gap: 12 }}>
        {quarters.map(q => {
          const isComplete = q.status === 'complete';
          const isPartial = q.status === 'in_progress';
          const adaptTotal = q.adaptive_total || 0;
          const boqTotal = q.boq_total || 0;
          const actual = q.actual || 0;
          const fwd = q.fwd_forecast || 0;
          const maxVal = Math.max(adaptTotal, boqTotal) * 1.1;
          const actualPct = maxVal > 0 ? (actual / maxVal) * 100 : 0;
          const fwdPct = maxVal > 0 ? (fwd / maxVal) * 100 : 0;
          const boqPct = maxVal > 0 ? (boqTotal / maxVal) * 100 : 0;

          return (
            <div key={q.quarter} style={{ textAlign: 'center' }}>
              {/* Stacked bar */}
              <div style={{ height: 120, display: 'flex', flexDirection: 'column', justifyContent: 'flex-end', alignItems: 'center', position: 'relative' }}>
                {/* BOQ reference line */}
                <div style={{
                  position: 'absolute', bottom: `${boqPct}%`, left: 0, right: 0,
                  borderTop: '2px dashed rgba(148,163,184,0.6)',
                }} />
                {/* Forecast portion */}
                {fwd > 0 && <div style={{ width: '60%', height: `${fwdPct}%`, background: 'linear-gradient(180deg, #60a5fa, #3b82f6)', borderRadius: '6px 6px 0 0' }} />}
                {/* Actual portion */}
                {actual > 0 && <div style={{ width: '60%', height: `${actualPct}%`, background: 'linear-gradient(180deg, #fbbf24, #f59e0b)', borderRadius: fwd > 0 ? '0' : '6px 6px 0 0' }} />}
              </div>
              {/* Label */}
              <div style={{ fontSize: 12, fontWeight: 700, color: '#e2e8f0', marginTop: 6 }}>{q.quarter}</div>
              <div style={{ fontSize: 11, color: isComplete ? '#10b981' : '#93c5fd', fontWeight: 600 }}>
                {fmtM(adaptTotal)}
              </div>
              {isPartial && <div style={{ fontSize: 9, color: '#64748b' }}>{fmtM(actual)} + {fmtM(fwd)}</div>}
              {!isComplete && q.pace_vs_boq_pct !== 0 && <PaceBadge pct={q.pace_vs_boq_pct} />}
              {isComplete && <span style={{ fontSize: 9, color: '#10b981' }}>✓ closed</span>}
            </div>
          );
        })}
      </div>
    </div>
  );

  const fyThisTotal = fyThis.reduce((s, q) => s + (q.adaptive_total || 0), 0);
  const fyNextTotal = fyNext.reduce((s, q) => s + (q.adaptive_total || 0), 0);

  return (
    <CardWrap downloadName="planning_view_boq_vs_adaptive">
      <SectionTitle>
        Planning View — BOQ vs Adaptive Trajectory
        {runDate && <span style={{ fontSize: 11, color: '#94a3b8', fontWeight: 400 }}> — as of {runDate}</span>}
      </SectionTitle>
      <div style={{ fontSize: 10, color: '#475569', marginBottom: 12, display: 'flex', gap: 14 }}>
        <span><span style={{ color: '#f59e0b' }}>■</span> Realized actuals</span>
        <span><span style={{ color: '#3b82f6' }}>■</span> Forecast — Most Likely</span>
        <span style={{ borderBottom: '2px dashed rgba(148,163,184,0.6)', paddingBottom: 1 }}>┄</span> BOQ model target
        <span style={{ marginLeft: 'auto', fontSize: 11, fontWeight: 700, color: '#e2e8f0' }}>FY{currentYear}: {fmtM(fyThisTotal)}</span>
        {fyNextTotal > 0 && <span style={{ fontSize: 11, fontWeight: 700, color: '#93c5fd' }}>FY{currentYear + 1}: {fmtM(fyNextTotal)}</span>}
      </div>
      {fyThis.length > 0 && renderQuarterBars(fyThis, `FY${currentYear}`)}
      {fyNext.length > 0 && renderQuarterBars(fyNext, `FY${currentYear + 1} (Forward Outlook)`)}
      <div style={{ fontSize: 9, color: '#334155', marginTop: 8 }}>
        In-progress quarters stack actuals (gold) + remaining forecast (blue). Dashed line = BOQ model baseline. Pace badge shows % above/below plan.
      </div>
    </CardWrap>
  );
};

export default PlanningViewSection;
```

**3. Update `OverviewTab.jsx` — Insert PlanningViewSection as the FIRST visual:**

At the top of OverviewTab, add:
```jsx
import PlanningViewSection from './PlanningViewTab';
```

Then inside the return, BEFORE the existing KPI hero cards (line 26), add:
```jsx
<PlanningViewSection data={planningData} loading={loading} runDate={runDate} />
```

**4. Update `ForecastingPanelContainer.jsx` — Fetch planning view data:**

Add state:
```javascript
const [planningData, setPlanningData] = useState(null);
```

In `fetchOverviewData`, add to the `Promise.allSettled` array:
```javascript
apiService.getForecastV2PlanningView(activePl, activeGeo),
```

And handle the result:
```javascript
if (planningRes.status === 'fulfilled') setPlanningData(planningRes.value?.data ?? []);
```

Pass to OverviewTab:
```jsx
<OverviewTab ... planningData={planningData} />
```

**5. Update `frontend/src/services/api.js` — Add new API calls:**
```javascript
getForecastV2PlanningView: (product, salesMarket) =>
  get('/api/forecast/v2/planning-view', {
    ...(product && { product }),
    ...(salesMarket && { sales_market: salesMarket }),
  }),

getForecastV2ForwardOutlook: (product, salesMarket, forecastView = 'adaptive') =>
  get('/api/forecast/v2/forward-outlook', {
    forecast_view: forecastView,
    ...(product && { product }),
    ...(salesMarket && { sales_market: salesMarket }),
  }),
```

---

### IMMEDIATE DATA FIX (Actuals are stale)

The Q3 actuals showing $2.6M instead of ~$5.4M is because the scheduled jobs were broken (cluster permission issue — now fixed, switched to serverless). To refresh immediately:

1. The `forecast_unify_publish` notebook must run — it reads from `ucc_forecast_v5` + `itsg_forecast_v5` + SFDC actuals and writes to `forecast_results` / `arr_forecast_v2` (via compat view)
2. Then `arr_forecast_all_models` must run to update the exec-wide view
3. Both jobs now use serverless compute (bundle deployed Aug 7 2026)

**Expected correct values after refresh:**
- UCC Q3 actual (4 closed weeks through Jul 27): **$4.578M**
- ITSG Q3 actual (4 closed weeks): **$0.809M**
- Combined Q3 actual so far: **$5.387M** (not the $2.6M currently showing)
- UCC Q3 total (4wk actual + 9wk forecast): **$14.593M** (+5.1% vs BOQ $13.89M)
- ITSG Q3 total (adaptive): **$3.548M** (-4.4% vs BOQ $3.713M)

---

### SUMMARY OF FILES TO MODIFY

| File | Change |
|------|--------|
| `backend/routes/forecast_v2_impl.py` | Add `forward_1yr` to VALID_FORECAST_TYPES; add `/forward-outlook` and `/planning-view` endpoints |
| `frontend/src/components/forecast/constants.js` | Add `forward_1yr` to FC_TYPES |
| `frontend/src/components/forecast/tabs/PlanningViewTab.jsx` | NEW — BOQ vs Adaptive bars component |
| `frontend/src/components/forecast/tabs/OverviewTab.jsx` | Import + render PlanningViewSection as first visual |
| `frontend/src/components/forecast/ForecastingPanelContainer.jsx` | Add planningData state, fetch in Overview, pass as prop |
| `frontend/src/services/api.js` | Add getForecastV2PlanningView + getForecastV2ForwardOutlook |

---

### KEY DESIGN DECISIONS
- **Planning View is FIRST on Overview** — executives see FY trajectory immediately
- **BOQ = dashed reference line** — the model's locked prediction at start of quarter
- **Adaptive = stacked actual (gold) + forecast (blue)** — shows live trajectory
- **Pace badges** — green/red % showing ahead/behind BOQ
- **2027 quarters appear automatically** — any row with `forecast_type='forward_1yr'` in the table gets rendered
- **Monthly grain not needed** — the bars aggregate weekly to quarterly (13 weeks per Q)